In [ ]:
# =======================
# Imports
# =======================
import json
import math
import os
import random
import re
import tempfile
import time
import io
import warnings
from collections import Counter
from itertools import combinations
from pathlib import Path
from typing import List, Tuple
import pdb  # keep if you actually use breakpoints
from tqdm.auto import tqdm
import numpy as np
import pandas as pd
import matplotlib
from collections import defaultdict
import gc
import copy
import seaborn as sns
import json
# Prefer inline plotting in notebooks; if not a notebook, fall back to Agg
try:
    matplotlib.use("module://matplotlib_inline.backend_inline")
except Exception:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image
warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 120  # crisp inline figures
sns.set_context("notebook")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils import weight_norm
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, TensorDataset
from torch import onnx  # only if you export models to ONNX

import pytorch_lightning as pl
from pytorch_lightning import LightningDataModule
from pytorch_lightning.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from pytorch_lightning.loggers import CSVLogger, WandbLogger
import wandb
from torchmetrics import Accuracy, Precision, Recall, F1Score, ConfusionMatrix
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    precision_recall_fscore_support,
)
from sklearn.metrics.pairwise import cosine_similarity
import scipy as scipy
from scipy.interpolate import CubicSpline
from scipy.integrate import cumulative_trapezoid as cumtrapz
from scipy.signal import find_peaks
from scipy.stats import pearsonr, spearmanr, entropy

from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.metrics import mutual_info_score
from sklearn.preprocessing import StandardScaler
from sklearn.multiclass import OneVsRestClassifier

from NormWear.main_model import NormWearModel
# ---- Reproducibility ----
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

In [ ]:
import sys
sys.path.append("/workspace/extra_packages")

import torch
import torchaudio

print("Torch version   :", torch.__version__)
print("Torchaudio ver. :", torchaudio.__version__)


In [ ]:
import os
from pathlib import Path

# Adjust if your username/path differs
BASE = Path(r"/workspace")

CODE_ROOT = BASE / "NormWear"
DATA_ROOT = BASE / "data"
ETH_ROOT  = DATA_ROOT / "eth_dataset"
UCI_ROOT  = DATA_ROOT / "UCI_HAR"

os.environ["DATASET_PATH"] = str(DATA_ROOT)     # parent folder containing eth_dataset
os.environ["ETH_SUBPATH"]  = "eth_dataset"
os.environ["DATA_ROOT"]    = str(DATA_ROOT)     # optional, keep consistent
os.environ["UCI_ROOT"]     = str(UCI_ROOT)
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

print("CODE_ROOT =", CODE_ROOT)
print("ETH_ROOT  =", ETH_ROOT)
print("DATASET_PATH =", os.environ["DATASET_PATH"])
print("ETH_SUBPATH  =", os.environ["ETH_SUBPATH"])
print("UCI_ROOT     =", os.environ["UCI_ROOT"])

assert CODE_ROOT.exists(), f"Missing CODE_ROOT: {CODE_ROOT}"
assert ETH_ROOT.exists(),  f"Missing ETH_ROOT: {ETH_ROOT}"


In [ ]:
# ---- Outputs root ----
if os.path.exists("/workspace") and os.access("/workspace", os.W_OK):
    OUTPUTS = "/workspace/base/src/outputs/Benchmark"
elif os.access(os.getcwd(), os.W_OK):
    OUTPUTS = os.path.join(os.getcwd(), "outputs/Benchmark")
else:
    OUTPUTS = "/tmp/outputs/Benchmark"

for sub in ["", "transformer", "apple", "inceptiontime", "hydra", "normwear_base", "normwear_lora"]:
    os.makedirs(os.path.join(OUTPUTS, sub), exist_ok=True)

# ---- NormWear LoRA run dir (ONLY) ----
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
NORMWEAR_DIR = os.path.join(OUTPUTS, "normwear_lora", f"run_{RUN_TAG}")
os.makedirs(NORMWEAR_DIR, exist_ok=True)

print("Output directory:", OUTPUTS)
print("NormWear LoRA run directory:", NORMWEAR_DIR)

# ---- All artifacts/checkpoints for THIS run ----
SAVE_PATH = os.path.join(NORMWEAR_DIR, "normwear_lora_finetune_best.pt")
SAVE_PATH_LORA_ONLY = os.path.join(NORMWEAR_DIR, "normwear_lora_weights_only.pt")
CONFIG_JSON = os.path.join(NORMWEAR_DIR, "config.json")
HISTORY_JSON = os.path.join(NORMWEAR_DIR, "history.json")

In [ ]:
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
NORMWEAR_BASE_DIR = os.path.join(OUTPUTS, "normwear", f"run_{RUN_TAG}")
NORMWEAR_LORA_DIR = os.path.join(OUTPUTS, "normwear_lora", f"run_{RUN_TAG}")

# For LoRA training, set:
NORMWEAR_DIR = NORMWEAR_LORA_DIR

SAVE_PATH = os.path.join(NORMWEAR_LORA_DIR, "normwear_lora_finetune_best.pt")
SAVE_PATH_LORA_ONLY = os.path.join(NORMWEAR_LORA_DIR, "normwear_lora_weights_only.pt")

In [ ]:
TRAIN = [
    "id_1","id_2","id_3","id_4","id_5","id_6","id_7","id_8","id_9","id_10","id_11","id_12",
    "id_13","id_14","id_15","id_16","id_17","id_18","id_19","id_20","id_21","id_22","id_23","id_24",
    "id_25","id_26","id_27","id_28","id_29","id_30","id_31","id_32","id_33","id_34","id_35",
]
# New mirrored participants created by augmentations:
NEW_MIRRORED = [
    "id_51","id_52","id_53","id_54","id_55","id_56","id_57","id_58","id_59","id_60",
    "id_61","id_62","id_63","id_64","id_65","id_66","id_67","id_68","id_69","id_70",
    "id_71","id_72","id_73","id_74","id_75","id_76","id_77","id_78","id_79","id_80",
    "id_81","id_82","id_83","id_84","id_85","id_85",
]

TRAIN_WITH_MIRRORS = TRAIN + NEW_MIRRORED

VAL = ["id_36", "id_37", "id_38", "id_39", "id_40"]

TEST = ["id_41", "id_42", "id_43", "id_44", "id_45","id_46", "id_47", "id_48", "id_49", "id_50"]

participants_split = {"train": TRAIN_WITH_MIRRORS, "val": VAL, "test": TEST}


In [ ]:
# ===== Helper & Utils Functions Definition =====
def compute_time_domain_features(x: torch.Tensor, eps=1e-8) -> torch.Tensor:
    """
    x: [T, C] tensor
    Returns: [N_features, C] tensor
    """
    T, C = x.shape
    mean = x.mean(dim=0)
    std = x.std(dim=0)
    var = x.var(dim=0)
    min_ = x.min(dim=0).values
    max_ = x.max(dim=0).values
    median = x.median(dim=0).values
    q75 = torch.quantile(x, 0.75, dim=0)
    q25 = torch.quantile(x, 0.25, dim=0)
    iqr = q75 - q25
    range_ = max_ - min_
    rms = torch.sqrt(torch.mean(x**2, dim=0))
    mad = torch.mean(torch.abs(x - mean), dim=0)
    energy = torch.sum(x**2, dim=0)
    zcr = ((x[:-1] * x[1:]) < 0).sum(dim=0)

    # Autocorrelation (lag 1 and lag 2) via batch normed dot products
    def autocorr_lag(x, lag):
        x1, x2 = x[:-lag], x[lag:]
        x1 = (x1 - x1.mean(0)) / (x1.std(0)+eps)
        x2 = (x2 - x2.mean(0)) / (x2.std(0)+eps)
        return (x1 * x2).mean(0)

    autocorr1 = autocorr_lag(x, 1)
    autocorr2 = autocorr_lag(x, 2)

    # Skew & Kurtosis (fallback to NumPy)
    x_np = x.cpu().numpy()
    skew = torch.from_numpy(scipy.stats.skew(x_np, axis=0)).to(x.device)
    kurt = torch.from_numpy(scipy.stats.kurtosis(x_np, axis=0)).to(x.device)

    # --- Hjorth parameters ---
    dx = torch.diff(x, dim=0)
    ddx = torch.diff(dx, dim=0)
    var_x = x.var(dim=0)
    var_dx = dx.var(dim=0)
    var_ddx = ddx.var(dim=0)
    activity = var_x
    mobility = torch.sqrt(var_dx / (var_x +eps))
    complexity = torch.sqrt(var_ddx / (var_dx+eps)) / mobility

    return torch.stack([
        mean, std, var, skew, kurt, min_, max_, median, range_, iqr,
        rms, mad, energy, zcr, autocorr1, autocorr2,
        activity, mobility, complexity
    ])
    
def compute_frequency_features(x: torch.Tensor, fs: float = 1.0, top_k: int = 3, eps = 1e-8) -> torch.Tensor:
    """
    x: [T, C] tensor
    Returns: [N_features, C] tensor
    """
    T, C = x.shape
    freqs = torch.fft.rfftfreq(T, d=1/fs).to(x.device)  # [F]
    X = torch.fft.rfft(x, dim=0).to(x.device)            # [F, C]
    mag = torch.abs(X)                     # [F, C]
    psd = mag ** 2                         # [F, C]
    psd_sum = psd.sum(dim=0)               # [C]

    # Spectral Centroid 
    centroid = (freqs[:, None] * psd).sum(dim=0) / (psd_sum+eps)

    # Spectral Entropy (fallback to NumPy for `scipy.stats.entropy`)
    psd_norm = psd / psd_sum
    spec_entropy = torch.from_numpy(
        entropy(psd_norm.cpu().numpy(), axis=0)
    ).to(x.device)

    # Spectral Flux
    flux = torch.sum(torch.diff(mag, dim=0) ** 2, dim=0)

    # Peak Frequency
    peak_idx = torch.argmax(mag, dim=0)
    peak_freq = freqs[peak_idx]

    # Band Energies
    low = psd[freqs < 0.1].sum(dim=0)
    mid = psd[(freqs >= 0.1) & (freqs < 0.25)].sum(dim=0)
    high = psd[freqs >= 0.25].sum(dim=0)

    # --- Spectral Roll-off (85%) ---
    threshold = 0.85 * psd_sum
    cum_energy = torch.cumsum(psd, dim=0)
    rolloff = torch.zeros(C, device=x.device)
    for c in range(C):
        idx = torch.searchsorted(cum_energy[:, c].contiguous(), threshold[c]) 
        rolloff[c] = freqs[min(idx.item(), freqs.shape[0] - 1)]

    # --- Spectral Flatness --- 
    geometric_mean = torch.exp((psd + eps).log().mean(dim=0))
    arithmetic_mean = psd.mean(dim=0)
    flatness = geometric_mean / (arithmetic_mean + eps)

    # --- Top-K Dominant Frequencies ---
    topk_vals, topk_idxs = torch.topk(mag, k=top_k, dim=0)
    topk_freqs = freqs[topk_idxs]  # [K, C]   
    
    return torch.cat([
        centroid.unsqueeze(0), spec_entropy.unsqueeze(0), psd_sum.unsqueeze(0), flux.unsqueeze(0),
        peak_freq.unsqueeze(0), low.unsqueeze(0), mid.unsqueeze(0), high.unsqueeze(0),
        rolloff.unsqueeze(0), flatness.unsqueeze(0),
        topk_freqs
    ], dim=0).to(torch.float32)


def compute_cross_channel_features(x: torch.Tensor) -> torch.Tensor:
    """
    x: [T, C]
    Returns: [N_cross_features, C] tensor, each cross-channel feature repeated over channels
    """
    x_np = x.cpu().numpy()
    C = x.shape[1]

    # Pearson correlation matrix (upper triangle, excluding diagonal)
    pearson = np.corrcoef(x_np.T)
    pearson_vals = pearson[np.triu_indices(C, k=1)]

    # Spearman correlation matrix
    spearman = np.zeros((C, C))
    for i in range(C):
        for j in range(C):
            spearman[i, j] = spearmanr(x_np[:, i], x_np[:, j]).correlation
    spearman_vals = spearman[np.triu_indices(C, k=1)]

    # Cosine similarity
    cosine_sim = cosine_similarity(x_np.T)
    cosine_vals = cosine_sim[np.triu_indices(C, k=1)]

    # PCA explained variance
    pca = PCA(n_components=C)
    pca.fit(x_np)
    pca_exp = pca.explained_variance_ratio_  # [C]

    # Mutual Information (discretize first)
    disc = KBinsDiscretizer(n_bins=20, encode='ordinal', strategy='uniform')
    x_disc = disc.fit_transform(x_np)
    mi = np.zeros((C, C))
    for i in range(C):
        for j in range(C):
            mi[i, j] = mutual_info_score(x_disc[:, i], x_disc[:, j])
    mi_vals = mi[np.triu_indices(C, k=1)]

    # Aggregate into a flat array of features
    features = np.concatenate([
        pearson_vals, spearman_vals, cosine_vals,
        pca_exp, mi_vals
    ])  # shape: [F]

    # Repeat over channels to get [F, C]
    features_tensor = torch.tensor(features, dtype=torch.float32, device=x.device).unsqueeze(1).repeat(1, C)
    return features_tensor


def compute_shape_features(x: torch.Tensor) -> torch.Tensor:
    """
    x: [T, C] tensor
    Returns: [N_features, C] tensor
    """
    T, C = x.shape
    t = np.arange(T).reshape(-1, 1)
    x_np = x.cpu().numpy()
    
    slopes = []
    n_peaks = []
    ptp_mean = []
    ptp_std = []
    aucs = []
    lis_all = []
    lds_all = []

    for c in range(C):
        sig = x_np[:, c]

        # Trend
        slope = LinearRegression().fit(t, sig).coef_[0]
        slopes.append(slope)

        # Peaks
        peaks, _ = find_peaks(sig)
        n_peaks.append(len(peaks))
        if len(peaks) > 1:
            dists = np.diff(peaks)
            ptp_mean.append(dists.mean())
            ptp_std.append(dists.std())
        else:
            ptp_mean.append(0.0)
            ptp_std.append(0.0)

        # AUC
        aucs.append(np.trapezoid(np.abs(sig)))

        # LIS/LDS
        def longest_subseq(arr, inc=True):
            dp = [1]*len(arr)
            for i in range(1, len(arr)):
                for j in range(i):
                    if (arr[i] > arr[j]) if inc else (arr[i] < arr[j]):
                        dp[i] = max(dp[i], dp[j]+1)
            return max(dp)
        lis_all.append(longest_subseq(sig, True))
        lds_all.append(longest_subseq(sig, False))

    return torch.tensor([
        slopes, n_peaks, ptp_mean, ptp_std, lis_all, lds_all, aucs
    ], dtype=torch.float32, device=x.device)

def get_statistical_features(x: torch.Tensor) -> torch.Tensor:
    """
    Returns: [total_num_features, num_channels] tensor
    """
    return torch.cat([
        compute_time_domain_features(x),
        compute_frequency_features(x),
        compute_shape_features(x),
        compute_cross_channel_features(x)
    ], dim=0).to(dtype=torch.float32)

def normalize_features(feat: torch.Tensor, eps=1e-8) -> torch.Tensor:
    """
    Normalize feature tensor [N_features, C] based on known feature groups.
    """

    N, C = feat.shape

    # Split indices by feature type (based on ordering in get_statistical_features)
    i = 0
    # --- Time Domain (19 feats) ---
    idx_time = torch.arange(i, i+19); i += 19

    # --- Frequency Domain (10 + top_k=3) = 13 ---
    idx_freq = torch.arange(i, i+13); i += 13

    # --- Shape Features (7) ---
    idx_shape = torch.arange(i, i+7); i += 7

    # --- Cross-channel (pearson, spearman, cosine, pca_exp, MI) ---
    idx_cross = torch.arange(i, N)

    # --- Normalize ---
    norm_feat = feat.clone()

    # Time & Shape: standardize
    for idx in [idx_time, idx_shape]:
        mean = norm_feat[idx].mean(dim=0, keepdim=True)
        std = norm_feat[idx].std(dim=0, keepdim=True) + eps
        norm_feat[idx] = (norm_feat[idx] - mean) / std

    # Frequency: log + standardize (log for PSD, energy, flux etc.)
    freq_vals = norm_feat[idx_freq]
    freq_vals = torch.log1p(freq_vals.abs())
    mean = freq_vals.mean(dim=0, keepdim=True)
    std = freq_vals.std(dim=0, keepdim=True) + eps
    norm_feat[idx_freq] = (freq_vals - mean) / std

    # Cross-channel:
    # - Pearson, Spearman, Cosine: bounded [-1, 1] — leave as is
    # - PCA explained variance: [0, 1] — leave as is
    # - Mutual Info: unbounded — apply log+z
    # Assuming order: [corr, spearman, cosine, pca, mi]
    # We don’t know C at compile-time, so infer sizes
    total_cross = N - i
    C = feat.shape[1]
    n_pairs = int(C * (C - 1) / 2)
    start_mi = i + 3 * n_pairs + C
    idx_mi = torch.arange(start_mi, N)

    mi_vals = norm_feat[idx_mi]
    mi_vals = torch.log1p(mi_vals)
    mean = mi_vals.mean(dim=0, keepdim=True)
    std = mi_vals.std(dim=0, keepdim=True) + eps
    norm_feat[idx_mi] = (mi_vals - mean) / std

    return norm_feat


def get_normalize_statistical_features(x: torch.Tensor) -> torch.Tensor:
    return normalize_features(get_statistical_features(x))

class IMURandomWalkDriftBiasTransform():
    def __init__(self, max_drift_std_per_sec=0.001, max_bias=0.02, sample_rate=50):
        self.max_drift_std_per_sec = max_drift_std_per_sec
        self.max_bias = max_bias
        self.sample_rate = sample_rate

    def __call__(self, x):
        T, C = x.shape
        assert C % 6 == 0
        dt = 1 / self.sample_rate
        drift = torch.zeros_like(x)
        bias = (torch.rand(C, device=x.device) - 0.5) * 2 * self.max_bias
        drift[:, 0] = 0
        for c in range(C):
            # Brownian motion (random walk) drift: cumulative sum of Gaussian noise
            increments = torch.randn(T, device=x.device) * self.max_drift_std_per_sec * np.sqrt(dt)
            drift[:, c] = torch.cumsum(increments, dim=0)
        return x + drift + bias.unsqueeze(0).repeat(T, 1)

class IMUGravityPerturbationTransform:
    def __init__(self, amp=0.1, freq=0.05):
        self.amp = amp
        self.freq = freq
    def __call__(self, x):
        T, C = x.shape
        t = torch.linspace(0, 1, T, device=x.device)
        tilt = self.amp * torch.sin(2 * math.pi * self.freq * t).unsqueeze(1)

        # Randomly add or remove
        direction = np.random.choice([-1, 1])
        x = x.clone()
        x[:, 2] += direction * tilt.squeeze()

        return x

class IMUAxisMisalignmentTransform():
    def __init__(self, max_angle_deg=15, p=0.5):
        self.max_angle = np.deg2rad(max_angle_deg)
        self.p = p
        
    def __call__(self, x): 
        
        T, C = x.shape
        assert C % 6 == 0
        n_sensors = C // 6

        # Limit rotations to ±max_angle only on pitch and roll, yaw=0 for physical realism
        angles = np.random.uniform(-self.max_angle, self.max_angle, size=2)  # pitch, roll
        pitch, roll = angles
        yaw = 0

        Rx = np.array([[1, 0, 0],
                       [0, np.cos(pitch), -np.sin(pitch)],
                       [0, np.sin(pitch), np.cos(pitch)]])
        Ry = np.array([[np.cos(roll), 0, np.sin(roll)],
                       [0, 1, 0],
                       [-np.sin(roll), 0, np.cos(roll)]])
        Rz = np.eye(3)  # no yaw rotation
        R = Rz @ Ry @ Rx

        x_rot = x.clone()
        for i in range(n_sensors):
            acc = x[:, 6*i:6*i+3].cpu().numpy()
            gyro = x[:, 6*i+3:6*i+6].cpu().numpy()
            acc_rot = acc @ R.T
            gyro_rot = gyro @ R.T
            x_rot[:, 6*i:6*i+3] = torch.tensor(acc_rot, dtype=x.dtype, device=x.device)
            x_rot[:, 6*i+3:6*i+6] = torch.tensor(gyro_rot, dtype=x.dtype, device=x.device)
        return x_rot


class IMUWalkingArtifactTransform:
    def __init__(self, freq_range=(1.0, 2.0), amp_range=(0.1, 0.3)):
        self.freq_range = freq_range
        self.amp_range = amp_range 

    def __call__(self, x): 
        T, C = x.shape
        device = x.device
        time = torch.arange(T, device=device) / T

        # Create sinusoidal walking signal
        freq = np.random.uniform(*self.freq_range)
        amp = np.random.uniform(*self.amp_range)
        walk_signal = amp * torch.sin(2 * math.pi * freq * time)

        # Apply to vertical channels (z-acc = 2, z-gyro = 5), if available
        apply_channels = [2, 5] if C >= 6 else list(range(C))

        x_aug = x.clone()
        for ch in apply_channels:
            x_aug[:, ch] += walk_signal
        return x_aug
    
    
class PPGLowFrequencyDriftTransform():
    def __init__(self, amplitude=0.05, freq=0.1, sample_rate=50):
        self.amplitude = amplitude
        self.freq = freq
        self.sample_rate = sample_rate

    def __call__(self, x):
        T = x.shape[0]
        time = np.arange(T) / self.sample_rate
        wander = self.amplitude * np.sin(2 * np.pi * self.freq * time)
        if x.ndim == 1:
            x_noisy = x + wander
        else:
            x_noisy = x + torch.tensor(wander, dtype=x.dtype, device=x.device).unsqueeze(1)
        return x_noisy

class PPGSpikeArtifactTransform():
    def __init__(self, max_spikes=3, amplitude=0.1):
        self.max_spikes = max_spikes
        self.amplitude = amplitude

    def __call__(self, x):
        T, C = x.shape if x.ndim > 1 else (x.shape[0], 1)
        n_spikes = np.random.randint(1, self.max_spikes + 1)
        x_out = x.clone()
        for _ in range(n_spikes):
            pos = np.random.randint(0, T)
            width = np.random.randint(1, max(2, T // 20))
            spike = torch.linspace(self.amplitude, 0, steps=width)
            for c in range(C):
                end = min(pos + width, T)
                x_out[pos:end, c] += spike[:end-pos]
        return x_out

class PPGRespiratoryModulationTransform:
    """Simulates physiological amplitude modulation in PPG."""
    def __init__(self, modulation_strength=0.2, freq=0.2, sample_rate=50):
        self.modulation_strength = modulation_strength
        self.freq = freq
        self.sample_rate = sample_rate

    def __call__(self, x):
        T, C = x.shape if x.ndim > 1 else (x.shape[0], 1)
        time = np.arange(T) / self.sample_rate
        mod = 1 + self.modulation_strength * np.sin(2 * np.pi * self.freq * time)
        mod = torch.tensor(mod, dtype=x.dtype, device=x.device).unsqueeze(1 if C > 1 else 0)
        return x * mod
    
class LowFrequencyNoiseTransform():
    def __init__(self, freq_range=(0.5, 2.0), amplitude=0.2, sample_rate=50):
        self.freq_range = freq_range
        self.amplitude = amplitude
        self.sample_rate = sample_rate

    def __call__(self, x):
        T, C = x.shape if x.ndim > 1 else (x.shape[0], 1)
        time = np.arange(T) / self.sample_rate
        freq = np.random.uniform(*self.freq_range)
        
        # Channel-wise random phase and amplitude
        phases = np.random.uniform(0, 2 * np.pi, size=C)
        amps = np.random.uniform(0.5 * self.amplitude, self.amplitude, size=C)

        noise = np.array([
            amps[c] * np.sin(2 * np.pi * freq * time + phases[c])
            for c in range(C)
        ]).T

        # Randomly decide to add or subtract noise (equal chance)
        sign = 1 if np.random.rand() > 0.5 else -1
        
        if C == 1:
            x_noisy = x + sign * noise.flatten()
        else:
            x_noisy = x + sign * torch.tensor(noise, dtype=x.dtype, device=x.device)

        return x_noisy


class ScalingTransform():
    """This class implements a scaling transformation on a time series, 
    i.e. a multiplication of all values by a constant factor.
    """
    def __init__(self, mean=1.0, std=0.2, clip_min=-2, clip_max=2):
        self.mean = mean
        self.std = std
        self.clip_min = clip_min
        self.clip_max = clip_max

    def __call__(self, x):
        factor = np.clip(np.random.normal(self.mean, self.std), self.clip_min, self.clip_max)
        return x * factor

class ZoomingTransform():
    def __init__(self, min_zoom=0.9, max_zoom=1.1):  # symmetric zoom range
        assert min_zoom > 0
        self.min_zoom = min_zoom
        self.max_zoom = max_zoom
    
    def __call__(self, x):
        zoom_factor = np.random.uniform(self.min_zoom, self.max_zoom)
        T = x.shape[0]
        new_T = int(np.round(T * zoom_factor))
        time_original = np.linspace(0, 1, T)
        time_new = np.linspace(0, 1, new_T)
        
        if x.ndim == 1:
            x_zoomed = np.interp(time_new, time_original, x)
            if new_T < T:
                pad = T - new_T
                pad_left = pad // 2
                pad_right = pad - pad_left
                x_zoomed = np.pad(x_zoomed, (pad_left, pad_right), mode='edge')
            elif new_T > T:
                crop = new_T - T
                crop_left = crop // 2
                crop_right = crop - crop_left
                x_zoomed = x_zoomed[crop_left:-crop_right if crop_right > 0 else None]
        else:
            x_zoomed = []
            for dim in range(x.shape[1]):
                x_dim = x[:, dim].cpu().numpy()
                x_z = np.interp(time_new, time_original, x_dim)
                if new_T < T:
                    pad = T - new_T
                    pad_left = pad // 2
                    pad_right = pad - pad_left
                    x_z = np.pad(x_z, (pad_left, pad_right), mode='edge')
                elif new_T > T:
                    crop = new_T - T
                    crop_left = crop // 2
                    crop_right = crop - crop_left
                    x_z = x_z[crop_left:-crop_right if crop_right > 0 else None]
                x_zoomed.append(x_z)
            x_zoomed = np.stack(x_zoomed, axis=1)
        
        return torch.tensor(x_zoomed, dtype=x.dtype, device=x.device)
    
    

class TimeWarpingTransform():
    def __init__(self, num_knots=4, mean=1.0, std=0.1, clip_min=0.7, clip_max=1.3):
        self.num_knots = num_knots + 2
        self.mean = mean
        self.std = std
        self.clip_min = clip_min
        self.clip_max = clip_max

    def __call__(self, x):
        T = x.shape[0]
        original_time = np.linspace(0, 1, T)
        knots = np.linspace(0, 1, self.num_knots)
        heights = np.clip(np.random.normal(self.mean, self.std, size=self.num_knots), self.clip_min, self.clip_max)
        spline = CubicSpline(knots, heights)
        tau = spline(original_time)
        tau = cumtrapz(tau, original_time, initial=0)
        tau = (tau - tau.min()) / (tau.max() - tau.min()) * (T - 1)
        # Clamp monotonicity (should be increasing)
        tau = np.maximum.accumulate(tau)
        
        if x.ndim == 1:
            x_warped = np.interp(tau, np.arange(T), x)
        else:
            x_warped = np.zeros_like(x.cpu().numpy())
            for dim in range(x.shape[1]):
                x_warped[:, dim] = np.interp(tau, np.arange(T), x[:, dim].cpu().numpy())
        return torch.tensor(x_warped, dtype=x.dtype, device=x.device)
    

class ProbabilisticTransform:
    """Applies a transform with probability p."""
    def __init__(self, transform, p=0.5):
        self.transform = transform
        self.p = p

    def __call__(self, x):
        return self.transform(x) if random.random() < self.p else x


class RandomizedTransform:
    """
    Applies a random subset of the given transforms (no repetition), 
    followed optionally by a final normalization or post-processing step.
    """
    def __init__(self, transforms, always_apply_last=None, min_transforms=1, max_transforms=None):
        self.transforms = transforms
        self.always_apply_last = always_apply_last
        self.min_transforms = min_transforms
        self.max_transforms = max_transforms or len(transforms)

    def __call__(self, x):
        n = random.randint(self.min_transforms, self.max_transforms)
        transforms_to_apply = random.sample(self.transforms, k=n)
        for transform in transforms_to_apply:
            x = transform(x)
        if self.always_apply_last:
            x = self.always_apply_last(x)
        return x

# --- Normalization transforms ---

class ShiftThenZScoreNormalizeTransform:
    def __init__(self, per_channel=True, mean=None, std=None):
        """
        shift: optional tensor/array [C] to subtract before normalization
        mean: optional tensor/array [C] to use for normalization
        std: optional tensor/array [C] to use for normalization
        """
        self.per_channel = per_channel 
        self.mean = self._to_tensor(mean)
        self.std = self._to_tensor(std)

    def _to_tensor(self, x):
        if x is None:
            return None
        return torch.tensor(x, dtype=torch.float32) if not isinstance(x, torch.Tensor) else x

    def __call__(self, x):  
        if isinstance(x, np.ndarray):
            x = torch.tensor(x, dtype=torch.float32)

        if self.mean is not None and self.std is not None:
            x = (x - self.mean) / (self.std + 1e-8)
        elif self.per_channel and x.ndim > 1:
            mean = x.mean(dim=0, keepdim=True)
            std = x.std(dim=0, keepdim=True)
            x = (x - mean) / (std + 1e-8)
        else:
            mean = x.mean()
            std = x.std()
            x = (x - mean) / (std + 1e-8)

        return x

class MinMaxNormalizeTransform:
    """Normalize data to the range [-1, 1]."""
    def __call__(self, x):
        min_val = x.min()
        max_val = x.max()
        if (max_val - min_val) < 1e-6:
            return torch.zeros_like(x)
        return 2 * (x - min_val) / (max_val - min_val) - 1

class CenterScaleTransform:
    """Center and scale data to a target mean/std."""
    def __init__(self, mean=0, std=1):
        self.mean = mean
        self.std = std

    def __call__(self, x):
        x = x - x.mean()
        return self.std * x / (x.std() + 1e-8) + self.mean
    
# --- Main transform factories ---

def get_default_transforms():
    transforms = [
        ZoomingTransform(min_zoom=0.9, max_zoom=1.1),
        ScalingTransform(),
        TimeWarpingTransform(),
        LowFrequencyNoiseTransform(freq_range=(0.5, 2.0), amplitude=0.2, sample_rate=50),
    ]
    return RandomizedTransform(transforms, always_apply_last=ShiftThenZScoreNormalizeTransform())

def get_imu_transforms(mean=None, std=None):
    temporal_distortions = [
        ZoomingTransform(0.9, 1.1),
        TimeWarpingTransform(num_knots=2, mean=1.0, std=0.05),
        LowFrequencyNoiseTransform(freq_range=(0.5, 2.0), amplitude=0.03, sample_rate=50),
        ScalingTransform(mean=1.0, std=0.2, clip_min=-2, clip_max=2),
    ]
    
    motion_artifacts = [
        IMUGravityPerturbationTransform(amp=0.2, freq=0.05),
        IMUWalkingArtifactTransform(freq_range=(1.0, 2.0), amp_range=(0.1, 0.3)),
        IMUAxisMisalignmentTransform(max_angle_deg=30),
        IMURandomWalkDriftBiasTransform(max_drift_std_per_sec=0.01, max_bias=0.02),
    ] 
    
    group1 = RandomizedTransform([ProbabilisticTransform(t, p=0.5) for t in temporal_distortions])
    group2 = RandomizedTransform([ProbabilisticTransform(t, p=0.5) for t in motion_artifacts])

    def transform(x):
        x = group1(x)
        x = group2(x) 
        x = ShiftThenZScoreNormalizeTransform(mean=mean, std=std)(x) 
        return x

    return transform

def get_ppg_transforms(mean=None, std=None):
    temporal_distortions = [
        ZoomingTransform(0.9, 1.1),
        TimeWarpingTransform(num_knots=2, mean=1.0, std=0.05),
        ScalingTransform(mean=1.0, std=0.2, clip_min=-2, clip_max=2),
    ]

    physiological_artifacts = [
        PPGLowFrequencyDriftTransform(amplitude=0.05, freq=0.1, sample_rate=50),
        PPGSpikeArtifactTransform(max_spikes=3, amplitude=0.2),
        PPGRespiratoryModulationTransform(modulation_strength=0.2, freq=0.2, sample_rate=50),
    ]
    
    group1 = RandomizedTransform([ProbabilisticTransform(t, p=0.5) for t in temporal_distortions])
    group2 = RandomizedTransform([ProbabilisticTransform(t, p=0.5) for t in physiological_artifacts])

    def transform(x):
        x = group1(x)
        x = group2(x) 
        x = ShiftThenZScoreNormalizeTransform(mean=mean, std=std)(x)
        return x

    return transform

def compute_channel_stats_over_dataset(ds):
    # ds must yield (data, _, _) where data is [T, C] and has NO transforms applied
    import torch
    from torch.utils.data import DataLoader
    s = ss = None
    n = 0
    for data, *_ in DataLoader(ds, batch_size=1, shuffle=False, num_workers=0):
        x = data.squeeze(0)                     # [T, C]
        if s is None:
            s  = x.sum(dim=0)                   # [C]
            ss = (x**2).sum(dim=0)              # [C]
        else:
            s  += x.sum(dim=0)
            ss += (x**2).sum(dim=0)
        n += x.shape[0]
    mean = s / n
    var  = ss / n - mean**2
    std  = torch.sqrt(var.clamp_min(1e-8))
    return mean, std



def _apply_time_slice(x: torch.Tensor | None, time_slice):
    """Return x[start:end] on time dimension if provided, robustly clamped."""
    if x is None or time_slice is None:
        return x
    start, end = time_slice
    T = x.shape[0]
    start = 0 if start is None else max(0, min(start, T))
    end   = T if end   is None else max(start + 1, min(end, T))
    return x[start:end]


# ---- IMU filterbank helper: raw + 3 bandpasses ----
try:
    from scipy.signal import butter, sosfiltfilt
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False

def _design_imu_filterbank(fs: float):
    """
    Returns 3 bandpass SOS filters:
      0.22–8 Hz, 8–32 Hz, 32–(Nyq - epsilon)
    """
    if not _HAS_SCIPY:
        raise RuntimeError("SciPy is required for IMU filterbank (pip install scipy).")

    nyq = 0.5 * fs
    bands = [
        (0.22, 8.0),
        (8.0, 32.0),
        (32.0, nyq - 1e-3),
    ]
    sos_list = []
    for low, high in bands:
        sos = butter(4, [low / nyq, high / nyq], btype="band", output="sos")
        sos_list.append(sos)
    return sos_list

def apply_imu_filterbank_torch(imu: torch.Tensor, fs: float = 100.0) -> torch.Tensor:
    """
    imu: (T, C_imu) float32 CPU tensor
    returns: (T, C_imu * 4) = [raw, bp1, bp2, bp3] per channel
    """
    if imu is None:
        return None
    if not _HAS_SCIPY:
        raise RuntimeError("SciPy is required for IMU filterbank (pip install scipy).")

    x = imu.detach().cpu().numpy()   # (T, C)
    sos_list = _design_imu_filterbank(fs)

    outs = [x]  # raw first
    for sos in sos_list:
        y = sosfiltfilt(sos, x, axis=0)
        outs.append(y.astype(np.float32))

    x_fb = np.concatenate(outs, axis=1)  # (T, C * 4)
    return torch.from_numpy(x_fb)


In [ ]:
# ===== Dataset and Dataset Module Definition =====
class ETHGestureDataset(Dataset):
    def __init__(self, 
                 root_dir, 
                 participants, 
                 sensors=None, 
                 imu_transform=None, 
                 ppg_transform=None, 
                 selected_classes=None,                  
                 add_negative_class=False, 
                 balance_classes=False, 
                 two_stage_balancing=False,
                 random_seed=42, 
                 separate_sensor_stats=False, 
                 compute_combined_stats=True,
                 negative_class_ids=None,
                 time_slice=None,
                 # NEW
                 apply_filterbank: bool = False,
                 imu_sample_rate: float = 100.0,
                 ):
        self.root_dir = root_dir
        self.participants = participants
        self.sensors = sensors or ["PPG", "Acc", "Gyro"]
        self.imu_transform = imu_transform
        self.ppg_transform = ppg_transform
        self.selected_classes = selected_classes
        self.add_negative_class = add_negative_class
        self.balance_classes = balance_classes
        self.two_stage_balancing = two_stage_balancing
        self.random_seed = random_seed
        self.separate_sensor_stats = separate_sensor_stats
        self.compute_combined_stats = compute_combined_stats
        self.time_slice = time_slice

        # NEW
        self.apply_filterbank = apply_filterbank
        self.imu_sample_rate = imu_sample_rate

        # NEW
        self.negative_class_ids = set(negative_class_ids) if negative_class_ids is not None else None

        label_map_path = os.path.join(root_dir, "eth_label_map.json")
        if not os.path.exists(label_map_path):
            raise FileNotFoundError(f"Label map not found at {label_map_path}")
        with open(label_map_path, "r") as f:
            self.original_label_map = json.load(f)   # gesture_name -> original_class_id

        # Build remapped label map
        self.label_map = self._create_filtered_label_map()

        # Collect samples that are either positive or (if enabled) belong to the negative_class_ids set
        self.samples = self._gather_samples()

        # Optionally balance classes
        if self.balance_classes:
            print(f"Applying class balancing for {len(self.participants)} participants...")
            self.samples = self._balance_classes(self.samples)
            print(f"Class balancing complete. Final sample count: {len(self.samples)}")

    def _create_filtered_label_map(self):
        """
        Remap labels:
          - Positives: original IDs in self.selected_classes  -> 0..K-1
          - Negative: original IDs in self.negative_class_ids -> K
          - Everything else: dropped (not mapped)
        """
        if self.selected_classes is None:
            return self.original_label_map

        if self.add_negative_class and (self.negative_class_ids is None or len(self.negative_class_ids) == 0):
            raise ValueError("add_negative_class=True requires non-empty negative_class_ids.")

        id_to_names = {}
        for gname, oid in self.original_label_map.items():
            id_to_names.setdefault(oid, []).append(gname)

        filtered_map = {}
        new_id = 0

        for oid in sorted(self.selected_classes):
            if oid in id_to_names:
                for gname in id_to_names[oid]:
                    filtered_map[gname] = new_id
                new_id += 1

        if self.add_negative_class:
            neg_id = new_id
            for oid in sorted(self.negative_class_ids):
                if oid in id_to_names:
                    for gname in id_to_names[oid]:
                        filtered_map[gname] = neg_id

        return filtered_map

    def _gather_samples(self):
        samples = []
        all_samples = []

        for participant in self.participants:
            participant_path = os.path.join(self.root_dir, participant)
            if not os.path.isdir(participant_path):
                continue
            for posture in os.listdir(participant_path):
                posture_path = os.path.join(participant_path, posture)
                if not os.path.isdir(posture_path):
                    continue
                for filename in os.listdir(posture_path):
                    if not filename.endswith(".csv"):
                        continue
                    label_str = filename[:-4]
                    clean_label = re.sub(r'\(\d+\)$', '', label_str)
                    all_samples.append({
                        "path": os.path.join(posture_path, filename),
                        "label": label_str,
                        "clean_label": clean_label
                    })

        if self.selected_classes is None:
            return all_samples

        for sample in all_samples:
            clean_label = sample["clean_label"]
            if clean_label not in self.original_label_map:
                continue
            oid = self.original_label_map[clean_label]

            keep_positive = (oid in self.selected_classes)
            keep_negative = (
                self.add_negative_class and
                self.negative_class_ids is not None and
                oid in self.negative_class_ids
            )

            if keep_positive or keep_negative:
                if clean_label in self.label_map:
                    samples.append(sample)

        return samples

    def _balance_classes(self, samples):
        import random
        random.seed(self.random_seed)

        class_samples = {}
        for s in samples:
            clean = re.sub(r'\(\d+\)$', '', s['label'])
            if clean in self.label_map:
                cid = self.label_map[clean]
                class_samples.setdefault(cid, []).append(s)

        K = len(self.selected_classes) if self.selected_classes is not None else 0
        pos_sizes = []
        for rid in range(K):
            if rid in class_samples:
                pos_sizes.append(len(class_samples[rid]))
                print(f"   Debug: Remapped class {rid} has {len(class_samples[rid])} samples")

        if not pos_sizes:
            print("Warning: No positive classes found for balancing")
            return samples

        if self.two_stage_balancing:
            target_negative_size = sum(pos_sizes)
            mode = "Two-stage (negative = sum of positive)"
        else:
            target_negative_size = int(round(sum(pos_sizes) / len(pos_sizes)))
            mode = "Negative = mean(positive)"

        target_negative_size = max(1, target_negative_size)

        print(f"CLASS BALANCING ({mode}):")
        print(f"   Positive classes total: {sum(pos_sizes)} samples")
        print(f"   Target negative size: {target_negative_size} samples")

        balanced = []
        neg_id = K if self.add_negative_class else None

        for cid, slist in class_samples.items():
            if cid == neg_id:
                orig = len(slist)
                if orig > target_negative_size:
                    slist = random.sample(slist, target_negative_size)
                    print(f"   Negative class: {orig} -> {target_negative_size} (downsampled)")
                else:
                    print(f"   Negative class: {orig} (no downsampling)")
                balanced.extend(slist)
            else:
                print(f"   Class {cid}: {len(slist)} (kept all)")
                balanced.extend(slist)

        print(f"   Total after balancing: {len(balanced)} samples")
        print(f"   Reduction: {len(samples)} -> {len(balanced)} ({len(balanced)/len(samples)*100:.1f}%)")
        return balanced

    def __len__(self):
        return len(self.samples)

    def get_label_list(self):
        labels = []
        for s in self.samples:
            clean = re.sub(r'\(\d+\)$', '', s['label'])
            labels.append(self.label_map[clean])
        return labels

# =========================
# Sliding-window integration
# =========================
class SlidingWindowETHDataset(Dataset):
    """
    Wraps an ETHGestureDataset and returns many fixed-length windows per original sample.
    Windows are sliced BEFORE transforms (so normalization/augs see the right window).
    """
    def __init__(
        self,
        base_ds: ETHGestureDataset,
        start: int = 70,
        end: int = 270,             # inclusive limit for window END
        window: int = 100,
        stride_samples: int | None = None,
        pace_hz: float | None = 8.0,
        sample_rate: float = 50.0,  # <-- set to 100.0 if your data is 100 Hz
    ):
        super().__init__()
        assert isinstance(base_ds, ETHGestureDataset), "base_ds must be ETHGestureDataset"
        self.base = base_ds
        self.window = int(window)

        # Derive stride from pace_hz if not given: stride_samples = fs / pace
        if stride_samples is None:
            if pace_hz is None:
                raise ValueError("Provide either stride_samples or pace_hz.")
            stride_samples = max(1, int(round(sample_rate / pace_hz)))
        self.stride = int(stride_samples)

        # Build per-sample start offsets (we assume clips are at least `end` long; your spec says 300)
        last_start = int(end) - self.window
        if last_start < start:
            raise ValueError(f"Window({window}) does not fit between start={start} and end={end}.")
        self.starts = list(range(int(start), last_start + 1, self.stride))

        # Flat index over (original_sample_idx, start_offset)
        self._index = [(i, s) for i in range(len(self.base)) for s in self.starts]

        # --- Mirror a few frequently-used attributes (optional but nice) ---
        self.label_map              = getattr(self.base, "label_map", None)
        self.original_label_map     = getattr(self.base, "original_label_map", None)
        self.sensors                = getattr(self.base, "sensors", None)
        self.separate_sensor_stats  = getattr(self.base, "separate_sensor_stats", False)
        self.compute_combined_stats = getattr(self.base, "compute_combined_stats", True)
        self.imu_transform          = getattr(self.base, "imu_transform", None)
        self.ppg_transform          = getattr(self.base, "ppg_transform", None)
        # NEW: filterbank settings mirrored from base dataset
        self.apply_filterbank       = getattr(self.base, "apply_filterbank", False)
        self.imu_sample_rate        = getattr(self.base, "imu_sample_rate", 100.0)
        # Quick status
        print(
            f"Sliding windows: start={start}, end={end}, window={window}, stride={self.stride} "
            f"-> {len(self.starts)} windows/clip, total items={len(self)}"
        )

    def __len__(self):
        return len(self._index)

    def get_label_list(self):
        """Repeat each base label for every start offset so samplers keep working."""
        base_labels = self.base.get_label_list()
        return [lbl for lbl in base_labels for _ in self.starts]

    # --- Passthrough for anything else ---
    def __getattr__(self, name):
        # Only called if attr not found on self
        base = super().__getattribute__("base")
        if hasattr(base, name):
            return getattr(base, name)
        raise AttributeError(f"{type(self).__name__} has no attribute '{name}'")

    def __getitem__(self, idx):
        base_idx, s = self._index[idx]
        sample_info = self.base.samples[base_idx]

        # --- Read CSV and pick sensors exactly like base ---
        df = pd.read_csv(sample_info["path"])

        imu_cols, ppg_cols = [], []
        if "Acc" in self.base.sensors:  imu_cols += ["AccX", "AccY", "AccZ"]
        if "Gyro" in self.base.sensors: imu_cols += ["GyroX", "GyroY", "GyroZ"]
        if "PPG" in self.base.sensors:  ppg_cols += [c for c in df.columns if c.startswith("PPG")]

        imu_full = torch.tensor(df[imu_cols].values.astype(np.float32)) if imu_cols else None
        ppg_full = torch.tensor(df[ppg_cols].values.astype(np.float32)) if ppg_cols else None

        e = s + self.window
        imu_raw = imu_full[s:e] if imu_full is not None else None
        ppg_raw = ppg_full[s:e] if ppg_full is not None else None

        # --- Apply the same transforms the base dataset would (no filterbank yet) ---
        imu_model = imu_raw
        ppg_model = ppg_raw
        if imu_model is not None and self.base.imu_transform:
            imu_model = self.base.imu_transform(imu_model)
        if ppg_model is not None and self.base.ppg_transform:
            ppg_model = self.base.ppg_transform(ppg_model)

        # --- CNN input: optional IMU filterbank ---
        if self.apply_filterbank and imu_model is not None:
            imu_for_cnn = apply_imu_filterbank_torch(imu_model, fs=self.imu_sample_rate)
        else:
            imu_for_cnn = imu_model

        data_parts = []
        if ppg_model is not None:
            data_parts.append(ppg_model)
        if imu_for_cnn is not None:
            data_parts.append(imu_for_cnn)
        if not data_parts:
            raise RuntimeError("No sensors enabled, data is empty.")
        data = torch.cat(data_parts, dim=1)

        # --- Stats: always from raw (normalized) channels, never filterbank ---
        if self.base.separate_sensor_stats and not self.base.compute_combined_stats:
            parts = []
            if ppg_model is not None:
                parts.append(get_normalize_statistical_features(ppg_model))
            if imu_model is not None:
                parts.append(get_normalize_statistical_features(imu_model))
            data_stats = torch.cat(parts, dim=1) if len(parts) > 1 else parts[0]

        elif self.base.compute_combined_stats and not self.base.separate_sensor_stats:
            stats_inputs = []
            if ppg_model is not None:
                stats_inputs.append(ppg_model)
            if imu_model is not None:
                stats_inputs.append(imu_model)
            if not stats_inputs:
                data_stats = torch.zeros((data.shape[0], 1))
            else:
                combo = stats_inputs[0] if len(stats_inputs) == 1 else torch.cat(stats_inputs, dim=1)
                data_stats = get_normalize_statistical_features(combo)

        elif self.base.separate_sensor_stats and self.base.compute_combined_stats:
            parts = []
            if ppg_model is not None:
                parts.append(get_normalize_statistical_features(ppg_model))
            if imu_model is not None:
                parts.append(get_normalize_statistical_features(imu_model))

            stats_inputs = []
            if ppg_model is not None:
                stats_inputs.append(ppg_model)
            if imu_model is not None:
                stats_inputs.append(imu_model)
            if stats_inputs:
                combo = stats_inputs[0] if len(stats_inputs) == 1 else torch.cat(stats_inputs, dim=1)
                parts.append(get_normalize_statistical_features(combo))

            data_stats = torch.cat(parts, dim=1)
        else:
            data_stats = torch.zeros((data.shape[0], 1))

        clean_label = re.sub(r'\(\d+\)$', '', sample_info["label"])
        label_id = self.base.label_map[clean_label]
        return data, data_stats, label_id


# ===== Extend DataModule to support sliding windows =====
class ETHGestureDataModule(LightningDataModule):
    def __init__(self, 
                 batch_size=32, sensors=None, participants_split=None, num_workers=None,
                 selected_classes=None, add_negative_class=False, balance_classes=False, balance_validation=False,
                 balance_test=False, two_stage_balancing=False, random_seed=42, separate_sensor_stats=False,
                 compute_combined_stats=True,
                 negative_class_ids=None,
                 # (keep existing phase params but unused here)
                 phase_label_mode=None, balance_composite=False,
                 pre_len=100, gest_len=100, post_len=100,
                 enable_sliding_pre=False, pre_shift_max=50, pre_step=5,
                 enable_sliding_gesture=False, gesture_shift_max=0, gesture_step=5,
                 enable_sliding_post=False, post_shift_max=50, post_step=5,
                 time_slice=None,
                 # NEW: sliding window dataset expansion
                 enable_sliding_windows: bool = True,
                 slide_start: int = 70,
                 slide_end: int = 270,
                 slide_window: int = 100,
                 slide_stride_samples: int | None = None,
                 slide_pace_hz: float | None = 8.0,
                 sample_rate: float = 50.0,
                 # NEW: IMU filterbank toggle
                 enable_imu_filterbank: bool = False,
                 ):
        super().__init__()
        self.batch_size = batch_size
        use_prior = True
        self.dataset_path = os.path.join(os.getenv("DATASET_PATH"), os.getenv("ETH_SUBPATH"))

        self.ppg_mean, self.imu_mean, self.ppg_std, self.imu_std = None, None, None, None
        if os.path.exists(os.path.join(self.dataset_path, 'saved_stats')) and use_prior:
            self.ppg_mean, self.imu_mean, self.ppg_std, self.imu_std = load_separate_stats(os.path.join(self.dataset_path, 'saved_stats'))

        self.sensors = sensors or ["PPG", "Acc", "Gyro"]
        self.participants_split = participants_split or {
            'train': ['id_1','id_3','id_4','id_5','id_6','id_7','id_8','id_10','id_11','id_12','id_13','id_14','id_16'],
            'val':   ['id_8','id_2'],
            'test':  ['id_17']
        }
        self.num_workers = num_workers if num_workers is not None else os.cpu_count()

        self.selected_classes = selected_classes
        self.add_negative_class = add_negative_class
        self.balance_classes = balance_classes
        self.balance_validation = balance_validation
        self.balance_test = balance_test
        self.two_stage_balancing = two_stage_balancing
        self.random_seed = random_seed

        # We won't use a fixed time_slice when sliding windows are enabled
        self.time_slice = None if enable_sliding_windows else time_slice

        self.separate_sensor_stats = separate_sensor_stats
        self.compute_combined_stats = compute_combined_stats
        self.negative_class_ids = set(negative_class_ids) if negative_class_ids is not None else None

        # New sliding config
        self.enable_sliding_windows = enable_sliding_windows
        self.slide_start = slide_start
        self.slide_end = slide_end
        self.slide_window = slide_window
        self.slide_stride_samples = slide_stride_samples
        self.slide_pace_hz = slide_pace_hz
        self.sample_rate = sample_rate

        self.enable_imu_filterbank = enable_imu_filterbank

        # keep phase params untouched (not used here)
        self.phase_label_mode = phase_label_mode
        self.balance_composite = balance_composite
        self.pre_len = pre_len; self.gest_len = gest_len; self.post_len = post_len
        self.enable_sliding_pre = enable_sliding_pre; self.pre_shift_max = pre_shift_max; self.pre_step = pre_step
        self.enable_sliding_gesture = enable_sliding_gesture; self.gesture_shift_max = gesture_shift_max; self.gesture_step = gesture_step
        self.enable_sliding_post = enable_sliding_post; self.post_shift_max = post_shift_max; self.post_step = post_step

    def _maybe_wrap(self, ds: ETHGestureDataset):
        if not self.enable_sliding_windows:
            return ds
        return SlidingWindowETHDataset(
            ds,
            start=self.slide_start,
            end=self.slide_end,
            window=self.slide_window,
            stride_samples=self.slide_stride_samples,
            pace_hz=self.slide_pace_hz,
            sample_rate=self.sample_rate,
        )

    def setup(self, stage=None):
        # ---- 1) Compute mu/sigma over the exact windows that we train on ----
        if self.ppg_mean is None or self.ppg_std is None or self.imu_mean is None or self.imu_std is None:
            imu_raw = ETHGestureDataset(
                self.dataset_path, self.participants_split["train"], ["Acc","Gyro"],
                imu_transform=None, ppg_transform=None,
                selected_classes=self.selected_classes,
                add_negative_class=self.add_negative_class,
                balance_classes=False,
                separate_sensor_stats=False, compute_combined_stats=False,
                negative_class_ids=self.negative_class_ids,
                time_slice=None,   # compute on full and slice via wrapper
            )
            ppg_raw = ETHGestureDataset(
                self.dataset_path, self.participants_split["train"], ["PPG"],
                imu_transform=None, ppg_transform=None,
                selected_classes=self.selected_classes,
                add_negative_class=self.add_negative_class,
                balance_classes=False,
                separate_sensor_stats=False, compute_combined_stats=False,
                negative_class_ids=self.negative_class_ids,
                time_slice=None,
            )
            imu_for_stats = self._maybe_wrap(imu_raw)
            ppg_for_stats = self._maybe_wrap(ppg_raw)
            self.imu_mean, self.imu_std = compute_channel_stats_over_dataset(imu_for_stats)
            self.ppg_mean, self.ppg_std = compute_channel_stats_over_dataset(ppg_for_stats)

        # ---- 2) Define transforms ----
        imu_train_tf = ShiftThenZScoreNormalizeTransform(mean=self.imu_mean, std=self.imu_std)
        ppg_train_tf = ShiftThenZScoreNormalizeTransform(mean=self.ppg_mean, std=self.ppg_std)
        imu_eval_tf  = ShiftThenZScoreNormalizeTransform(mean=self.imu_mean, std=self.imu_std)
        ppg_eval_tf  = ShiftThenZScoreNormalizeTransform(mean=self.ppg_mean, std=self.ppg_std)

        common = {
            'selected_classes': self.selected_classes,
            'add_negative_class': self.add_negative_class,
            'random_seed': self.random_seed,
            'separate_sensor_stats': self.separate_sensor_stats,
            'compute_combined_stats': self.compute_combined_stats,
            'negative_class_ids': self.negative_class_ids,
            'time_slice': None if self.enable_sliding_windows else self.time_slice,
            # NEW: tell ETHGestureDataset to apply IMU filterbank
            'apply_filterbank': self.enable_imu_filterbank,
            'imu_sample_rate': self.sample_rate,
        }

        # ---- 3) Train / Val / Test datasets ----
        self.train_dataset = self._maybe_wrap(ETHGestureDataset(
            self.dataset_path, self.participants_split["train"], self.sensors,
            imu_transform=imu_train_tf, ppg_transform=ppg_train_tf,
            balance_classes=self.balance_classes,
            **common
        ))

        val_bal = self.balance_validation if self.phase_label_mode is None else False
        self.val_dataset = self._maybe_wrap(ETHGestureDataset(
            self.dataset_path, self.participants_split["val"], self.sensors,
            imu_transform=imu_eval_tf, ppg_transform=ppg_eval_tf,
            balance_classes=val_bal,
            **common
        ))

        test_bal = self.balance_test if self.phase_label_mode is None else False
        self.test_dataset = self._maybe_wrap(ETHGestureDataset(
            self.dataset_path, self.participants_split["test"], self.sensors,
            imu_transform=imu_eval_tf, ppg_transform=ppg_eval_tf,
            balance_classes=test_bal,
            **common
        ))

        # Small sanity print
        if self.enable_sliding_windows:
            last_start = self.slide_end - self.slide_window
            stride = self.slide_stride_samples if self.slide_stride_samples is not None else max(
                1, int(round(self.sample_rate / self.slide_pace_hz))
            )
            n_per_clip = (last_start - self.slide_start)//stride + 1
            print(f"Expected windows/clip: {n_per_clip}  (stride={stride} samples at fs={self.sample_rate})")

    def train_dataloader(self):
        labels = self.train_dataset.get_label_list()
        counts = Counter(labels)
        class_weights = {c: 1.0 / max(1, n) for c, n in counts.items()}
        sample_weights = [class_weights[l] for l in labels]
        sampler = WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)
        return DataLoader(self.train_dataset, batch_size=self.batch_size, sampler=sampler, num_workers=self.num_workers)

    def val_dataloader(self):
        return DataLoader(self.val_dataset, batch_size=self.batch_size, num_workers=self.num_workers)

    def test_dataloader(self):
        return DataLoader(self.test_dataset, batch_size=self.batch_size, num_workers=self.num_workers)


In [ ]:
SELECTED_CLASSES    = [3, 4, 13, 16, 20] #clench0 double_clench3 double_pinch4 pinch_down13 pinch_left14 pinch_right15 pinch_up16 rotate_in18 rotate_out19 slide20 snap21
NEGATIVE_CLASS_IDS = {
    27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
    43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58
}

ADD_NEGATIVE_CLASS  = True
BALANCE_CLASSES     = False
BALANCE_VALIDATION  = False
ENABLE_SLIDING_VALIDATION = True
SENSORS             = ["Acc", "Gyro"]  

# ---- Training defaults (you can override per-model below) ----
BATCH_SIZE   = 16
MONITOR  = "val_macro_f1"  # "val_macro_f1" | "val_weighted_f1" | "val_acc" | "val_loss"
MODE     = "max"

dm = ETHGestureDataModule(
    batch_size=BATCH_SIZE,
    sensors=SENSORS,
    participants_split=participants_split,
    num_workers=0,
    selected_classes=SELECTED_CLASSES,
    add_negative_class=ADD_NEGATIVE_CLASS,
    negative_class_ids=NEGATIVE_CLASS_IDS,
    balance_classes=BALANCE_CLASSES,
    balance_validation=BALANCE_VALIDATION,
    separate_sensor_stats=True,
    compute_combined_stats=False,

    enable_sliding_windows=ENABLE_SLIDING_VALIDATION,
    slide_start=70,
    slide_end=270,          # last window ends at 270
    slide_window=100,
    slide_pace_hz=5,      # pace in Hz
    slide_stride_samples=None,
    sample_rate=100.0,       
    enable_imu_filterbank=False,
)

dm_apple = ETHGestureDataModule(
    batch_size=BATCH_SIZE,
    sensors=SENSORS,
    participants_split=participants_split,
    num_workers=0,
    selected_classes=SELECTED_CLASSES,
    add_negative_class=ADD_NEGATIVE_CLASS,
    negative_class_ids=NEGATIVE_CLASS_IDS,
    balance_classes=BALANCE_CLASSES,
    balance_validation=BALANCE_VALIDATION,
    separate_sensor_stats=True,
    compute_combined_stats=False,

    enable_sliding_windows=ENABLE_SLIDING_VALIDATION,
    slide_start=70,
    slide_end=270,          # last window ends at 270
    slide_window=100,
    slide_pace_hz=5,      # pace in Hz
    slide_stride_samples=None,
    sample_rate=100.0,       
    enable_imu_filterbank=True,
)

dm_transformer = ETHGestureDataModule(
    batch_size=BATCH_SIZE,
    sensors=SENSORS,
    participants_split=participants_split,
    num_workers=0,
    selected_classes=SELECTED_CLASSES,
    add_negative_class=ADD_NEGATIVE_CLASS,
    negative_class_ids=NEGATIVE_CLASS_IDS,
    balance_classes=BALANCE_CLASSES,
    balance_validation=BALANCE_VALIDATION,
    separate_sensor_stats=False,
    compute_combined_stats=True,

    enable_sliding_windows=True,
    slide_start=70,
    slide_end=270,          # last window ends at 270
    slide_window=100,
    slide_pace_hz=5,      # pace in Hz
    slide_stride_samples=None,
    sample_rate=100.0,       
    enable_imu_filterbank=True,
)

dm.setup()
dm_apple.setup()
dm_transformer.setup()

train_loader = dm.train_dataloader()
val_loader   = dm.val_dataloader()
test_loader  = dm.test_dataloader()

train_loader = dm_apple.train_dataloader()
val_loader   = dm_apple.val_dataloader()
test_loader  = dm_apple.test_dataloader()

train_loader = dm_transformer.train_dataloader()
val_loader   = dm_transformer.val_dataloader()
test_loader  = dm_transformer.test_dataloader()

## Normwear Model

In [ ]:
# ==== NormWear Base (Frozen) + Head Training with Pre-computed Embeddings ====

# ---- NormWear BASE run dir ----
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
NORMWEAR_DIR = os.path.join(OUTPUTS, "normwear_base", f"run_{RUN_TAG}")
os.makedirs(NORMWEAR_DIR, exist_ok=True)
print("Output directory:", OUTPUTS)
print("NormWear BASE run directory:", NORMWEAR_DIR)

def save_text(filename: str, text: str) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    with open(path, "w", encoding='utf-8') as f:
        f.write(text)
    return path

def save_json(filename: str, obj) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    with open(path, "w", encoding='utf-8') as f:
        json.dump(obj, f, indent=2)
    return path

def save_npy(filename: str, arr: np.ndarray) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    np.save(path, arr)
    return path

def save_fig(filename: str, dpi: int = 200) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path


# =======================
# Config
# =======================
SEED = 42
SAMPLING_RATE = 100.0
NORMWEAR_ENCODER_CKPT = "NormWear/normwear_last_checkpoint-15470-correct.pth"

BATCH = 64
EPOCHS = 100
PATIENCE = 8

PATCH_POOL = "mean+max"
CHANNEL_POOL = "max"

LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-3

# Exponential LR scheduler
DECAY_TARGET = 0.01
GAMMA = float(math.exp(math.log(DECAY_TARGET) / max(1, int(EPOCHS))))

# Clip-level aggregation config
CLIP_K_MAP = None
CLIP_DEFAULT_K = 3

idx_to_name = {
    0: "double_clench",
    1: "double_pinch",
    2: "pinch_down",
    3: "pinch_up",
    4: "slide",
}
NEG_NAME = "Negative"
NEG_ID_EXPECTED = 5

run_config = {
    "SEED": SEED,
    "SAMPLING_RATE": SAMPLING_RATE,
    "NORMWEAR_ENCODER_CKPT": NORMWEAR_ENCODER_CKPT,
    "BATCH": BATCH,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "MODE": "BASE_ONLY",
    "POOLING": {"PATCH_POOL": PATCH_POOL, "CHANNEL_POOL": CHANNEL_POOL},
    "OPT": {"LR_HEAD": LR_HEAD, "WEIGHT_DECAY": WEIGHT_DECAY, "DECAY_TARGET": DECAY_TARGET, "GAMMA": GAMMA},
    "CLIP": {"DEFAULT_K": CLIP_DEFAULT_K, "K_MAP": CLIP_K_MAP},
}
save_json("config.json", run_config)


# =======================
# Reset + seeds
# =======================
for name in ["model", "optimizer", "scheduler", "history", "best_state"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =======================
# Dataloaders from dm
# =======================
train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

x0, _, y0 = next(iter(train_loader))
T, C = x0.shape[1], x0.shape[2]

train_labels = torch.tensor(dm.train_dataset.get_label_list(), dtype=torch.long)
n_classes = int(train_labels.max().item()) + 1

print(f"Window shape: [T={T}, C={C}], n_classes={n_classes}")

uniq = torch.unique(train_labels).cpu().tolist()
print(f"Train labels range: [{int(train_labels.min())}, {int(train_labels.max())}], unique={len(uniq)}")
assert train_labels.min().item() >= 0, "Found negative label id; labels must be >= 0 for CrossEntropyLoss."
assert max(uniq) == n_classes - 1, "Non-contiguous labels detected; expected 0..n_classes-1."


# =======================
# Build NormWear + load checkpoint (FROZEN)
# =======================
normwear = NormWearModel(weight_path="", optimized_cwt=True).to(device)

ckpt = torch.load(NORMWEAR_ENCODER_CKPT, map_location="cpu", weights_only=False)
if isinstance(ckpt, dict) and "model" in ckpt:
    state_dict = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

missing, unexpected = normwear.backbone.load_state_dict(state_dict, strict=False)
print("Loaded NormWear pretrained backbone.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

# Freeze entire backbone
for p in normwear.backbone.parameters():
    p.requires_grad = False

normwear.eval()


# =======================
# Embedding aggregation
# =======================
def aggregate_normwear_embeddings(
    out: torch.Tensor,
    patch_pool: str = "mean+max",
    channel_pool: str = "mean+max",
) -> torch.Tensor:
    B, Cc, P, D = out.shape

    if patch_pool == "mean":
        emb = out.mean(dim=2)
    elif patch_pool == "max":
        emb, _ = out.max(dim=2)
    elif patch_pool == "mean+max":
        mean_p = out.mean(dim=2)
        max_p, _ = out.max(dim=2)
        emb = torch.cat([mean_p, max_p], dim=-1)
    elif patch_pool == "cls":
        emb = out[:, :, 0, :]
    elif patch_pool == "softmax-norm":
        weights = out.norm(dim=-1)
        weights = torch.softmax(weights, dim=-1)[..., None]
        emb = (out * weights).sum(dim=2)
    else:
        raise ValueError(f"Unknown patch_pool: {patch_pool}")

    if channel_pool == "mean":
        emb = emb.mean(dim=1)
    elif channel_pool == "max":
        emb, _ = emb.max(dim=1)
    elif channel_pool == "mean+max":
        mean_c = emb.mean(dim=1)
        max_c, _ = emb.max(dim=1)
        emb = torch.cat([mean_c, max_c], dim=-1)
    elif channel_pool == "concat":
        emb = emb.reshape(B, -1)
    elif channel_pool == "softmax-norm":
        weights = emb.norm(dim=-1)
        weights = torch.softmax(weights, dim=1)[..., None]
        emb = (emb * weights).sum(dim=1)
    else:
        raise ValueError(f"Unknown channel_pool: {channel_pool}")

    return emb


# Infer embedding dimension
with torch.no_grad():
    x0_nw = x0.permute(0, 2, 1).contiguous().to(device)
    spec0 = normwear.calc_cwt(x0_nw, device=device).float()
    fn = normwear.backbone.get_signal_embedding
    if hasattr(fn, "__wrapped__"):
        out0 = fn.__wrapped__(normwear.backbone, spec0, hidden_out=False, device=device)
    else:
        out0 = fn(spec0, hidden_out=False, device=device)
    emb0 = aggregate_normwear_embeddings(out0, PATCH_POOL, CHANNEL_POOL)

in_dim = emb0.shape[1]
print("Embedding dim from NormWear (after pooling):", in_dim)


# =======================
# PRE-COMPUTE EMBEDDINGS FOR ALL SPLITS
# =======================
print("\n" + "=" * 60)
print("PRE-COMPUTING EMBEDDINGS (FROZEN BACKBONE)")
print("=" * 60)

@torch.no_grad()
def extract_embeddings_from_loader(loader, desc="Extract"):
    """Extract embeddings for entire dataset."""
    normwear.eval()
    all_embeddings = []
    all_labels = []
    
    for xb, _, yb in tqdm(loader, desc=desc, leave=False):
        xb = xb.to(device)
        
        # NormWear forward
        x_nw = xb.permute(0, 2, 1).contiguous()
        spec = normwear.calc_cwt(x_nw, device=device).float()
        
        fn = normwear.backbone.get_signal_embedding
        if hasattr(fn, "__wrapped__"):
            out = fn.__wrapped__(normwear.backbone, spec, hidden_out=False, device=device)
        else:
            out = fn(spec, hidden_out=False, device=device)
        
        # Aggregate
        emb = aggregate_normwear_embeddings(out, PATCH_POOL, CHANNEL_POOL)
        
        all_embeddings.append(emb.cpu())
        all_labels.append(yb.cpu())
    
    embeddings = torch.cat(all_embeddings, dim=0)
    labels = torch.cat(all_labels, dim=0)
    
    return embeddings, labels

# Extract for all splits
train_embeddings, train_labels_full = extract_embeddings_from_loader(train_loader, desc="Extract Train")
val_embeddings, val_labels_full = extract_embeddings_from_loader(val_loader, desc="Extract Val")
test_embeddings, test_labels_full = extract_embeddings_from_loader(test_loader, desc="Extract Test")

print(f"Train embeddings: {train_embeddings.shape}, labels: {train_labels_full.shape}")
print(f"Val embeddings:   {val_embeddings.shape}, labels: {val_labels_full.shape}")
print(f"Test embeddings:  {test_embeddings.shape}, labels: {test_labels_full.shape}")

# Save embeddings to disk
save_npy("train_embeddings.npy", train_embeddings.numpy())
save_npy("train_labels.npy", train_labels_full.numpy())
save_npy("val_embeddings.npy", val_embeddings.numpy())
save_npy("val_labels.npy", val_labels_full.numpy())
save_npy("test_embeddings.npy", test_embeddings.numpy())
save_npy("test_labels.npy", test_labels_full.numpy())


# =======================
# Create TensorDatasets from embeddings
# =======================
from torch.utils.data import TensorDataset, DataLoader

train_dataset_emb = TensorDataset(train_embeddings, train_labels_full)
val_dataset_emb = TensorDataset(val_embeddings, val_labels_full)
test_dataset_emb = TensorDataset(test_embeddings, test_labels_full)


# =======================
# Residual MLP head
# =======================
class ResidualMLPClassifier(nn.Module):
    def __init__(self, in_dim: int, n_classes: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


# =======================
# Build model (HEAD ONLY)
# =======================
model = ResidualMLPClassifier(in_dim=in_dim, n_classes=n_classes).to(device)


# =======================
# (ADDED) Full pipeline wrapper for benchmarking
# =======================
class NormWearBaseFullPipeline(nn.Module):
    """Wrapper for full inference: NormWear + Head (for benchmarking)"""
    def __init__(self, normwear_model, head, patch_pool="mean+max", channel_pool="max"):
        super().__init__()
        self.normwear = normwear_model
        self.head = head
        self.patch_pool = patch_pool
        self.channel_pool = channel_pool
    
    def forward(self, x):
        # x: [B, T, C]
        x_nw = x.permute(0, 2, 1).contiguous()  # [B, C, T]
        spec = self.normwear.calc_cwt(x_nw, device=x.device).float()
        
        fn = self.normwear.backbone.get_signal_embedding
        if hasattr(fn, "__wrapped__"):
            out = fn.__wrapped__(self.normwear.backbone, spec, hidden_out=False, device=x.device)
        else:
            out = fn(spec, hidden_out=False, device=x.device)
        
        # Aggregate
        emb = aggregate_normwear_embeddings(out, self.patch_pool, self.channel_pool)
        return self.head(emb)


# =======================
# Parameter stats
# =======================
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_backbone_total = sum(p.numel() for p in normwear.backbone.parameters())
n_backbone_train = 0

param_text = (
    "\n" + "=" * 60 + "\n"
    "PARAMETER STATISTICS (BASE ONLY - NO LORA)\n"
    + "=" * 60 + "\n"
    f"Head params (total):       {n_total:,}\n"
    f"Head params (trainable):   {n_trainable:,}\n"
    f"Backbone params (frozen):  {n_backbone_total:,}\n"
    f"Total system params:       {n_backbone_total + n_total:,}\n"
    f"Trainable %:               {100*n_trainable/(n_backbone_total + n_total):.4f}%\n"
    + "=" * 60 + "\n"
)
print(param_text)
save_text("parameter_stats.txt", param_text)

param_summary = {
    "model_total_params": int(n_total),
    "model_trainable_params": int(n_trainable),
    "backbone_total_params": int(n_backbone_total),
    "backbone_trainable_params": int(n_backbone_train),
    "head_trainable_params": int(n_trainable),
    "lora_params": 0,
    "total_system_params": int(n_backbone_total + n_total),
    "trainable_percent": float(100.0 * n_trainable / max(1, n_backbone_total + n_total)),
}
save_json("parameter_summary.json", param_summary)


# =======================
# Class weights & sampler
# =======================
class_counts = torch.bincount(train_labels_full, minlength=n_classes).float()
class_weights = 1.0 / (class_counts + 1e-8)
class_weights = class_weights * (n_classes / class_weights.sum())

save_json("train_class_counts.json", {"counts": class_counts.tolist()})
save_json("train_class_weights.json", {"weights": class_weights.tolist()})

sample_weights = class_weights[train_labels_full]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader_emb = DataLoader(
    train_dataset_emb,
    batch_size=BATCH,
    sampler=sampler,
    num_workers=0,
)

val_loader_emb = DataLoader(val_dataset_emb, batch_size=BATCH, shuffle=False, num_workers=0)
test_loader_emb = DataLoader(test_dataset_emb, batch_size=BATCH, shuffle=False, num_workers=0)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.05,
)


# =======================
# Optimizer + scheduler
# =======================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(LR_HEAD),
    weight_decay=float(WEIGHT_DECAY)
)

scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=float(GAMMA))


# =======================
# STANDARDIZED CONFUSION MATRIX PLOTTING
# =======================
def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str,
    annot: bool = True,
    annot_fmt: str = ".2f",
    dpi: int = 200,
):
    """Plot row-normalized confusion matrix (0-1 scale)"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_norm01.npy"), cm_norm.astype(np.float32))
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)

    return cm_norm, cm_counts

def plot_conf_mat_counts(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str = "Blues",
    annot: bool = True,
    dpi: int = 200,
    max_annot: int = 9999,
):
    """Plot count-based confusion matrix"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_counts,
        annot=annot,
        fmt="d",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
        vmin=0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    if annot and max_annot is not None:
        for t in ax.texts:
            try:
                v = int(t.get_text())
                if v > int(max_annot):
                    t.set_text(f">{max_annot}")
            except Exception:
                pass

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)
    return cm_counts


# =======================
# COMPREHENSIVE METRICS HELPERS
# =======================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out

def per_class_f1(y_true, y_pred, n_classes_local: int, zero_division=0):
    """Compute per-class precision, recall, F1, and support"""
    labels = np.arange(int(n_classes_local))
    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=zero_division
    )
    return {
        "precision": [float(v) for v in p],
        "recall":    [float(v) for v in r],
        "f1":        [float(v) for v in f1],
        "support":   [int(v) for v in s],
    }

def _cuda_sync_if_needed(dev: torch.device):
    """Synchronize CUDA if needed for accurate timing"""
    if str(dev).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()


# =======================
# ENHANCED EVALUATION WITH TIMING + COMPREHENSIVE METRICS
# =======================
@torch.no_grad()
def evaluate_comprehensive(loader, desc="Eval", warmup_batches=5):
    """
    Comprehensive evaluation with timing and all metrics.
    Operates on pre-computed embeddings.
    Returns: metrics dict, y_true, y_pred
    """
    model.eval()
    total_loss = 0.0
    preds_all, targs_all = [], []
    n_seen = 0
    
    times = []
    did_sync_after_warmup = False
    
    for bi, (emb, yb) in enumerate(tqdm(loader, desc=desc, leave=False)):
        emb, yb = emb.to(device), yb.to(device)
        
        do_time = (bi >= int(warmup_batches))
        
        if do_time and (not did_sync_after_warmup):
            _cuda_sync_if_needed(device)
            did_sync_after_warmup = True
        
        if do_time:
            t0 = time.perf_counter()
            logits = model(emb)
            _cuda_sync_if_needed(device)
            t1 = time.perf_counter()
            times.append((t1 - t0))
        else:
            logits = model(emb)
        
        loss = criterion(logits, yb)
        total_loss += float(loss.item()) * emb.size(0)
        
        preds = torch.argmax(logits, dim=1).detach().cpu()
        targs_all.append(yb.detach().cpu())
        preds_all.append(preds)
        n_seen += int(yb.size(0))
    
    y_pred = torch.cat(preds_all).numpy() if preds_all else np.array([], dtype=int)
    y_true = torch.cat(targs_all).numpy() if targs_all else np.array([], dtype=int)
    
    loss_avg = total_loss / max(1, n_seen)
    
    if y_true.size:
        acc = float((y_pred == y_true).mean())
        prf = prf_all_averages(y_true, y_pred, zero_division=0)
        per_cls = per_class_f1(y_true, y_pred, n_classes_local=int(n_classes), zero_division=0)
    else:
        acc = float("nan")
        prf = {
            "macro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "micro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "weighted": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
        }
        per_cls = {"precision": [], "recall": [], "f1": [], "support": []}
    
    # Timing stats (head-only inference)
    if len(times) > 0 and n_seen > 0:
        total_t = float(np.sum(times))
        ms_per_batch = float(1000.0 * np.mean(times))
        ms_per_sample = float(1000.0 * (total_t / float(n_seen)))
        n_timed_batches = int(len(times))
    else:
        total_t = 0.0
        ms_per_batch = float("nan")
        ms_per_sample = float("nan")
        n_timed_batches = 0
    
    timing = {
        "warmup_batches": int(warmup_batches),
        "timed_batches": int(n_timed_batches),
        "timed_total_seconds": float(total_t),
        "ms_per_batch": float(ms_per_batch),
        "ms_per_sample": float(ms_per_sample),
        "note": "Timing is HEAD-ONLY inference on pre-computed embeddings",
    }
    
    metrics = {
        "loss": float(loss_avg),
        "acc": float(acc),
        "prf": prf,
        "per_class": per_cls,
        "timing": timing,
    }
    
    return metrics, y_true, y_pred


# =======================
# TRAINING LOOP WITH COMPREHENSIVE METRICS
# =======================
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1_macro": [],
    "train_f1_micro": [],
    "train_f1_weighted": [],
    "val_f1_macro": [],
    "val_f1_micro": [],
    "val_f1_weighted": [],
    "train_precision_macro": [],
    "train_recall_macro": [],
    "val_precision_macro": [],
    "val_recall_macro": [],
    "lr": [],
}

best_state = None
best_f1 = -1.0
no_improve = 0

print("\n" + "=" * 60)
print("STARTING TRAINING (HEAD ONLY, PRE-COMPUTED EMBEDDINGS)")
print("=" * 60 + "\n")

for epoch in range(1, int(EPOCHS) + 1):
    model.train()
    total_loss = 0.0
    all_preds, all_targs = [], []

    pbar = tqdm(train_loader_emb, desc=f"Epoch {epoch}/{int(EPOCHS)}", leave=False)
    for emb, yb in pbar:
        emb, yb = emb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(emb)
        loss = criterion(logits, yb)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += float(loss.item()) * emb.size(0)
        pred = logits.argmax(1)
        all_preds.append(pred.detach().cpu().numpy())
        all_targs.append(yb.detach().cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))

    scheduler.step()
    lr_now = float(optimizer.param_groups[0]["lr"])

    # Train metrics
    y_train_pred = np.concatenate(all_preds) if all_preds else np.array([], dtype=int)
    y_train_true = np.concatenate(all_targs) if all_targs else np.array([], dtype=int)

    train_loss = total_loss / max(1, len(train_loader_emb.dataset))
    train_acc = float((y_train_pred == y_train_true).mean())
    train_prf = prf_all_averages(y_train_true, y_train_pred, zero_division=0)

    # Val metrics
    val_metrics, _, _ = evaluate_comprehensive(val_loader_emb, desc="Val", warmup_batches=5)

    # Record history
    history["epoch"].append(int(epoch))
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_metrics["loss"]))
    history["train_acc"].append(float(train_acc))
    history["val_acc"].append(float(val_metrics["acc"]))
    
    history["train_f1_macro"].append(float(train_prf["macro"]["f1"]))
    history["train_f1_micro"].append(float(train_prf["micro"]["f1"]))
    history["train_f1_weighted"].append(float(train_prf["weighted"]["f1"]))
    history["val_f1_macro"].append(float(val_metrics["prf"]["macro"]["f1"]))
    history["val_f1_micro"].append(float(val_metrics["prf"]["micro"]["f1"]))
    history["val_f1_weighted"].append(float(val_metrics["prf"]["weighted"]["f1"]))
    
    history["train_precision_macro"].append(float(train_prf["macro"]["precision"]))
    history["train_recall_macro"].append(float(train_prf["macro"]["recall"]))
    history["val_precision_macro"].append(float(val_metrics["prf"]["macro"]["precision"]))
    history["val_recall_macro"].append(float(val_metrics["prf"]["macro"]["recall"]))
    
    history["lr"].append(float(lr_now))

    print(
        f"Epoch {epoch:03d} | lr={lr_now:.3e} | "
        f"train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} | "
        f"train_acc={train_acc:.4f} val_acc={val_metrics['acc']:.4f} | "
        f"train_f1={train_prf['macro']['f1']:.4f} val_f1={val_metrics['prf']['macro']['f1']:.4f}"
    )

    val_f1 = val_metrics["prf"]["macro"]["f1"]
    if val_f1 > best_f1 + 1e-4:
        best_f1 = float(val_f1)
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= int(PATIENCE):
            print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {int(PATIENCE)} epochs).")
            break

save_json("history.json", history)


# =======================
# Load best model & save checkpoint
# =======================
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model with val macro-F1 = {best_f1:.4f}")
else:
    print("\nWarning: no improvement tracked; using last epoch weights.")

SAVE_PATH = os.path.join(NORMWEAR_DIR, "normwear_base_head_best.pt")
torch.save(
    {
        "model_state": model.state_dict(),
        "config": {
            "SAMPLING_RATE": SAMPLING_RATE,
            "BATCH": BATCH,
            "EPOCHS": EPOCHS,
            "LR_HEAD": LR_HEAD,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "n_classes": n_classes,
            "in_dim": in_dim,
            "PATCH_POOL": PATCH_POOL,
            "CHANNEL_POOL": CHANNEL_POOL,
            "DECAY_TARGET": DECAY_TARGET,
            "GAMMA": GAMMA,
            "param_summary": param_summary,
            "MODE": "BASE_ONLY",
        },
    },
    SAVE_PATH,
)
print("Saved best model to:", SAVE_PATH)


# =======================
# (ADDED) FULL PIPELINE INFERENCE BENCHMARK
# =======================
print("\n" + "=" * 60)
print("BENCHMARKING FULL PIPELINE INFERENCE (CWT + Backbone + Head)")
print("=" * 60)

@torch.no_grad()
def benchmark_inference_full_pipeline(
    model: nn.Module,
    example_batch: torch.Tensor,
    device: torch.device,
    n_warmup: int = 25,
    n_runs: int = 100,
):
    """Benchmark full pipeline: raw windows -> predictions"""
    model.eval()
    xb = example_batch.to(device)

    def _sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    # Warmup
    for _ in range(n_warmup):
        _ = model(xb)
    _sync()

    times_ms = []
    for _ in range(n_runs):
        _sync()
        t0 = time.perf_counter()
        _ = model(xb)
        _sync()
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

    times_ms = np.array(times_ms, dtype=float)
    median_ms = float(np.median(times_ms))
    mean_ms = float(times_ms.mean())
    p90_ms = float(np.percentile(times_ms, 90))
    p95_ms = float(np.percentile(times_ms, 95))

    per_window_ms = median_ms / max(1, int(xb.shape[0]))
    throughput = 1000.0 / per_window_ms if per_window_ms > 0 else 0.0

    return {
        "device": str(device),
        "batch_size": int(xb.shape[0]),
        "T": int(xb.shape[1]),
        "C": int(xb.shape[2]),
        "n_warmup": int(n_warmup),
        "n_runs": int(n_runs),
        "median_batch_ms": median_ms,
        "mean_batch_ms": mean_ms,
        "p90_batch_ms": p90_ms,
        "p95_batch_ms": p95_ms,
        "median_per_window_ms": float(per_window_ms),
        "throughput_windows_per_s": float(throughput),
    }


# Build full pipeline
full_pipeline = NormWearBaseFullPipeline(
    normwear_model=normwear,
    head=model,
    patch_pool=PATCH_POOL,
    channel_pool=CHANNEL_POOL,
).to(device)
full_pipeline.eval()

# Run benchmark
x_bench = x0[:min(BATCH, int(x0.shape[0]))].contiguous()
bench = benchmark_inference_full_pipeline(
    model=full_pipeline,
    example_batch=x_bench,
    device=device,
    n_warmup=25,
    n_runs=100,
)

bench_text = (
    "INFERENCE BENCHMARK (FULL PIPELINE: CWT + Backbone + Aggregation + Head)\n"
    f"Device: {bench['device']}\n"
    f"Input: B={bench['batch_size']} T={bench['T']} C={bench['C']}\n"
    f"Median batch: {bench['median_batch_ms']:.3f} ms | "
    f"Median/window: {bench['median_per_window_ms']:.3f} ms | "
    f"Throughput: {bench['throughput_windows_per_s']:.2f} windows/s\n"
    f"P90/P95 batch: {bench['p90_batch_ms']:.3f}/{bench['p95_batch_ms']:.3f} ms\n"
)
print(bench_text)
save_text("inference_benchmark_summary.txt", bench_text)
save_json("inference_benchmark_window.json", bench)


# =======================
# TRAINING CURVES
# =======================
print("\n" + "=" * 60)
print("GENERATING TRAINING CURVES")
print("=" * 60)

epochs_hist = history["epoch"]

# Loss curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_loss"], label="Train loss")
plt.plot(epochs_hist, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_loss_train_val.png", dpi=300)

# Accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_acc"], label="Train accuracy")
plt.plot(epochs_hist, history["val_acc"], label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_acc_train_val.png", dpi=300)

# F1 curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_f1_macro"], label="Train F1 (macro)")
plt.plot(epochs_hist, history["val_f1_macro"], label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Train vs Validation F1 (macro)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_f1_train_val.png", dpi=300)

# LR curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["lr"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule (ExponentialLR)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_lr.png", dpi=300)


# =======================
# WINDOW-LEVEL EVALUATION (UPDATED WITH FULL PIPELINE TIMING)
# =======================
print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION")
print("=" * 60)

val_metrics, y_val_true, y_val_pred = evaluate_comprehensive(val_loader_emb, desc="Val-final", warmup_batches=5)
test_metrics, y_test_true, y_test_pred = evaluate_comprehensive(test_loader_emb, desc="Test", warmup_batches=5)

# UPDATE: Replace head-only timing with full pipeline timing
val_metrics["timing"]["ms_per_sample"] = bench["median_per_window_ms"]
val_metrics["timing"]["ms_per_batch"] = bench["median_batch_ms"]
val_metrics["timing"]["note"] = "Full pipeline: CWT + Backbone + Aggregation + Head"

test_metrics["timing"]["ms_per_sample"] = bench["median_per_window_ms"]
test_metrics["timing"]["ms_per_batch"] = bench["median_batch_ms"]
test_metrics["timing"]["note"] = "Full pipeline: CWT + Backbone + Aggregation + Head"

save_json(
    "metrics_window.json",
    {"val": val_metrics, "test": test_metrics, "best_ckpt": str(SAVE_PATH)},
)

# Detailed summary
save_text(
    "metrics_summary_window.txt",
    "WINDOW-LEVEL METRICS (FULL PIPELINE: CWT + Backbone + Aggregation + Head)\n"
    f"VAL  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} ms/sample={val_metrics['timing']['ms_per_sample']:.4f}\n"
    f"VAL  macro    P/R/F1={val_metrics['prf']['macro']['precision']:.4f}/{val_metrics['prf']['macro']['recall']:.4f}/{val_metrics['prf']['macro']['f1']:.4f}\n"
    f"VAL  micro    P/R/F1={val_metrics['prf']['micro']['precision']:.4f}/{val_metrics['prf']['micro']['recall']:.4f}/{val_metrics['prf']['micro']['f1']:.4f}\n"
    f"VAL  weighted P/R/F1={val_metrics['prf']['weighted']['precision']:.4f}/{val_metrics['prf']['weighted']['recall']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} ms/sample={test_metrics['timing']['ms_per_sample']:.4f}\n"
    f"TEST macro    P/R/F1={test_metrics['prf']['macro']['precision']:.4f}/{test_metrics['prf']['macro']['recall']:.4f}/{test_metrics['prf']['macro']['f1']:.4f}\n"
    f"TEST micro    P/R/F1={test_metrics['prf']['micro']['precision']:.4f}/{test_metrics['prf']['micro']['recall']:.4f}/{test_metrics['prf']['micro']['f1']:.4f}\n"
    f"TEST weighted P/R/F1={test_metrics['prf']['weighted']['precision']:.4f}/{test_metrics['prf']['weighted']['recall']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
)

save_json(
    "val_test_support.json",
    {
        "val_support": np.bincount(y_val_true, minlength=int(n_classes)).tolist(),
        "test_support": np.bincount(y_test_true, minlength=int(n_classes)).tolist(),
    },
)

# Build class names
class_names = [idx_to_name.get(i, f"Class {i}") for i in range(n_classes)]
if NEG_ID_EXPECTED < n_classes:
    class_names[NEG_ID_EXPECTED] = NEG_NAME
save_json("class_names.json", {"class_names": class_names})

# Confusion matrices - WINDOW LEVEL (Blues)
_ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, normalized 0-1)",
    class_names=class_names,
    out_png="cm_val_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, normalized 0-1)",
    class_names=class_names,
    out_png="cm_test_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, counts)",
    class_names=class_names,
    out_png="cm_val_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, counts)",
    class_names=class_names,
    out_png="cm_test_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Classification reports
labels = np.arange(n_classes)
val_report = classification_report(y_val_true, y_val_pred, labels=labels, target_names=class_names, digits=3, zero_division=0)
test_report = classification_report(y_test_true, y_test_pred, labels=labels, target_names=class_names, digits=3, zero_division=0)
save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)


# =======================
# CLIP-LEVEL EVALUATION
# =======================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION")
print("=" * 60)

def early_stop_classwise(seq, k_map=None, default_k=3):
    """Early-stop aggregation with class-wise k values"""
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        v = int(v)
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = int(k_map.get(v, default_k))
        if run_len >= k_c:
            return v
    return None

test_sw_ds = dm.test_dataset
base_ds = getattr(test_sw_ds, "base", test_sw_ds)
n_clips = len(base_ds)

if hasattr(test_sw_ds, "_index"):
    clip_ids_for_windows = [pair[0] for pair in test_sw_ds._index]
else:
    clip_ids_for_windows = list(range(len(test_sw_ds)))

clip_metrics = None
clip_targets = None
clip_preds = None

if len(clip_ids_for_windows) != len(y_test_true):
    save_text(
        "clip_eval_note.txt",
        f"Skipping clip-level: clip_ids_for_windows={len(clip_ids_for_windows)} != n_windows={len(y_test_true)}"
    )
else:
    clip_to_preds = defaultdict(list)
    clip_to_targs = defaultdict(list)
    for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
        clip_to_preds[int(cid)].append(int(p))
        clip_to_targs[int(cid)].append(int(t))

    # Get clip targets
    clip_targets_list = []
    for cid in range(int(n_clips)):
        if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
            clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
            clip_targets_list.append(int(base_ds.label_map[clean]))
        else:
            tgts = clip_to_targs.get(cid, [])
            clip_targets_list.append(int(max(set(tgts), key=tgts.count)) if tgts else 0)
    clip_targets = np.array(clip_targets_list, dtype=int)

    # Get clip predictions with early-stop
    k_map = CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else {}
    clip_preds_list = []
    missed_early = 0
    for cid in range(int(n_clips)):
        seq = clip_to_preds.get(cid, [])
        chosen = early_stop_classwise(seq, k_map=k_map, default_k=int(CLIP_DEFAULT_K))
        if chosen is None:
            missed_early += 1
            chosen = int(max(set(seq), key=seq.count) if seq else clip_targets[cid])
        clip_preds_list.append(int(chosen))
    clip_preds = np.array(clip_preds_list, dtype=int)

    # Compute comprehensive clip metrics
    clip_acc = float((clip_preds == clip_targets).mean()) if clip_targets.size else float("nan")
    clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0) if clip_targets.size else None
    clip_per_cls = per_class_f1(clip_targets, clip_preds, n_classes_local=int(n_classes), zero_division=0) if clip_targets.size else None

    # Timing for aggregation (minimal)
    t0 = time.perf_counter()
    _ = clip_preds
    t1 = time.perf_counter()
    clip_agg_ms_total = float(1000.0 * (t1 - t0))
    clip_agg_ms_per_clip = float(clip_agg_ms_total / max(1, int(n_clips)))

    clip_metrics = {
        "acc": float(clip_acc),
        "prf": clip_prf,
        "per_class": clip_per_cls,
        "timing": {
            "window_forward_ms_per_sample": float(test_metrics["timing"]["ms_per_sample"]),
            "aggregation_ms_total": float(clip_agg_ms_total),
            "aggregation_ms_per_clip": float(clip_agg_ms_per_clip),
        },
        "early_stop": {
            "default_k": int(CLIP_DEFAULT_K),
            "k_map_size": int(len(k_map)),
            "missed_early": int(missed_early),
            "n_clips": int(n_clips),
        },
    }
    save_json("metrics_clip.json", clip_metrics)

    # Detailed summary
    save_text(
        "metrics_summary_clip.txt",
        "CLIP-LEVEL METRICS\n"
        f"TEST acc={clip_acc:.4f}\n"
        f"TEST macro    P/R/F1={clip_prf['macro']['precision']:.4f}/{clip_prf['macro']['recall']:.4f}/{clip_prf['macro']['f1']:.4f}\n"
        f"TEST micro    P/R/F1={clip_prf['micro']['precision']:.4f}/{clip_prf['micro']['recall']:.4f}/{clip_prf['micro']['f1']:.4f}\n"
        f"TEST weighted P/R/F1={clip_prf['weighted']['precision']:.4f}/{clip_prf['weighted']['recall']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
        f"Forward ms/sample (from window eval)={test_metrics['timing']['ms_per_sample']:.4f}\n"
        f"Aggregation ms/clip={clip_agg_ms_per_clip:.4f}\n"
        f"Early-stop default_k={int(CLIP_DEFAULT_K)} k_map_size={len(k_map)} majority_fallback={missed_early}/{n_clips}\n"
    )

    # Confusion matrices - CLIP LEVEL (Greens)
    _ = plot_conf_mat_norm01(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, normalized 0-1)",
        class_names=class_names,
        out_png="cm_test_clip_norm01.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
        dpi=200,
    )

    _ = plot_conf_mat_counts(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, counts)",
        class_names=class_names,
        out_png="cm_test_clip_counts.png",
        cmap="Greens",
        annot=True,
        dpi=200,
    )

    # Classification report
    clip_report = classification_report(clip_targets, clip_preds, labels=labels, target_names=class_names, digits=3, zero_division=0)
    save_text("report_test_clip.txt", clip_report)


# =======================
# FINAL SUMMARY
# =======================
final_summary = (
    "FINAL SUMMARY (BASE ONLY - NO LORA)\n"
    f"Head params: {n_total:,} (trainable), Backbone params: {n_backbone_total:,} (frozen)\n"
    f"Best checkpoint: {SAVE_PATH}\n"
    f"VAL window:  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={val_metrics['prf']['macro']['f1']:.4f}/"
    f"{val_metrics['prf']['micro']['f1']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={test_metrics['prf']['macro']['f1']:.4f}/"
    f"{test_metrics['prf']['micro']['f1']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
    + (
        f"TEST clip:   acc={clip_metrics['acc']:.4f} "
        f"F1(macro/micro/weighted)={clip_metrics['prf']['macro']['f1']:.4f}/"
        f"{clip_metrics['prf']['micro']['f1']:.4f}/{clip_metrics['prf']['weighted']['f1']:.4f}\n"
        if clip_metrics is not None and clip_metrics.get("prf") is not None else
        "TEST clip:   SKIPPED\n"
    ) +
    f"Full pipeline timing: {bench['median_per_window_ms']:.4f} ms/window | {bench['throughput_windows_per_s']:.2f} windows/s\n"
    f"Artifacts:   {NORMWEAR_DIR}\n"
)
print(final_summary)
save_text("final_summary.txt", final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", NORMWEAR_DIR)

In [ ]:
# ==== LoRA Fine-tuning NormWear + ResidualMLP on RAW windows ====
# ---- NormWear LORA run dir ----
RUN_TAG = time.strftime("%Y%m%d_%H%M%S")
NORMWEAR_DIR = os.path.join(OUTPUTS, "normwear_lora", f"run_{RUN_TAG}")
os.makedirs(NORMWEAR_DIR, exist_ok=True)
print("Output directory:", OUTPUTS)
print("NormWear LORA run directory:", NORMWEAR_DIR)

def save_text(filename: str, text: str) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    with open(path, "w", encoding='utf-8') as f:
        f.write(text)
    return path

def save_json(filename: str, obj) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    with open(path, "w", encoding='utf-8') as f:
        json.dump(obj, f, indent=2)
    return path

def save_fig(filename: str, dpi: int = 200) -> str:
    path = os.path.join(NORMWEAR_DIR, filename)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path


# =======================
# LoRA implementation
# =======================
class LoRALayer(nn.Module):
    """Low-Rank Adaptation layer wrapping nn.Linear."""
    def __init__(self, original_layer: nn.Linear, r: int = 8, alpha: float = 16, dropout: float = 0.05):
        super().__init__()
        self.original_layer = original_layer
        self.r = r
        self.scaling = alpha / r

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        # Freeze original weights
        for param in self.original_layer.parameters():
            param.requires_grad = False

        # LoRA matrices
        self.lora_A = nn.Parameter(torch.zeros(r, in_features))
        self.lora_B = nn.Parameter(torch.zeros(out_features, r))

        # Init: A with Kaiming, B with zeros
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        original_output = self.original_layer(x)
        lora_output = self.dropout(x) @ self.lora_A.T @ self.lora_B.T * self.scaling
        return original_output + lora_output


def apply_lora_to_linear_layers(
    module: nn.Module,
    target_names: List[str],
    r: int = 8,
    alpha: float = 16,
    dropout: float = 0.05,
    prefix: str = ""
) -> List[str]:
    modified = []
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, nn.Linear):
            if any(t.lower() in name.lower() for t in target_names):
                lora_layer = LoRALayer(child, r=r, alpha=alpha, dropout=dropout)
                setattr(module, name, lora_layer)
                modified.append(full_name)
        else:
            modified.extend(apply_lora_to_linear_layers(child, target_names, r, alpha, dropout, full_name))
    return modified


def apply_lora_to_encoder_blocks(
    backbone: nn.Module,
    target_names: List[str],
    last_k: int = 4,
    r: int = 8,
    alpha: float = 16,
    dropout: float = 0.05,
    include_cls_fusion: bool = True,
) -> List[str]:
    modified = []
    if not hasattr(backbone, "encoder_blocks"):
        print("backbone has no encoder_blocks; no LoRA applied.")
        return modified

    blocks = backbone.encoder_blocks
    depth = len(blocks)
    if last_k is None or last_k <= 0 or last_k > depth:
        block_indices = range(depth)
    else:
        block_indices = range(depth - last_k, depth)

    for i in block_indices:
        encoder_layer = blocks[i]
        if hasattr(encoder_layer, "variate_encoder"):
            modified.extend(
                apply_lora_to_linear_layers(
                    encoder_layer.variate_encoder,
                    target_names=target_names,
                    r=r, alpha=alpha, dropout=dropout,
                    prefix=f"encoder_blocks.{i}.variate_encoder"
                )
            )
        if include_cls_fusion and hasattr(encoder_layer, "cls_fusion") and encoder_layer.cls_fusion is not None:
            modified.extend(
                apply_lora_to_linear_layers(
                    encoder_layer.cls_fusion,
                    target_names=target_names,
                    r=r, alpha=alpha, dropout=dropout,
                    prefix=f"encoder_blocks.{i}.cls_fusion"
                )
            )
    return modified


def get_lora_params(model: nn.Module) -> List[nn.Parameter]:
    params = []
    for m in model.modules():
        if isinstance(m, LoRALayer):
            params.extend([m.lora_A, m.lora_B])
    return params


def save_lora_weights(model: nn.Module, path: str):
    state = {}
    for name, m in model.named_modules():
        if isinstance(m, LoRALayer):
            state[f"{name}.lora_A"] = m.lora_A.data.cpu()
            state[f"{name}.lora_B"] = m.lora_B.data.cpu()
    torch.save(state, path)
    print("Saved LoRA weights to:", path)


def load_lora_weights(model: nn.Module, path: str):
    state = torch.load(path, map_location="cpu")
    for name, m in model.named_modules():
        if isinstance(m, LoRALayer):
            if f"{name}.lora_A" in state:
                m.lora_A.data = state[f"{name}.lora_A"].to(m.lora_A.device)
            if f"{name}.lora_B" in state:
                m.lora_B.data = state[f"{name}.lora_B"].to(m.lora_B.device)
    print("Loaded LoRA weights from:", path)


# =======================
# Config
# =======================
SEED = 42
SAMPLING_RATE = 100.0
NORMWEAR_ENCODER_CKPT = "NormWear/normwear_last_checkpoint-15470-correct.pth"

BATCH = 16
EPOCHS = 100
PATIENCE = 5

LORA_R = 8
LORA_ALPHA = 32
LORA_DROPOUT = 0.1
LORA_TARGET_MODULES = ["qkv", "proj"]
LORA_LAST_K = 2
LORA_INCLUDE_CLS_FUSION = True

PATCH_POOL = "mean+max"
CHANNEL_POOL = "max"

LR_LORA = 1e-3
LR_HEAD = 1e-3
WEIGHT_DECAY = 1e-2

# Clip-level aggregation config
CLIP_K_MAP = {}
CLIP_DEFAULT_K = 3

idx_to_name = {
    0:  "double_clench",
    1:  "double_pinch",
    2:  "pinch_down",
    3:  "pinch_up",
    4:  "slide",
}
NEG_NAME = "Negative"
NEG_ID_EXPECTED = 5

run_config = {
    "SEED": SEED,
    "SAMPLING_RATE": SAMPLING_RATE,
    "NORMWEAR_ENCODER_CKPT": NORMWEAR_ENCODER_CKPT,
    "BATCH": BATCH,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "LORA": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "last_k": LORA_LAST_K,
        "include_cls_fusion": LORA_INCLUDE_CLS_FUSION,
    },
    "POOLING": {"PATCH_POOL": PATCH_POOL, "CHANNEL_POOL": CHANNEL_POOL},
    "OPT": {"LR_LORA": LR_LORA, "LR_HEAD": LR_HEAD, "WEIGHT_DECAY": WEIGHT_DECAY},
    "CLIP": {"DEFAULT_K": CLIP_DEFAULT_K, "K_MAP": CLIP_K_MAP},
}
save_json("config.json", run_config)


# =======================
# Reset + seeds
# =======================
for name in ["model", "optimizer", "scheduler", "history", "best_state"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# =======================
# Dataloaders from dm
# =======================
train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

x0, _, y0 = next(iter(train_loader))
T, C = x0.shape[1], x0.shape[2]

train_labels = torch.tensor(dm.train_dataset.get_label_list(), dtype=torch.long)
n_classes = int(train_labels.max().item()) + 1

print(f"Window shape: [T={T}, C={C}], n_classes={n_classes}")

uniq = torch.unique(train_labels).cpu().tolist()
print(f"Train labels range: [{int(train_labels.min())}, {int(train_labels.max())}], unique={len(uniq)}")
assert train_labels.min().item() >= 0, "Found negative label id; labels must be >= 0 for CrossEntropyLoss."
assert max(uniq) == n_classes - 1, "Non-contiguous labels detected; expected 0..n_classes-1."


# =======================
# Build NormWear + load checkpoint
# =======================
normwear = NormWearModel(weight_path="", optimized_cwt=True).to(device)

ckpt = torch.load(NORMWEAR_ENCODER_CKPT, map_location="cpu", weights_only=False)
if isinstance(ckpt, dict) and "model" in ckpt:
    state_dict = ckpt["model"]
elif isinstance(ckpt, dict) and "state_dict" in ckpt:
    state_dict = ckpt["state_dict"]
else:
    state_dict = ckpt

missing, unexpected = normwear.backbone.load_state_dict(state_dict, strict=False)
print("Loaded NormWear pretrained backbone.")
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))


# =======================
# Embedding aggregation
# =======================
def aggregate_normwear_embeddings(
    out: torch.Tensor,
    patch_pool: str = "mean+max",
    channel_pool: str = "mean+max",
) -> torch.Tensor:
    B, Cc, P, D = out.shape

    if patch_pool == "mean":
        emb = out.mean(dim=2)
    elif patch_pool == "max":
        emb, _ = out.max(dim=2)
    elif patch_pool == "mean+max":
        mean_p = out.mean(dim=2)
        max_p, _ = out.max(dim=2)
        emb = torch.cat([mean_p, max_p], dim=-1)
    elif patch_pool == "cls":
        emb = out[:, :, 0, :]
    elif patch_pool == "softmax-norm":
        weights = out.norm(dim=-1)
        weights = torch.softmax(weights, dim=-1)[..., None]
        emb = (out * weights).sum(dim=2)
    else:
        raise ValueError(f"Unknown patch_pool: {patch_pool}")

    if channel_pool == "mean":
        emb = emb.mean(dim=1)
    elif channel_pool == "max":
        emb, _ = emb.max(dim=1)
    elif channel_pool == "mean+max":
        mean_c = emb.mean(dim=1)
        max_c, _ = emb.max(dim=1)
        emb = torch.cat([mean_c, max_c], dim=-1)
    elif channel_pool == "concat":
        emb = emb.reshape(B, -1)
    elif channel_pool == "softmax-norm":
        weights = emb.norm(dim=-1)
        weights = torch.softmax(weights, dim=1)[..., None]
        emb = (emb * weights).sum(dim=1)
    else:
        raise ValueError(f"Unknown channel_pool: {channel_pool}")

    return emb


# Infer embedding dimension after pooling
normwear.eval()
with torch.no_grad():
    x0_nw = x0.permute(0, 2, 1).contiguous().to(device)
    spec0 = normwear.calc_cwt(x0_nw, device=device).float()
    fn = normwear.backbone.get_signal_embedding
    if hasattr(fn, "__wrapped__"):
        out0 = fn.__wrapped__(normwear.backbone, spec0, hidden_out=False, device=device)
    else:
        out0 = fn(spec0, hidden_out=False, device=device)
    emb0 = aggregate_normwear_embeddings(out0, PATCH_POOL, CHANNEL_POOL)

in_dim = emb0.shape[1]
print("Embedding dim from NormWear (after pooling):", in_dim)


# =======================
# Residual MLP head
# =======================
class ResidualMLPClassifier(nn.Module):
    def __init__(self, in_dim: int, n_classes: int, hidden_dim: int = 128, dropout: float = 0.5):
        super().__init__()
        self.head = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(x)


# =======================
# NormWear with head + LoRA
# =======================
class NormWearWithHeadLoRA(nn.Module):
    def __init__(
        self,
        normwear_model: NormWearModel,
        head: nn.Module,
        lora_r=8,
        lora_alpha=16,
        lora_dropout=0.05,
        lora_target_modules=None,
        lora_last_k=4,
        lora_include_cls_fusion=True,
    ):
        super().__init__()
        self.normwear = normwear_model
        self.head = head

        for p in self.normwear.backbone.parameters():
            p.requires_grad = False

        target_modules = lora_target_modules or ["qkv", "proj"]
        modified = apply_lora_to_encoder_blocks(
            self.normwear.backbone,
            target_names=target_modules,
            last_k=lora_last_k,
            r=lora_r,
            alpha=lora_alpha,
            dropout=lora_dropout,
            include_cls_fusion=lora_include_cls_fusion,
        )
        print(f"\nApplied LoRA (r={lora_r}, alpha={lora_alpha}) to {len(modified)} linear layers:")
        for name in modified:
            print("  -", name)

        self.to(next(self.normwear.parameters()).device)

        for p in self.head.parameters():
            p.requires_grad = True

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x_nw = x.permute(0, 2, 1).contiguous()  # [B, C, T]
        spec = self.normwear.calc_cwt(x_nw, device=x.device).float()

        fn = self.normwear.backbone.get_signal_embedding
        if hasattr(fn, "__wrapped__"):
            out = fn.__wrapped__(self.normwear.backbone, spec, hidden_out=False, device=x.device)
        else:
            out = fn(spec, hidden_out=False, device=x.device)

        emb = aggregate_normwear_embeddings(out, PATCH_POOL, CHANNEL_POOL)
        return self.head(emb)


# =======================
# Build model
# =======================
head = ResidualMLPClassifier(in_dim=in_dim, n_classes=n_classes).to(device)

model = NormWearWithHeadLoRA(
    normwear_model=normwear,
    head=head,
    lora_r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    lora_target_modules=LORA_TARGET_MODULES,
    lora_last_k=LORA_LAST_K,
    lora_include_cls_fusion=LORA_INCLUDE_CLS_FUSION,
).to(device)


# =======================
# Inference benchmark helpers
# =======================
@torch.no_grad()
def benchmark_inference_window(
    model: nn.Module,
    example_batch: torch.Tensor,
    device: torch.device,
    n_warmup: int = 25,
    n_runs: int = 100,
):
    """Benchmark full pipeline: raw windows -> predictions"""
    model.eval()
    xb = example_batch.to(device)

    def _sync():
        if device.type == "cuda":
            torch.cuda.synchronize()

    # Warmup
    for _ in range(n_warmup):
        _ = model(xb)
    _sync()

    times_ms = []
    for _ in range(n_runs):
        _sync()
        t0 = time.perf_counter()
        _ = model(xb)
        _sync()
        t1 = time.perf_counter()
        times_ms.append((t1 - t0) * 1000.0)

    times_ms = np.array(times_ms, dtype=float)
    median_ms = float(np.median(times_ms))
    mean_ms = float(times_ms.mean())
    p90_ms = float(np.percentile(times_ms, 90))
    p95_ms = float(np.percentile(times_ms, 95))

    per_window_ms = median_ms / max(1, int(xb.shape[0]))
    throughput = 1000.0 / per_window_ms if per_window_ms > 0 else 0.0

    return {
        "device": str(device),
        "batch_size": int(xb.shape[0]),
        "T": int(xb.shape[1]),
        "C": int(xb.shape[2]),
        "n_warmup": int(n_warmup),
        "n_runs": int(n_runs),
        "median_batch_ms": median_ms,
        "mean_batch_ms": mean_ms,
        "p90_batch_ms": p90_ms,
        "p95_batch_ms": p95_ms,
        "median_per_window_ms": float(per_window_ms),
        "throughput_windows_per_s": float(throughput),
    }


# =======================
# Parameter stats
# =======================
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_backbone_total = sum(p.numel() for p in model.normwear.backbone.parameters())
n_backbone_train = sum(p.numel() for p in model.normwear.backbone.parameters() if p.requires_grad)
n_head_train = sum(p.numel() for p in model.head.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in get_lora_params(model))

param_text = (
    "\n" + "=" * 60 + "\n"
    "PARAMETER STATISTICS\n"
    + "=" * 60 + "\n"
    f"Total params:              {n_total:,}\n"
    f"Trainable params:          {n_trainable:,}\n"
    f"  - LoRA params:           {n_lora:,}\n"
    f"  - Head params:           {n_head_train:,}\n"
    f"Backbone total:            {n_backbone_total:,}\n"
    f"Backbone trainable:        {n_backbone_train:,}\n"
    f"Trainable %:               {100*n_trainable/n_total:.4f}%\n"
    + "=" * 60 + "\n"
)
print(param_text)
save_text("parameter_stats.txt", param_text)

param_summary = {
    "model_total_params": int(n_total),
    "model_trainable_params": int(n_trainable),
    "backbone_total_params": int(n_backbone_total),
    "backbone_trainable_params": int(n_backbone_train),
    "head_trainable_params": int(n_head_train),
    "lora_params": int(n_lora),
    "trainable_percent": float(100.0 * n_trainable / max(1, n_total)),
}
save_json("parameter_summary.json", param_summary)


# =======================
# Class weights & sampler
# =======================
all_train_labels = torch.tensor(dm.train_dataset.get_label_list(), dtype=torch.long)
class_counts = torch.bincount(all_train_labels, minlength=n_classes).float()
assert class_counts.numel() == n_classes

class_weights = 1.0 / (class_counts + 1e-8)
class_weights = class_weights * (n_classes / class_weights.sum())
assert class_weights.numel() == n_classes

save_json("train_class_counts.json", {"counts": class_counts.tolist()})
save_json("train_class_weights.json", {"weights": class_weights.tolist()})

sample_weights = class_weights[all_train_labels]
sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True,
)

train_loader = DataLoader(
    dm.train_dataset,
    batch_size=BATCH,
    sampler=sampler,
    num_workers=0,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=0.05,
)


# =======================
# Optimizer + scheduler (exp decay per group)
# =======================
lora_params = get_lora_params(model)
head_params = list(model.head.parameters())

optimizer = torch.optim.AdamW(
    [{"params": lora_params, "lr": LR_LORA},
     {"params": head_params, "lr": LR_HEAD}],
    weight_decay=WEIGHT_DECAY
)

DECAY_TARGET_LORA = 0.01
DECAY_TARGET_HEAD = 0.01

GAMMA_LORA = float(math.exp(math.log(DECAY_TARGET_LORA) / max(1, int(EPOCHS))))
GAMMA_HEAD = float(math.exp(math.log(DECAY_TARGET_HEAD) / max(1, int(EPOCHS))))

def lr_lambda_lora(epoch: int) -> float:
    return GAMMA_LORA ** epoch

def lr_lambda_head(epoch: int) -> float:
    return GAMMA_HEAD ** epoch

scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=[lr_lambda_lora, lr_lambda_head]
)

run_config["OPT"].update({
    "DECAY_TARGET_LORA": DECAY_TARGET_LORA,
    "DECAY_TARGET_HEAD": DECAY_TARGET_HEAD,
    "GAMMA_LORA": GAMMA_LORA,
    "GAMMA_HEAD": GAMMA_HEAD,
})
save_json("config.json", run_config)


# =======================
# Metrics helpers
# =======================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out


# =======================
# Evaluation helper
# =======================
@torch.no_grad()
def evaluate(loader, desc="Eval"):
    model.eval()
    total_loss = 0.0
    all_y, all_p = [], []

    for xb, _, yb in tqdm(loader, desc=desc, leave=False):
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = criterion(logits, yb)
        total_loss += loss.item() * xb.size(0)

        preds = logits.argmax(dim=1).cpu()
        all_y.append(yb.cpu())
        all_p.append(preds)

    y_true = torch.cat(all_y).numpy()
    y_pred = torch.cat(all_p).numpy()

    loss_avg = total_loss / len(loader.dataset)
    acc = float((y_true == y_pred).mean())
    prf = prf_all_averages(y_true, y_pred, zero_division=0)

    return {
        "loss": float(loss_avg),
        "acc": acc,
        "prf": prf,
        "y_true": y_true,
        "y_pred": y_pred,
    }


# =======================
# Training loop
# =======================
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "lr_lora": [],
    "lr_head": [],

    "train_p_macro": [], "train_r_macro": [], "train_f1_macro": [],
    "train_p_micro": [], "train_r_micro": [], "train_f1_micro": [],
    "train_p_weighted": [], "train_r_weighted": [], "train_f1_weighted": [],

    "val_p_macro": [], "val_r_macro": [], "val_f1_macro": [],
    "val_p_micro": [], "val_r_micro": [], "val_f1_micro": [],
    "val_p_weighted": [], "val_r_weighted": [], "val_f1_weighted": [],
}

best_state = None
best_f1 = -1.0
no_improve = 0
printed_grad_info = False

print("\n" + "=" * 60)
print("STARTING TRAINING (LoRA + HEAD)")
print("=" * 60 + "\n")

for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    all_y_train, all_p_train = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False)
    for xb, _, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        if not printed_grad_info:
            lora_grad_norm = sum(
                p.grad.norm().item() for p in get_lora_params(model) if p.grad is not None
            )
            head_grad_norm = sum(
                p.grad.norm().item() for p in model.head.parameters() if p.grad is not None
            )
            grad_text = (
                f"[GRAD CHECK] LoRA grad norm: {lora_grad_norm:.6f}\n"
                f"[GRAD CHECK] Head grad norm: {head_grad_norm:.6f}\n"
            )
            print(grad_text)
            save_text("grad_check.txt", grad_text)
            printed_grad_info = True

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += loss.item() * xb.size(0)
        preds = logits.argmax(dim=1).detach().cpu()
        all_p_train.append(preds)
        all_y_train.append(yb.detach().cpu())
        pbar.set_postfix(loss=float(loss.item()))

    train_loss = running_loss / len(train_loader.dataset)
    y_train_true = torch.cat(all_y_train).numpy()
    y_train_pred = torch.cat(all_p_train).numpy()
    train_acc = float((y_train_true == y_train_pred).mean())
    train_prf = prf_all_averages(y_train_true, y_train_pred, zero_division=0)

    val_out = evaluate(val_loader, desc="Val")
    val_loss = val_out["loss"]
    val_acc = val_out["acc"]
    val_prf = val_out["prf"]
    y_val_true, y_val_pred = val_out["y_true"], val_out["y_pred"]

    scheduler.step()

    lr_lora = float(optimizer.param_groups[0]["lr"])
    lr_head = float(optimizer.param_groups[1]["lr"])

    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)
    history["lr_lora"].append(lr_lora)
    history["lr_head"].append(lr_head)

    history["train_p_macro"].append(train_prf["macro"]["precision"])
    history["train_r_macro"].append(train_prf["macro"]["recall"])
    history["train_f1_macro"].append(train_prf["macro"]["f1"])
    history["train_p_micro"].append(train_prf["micro"]["precision"])
    history["train_r_micro"].append(train_prf["micro"]["recall"])
    history["train_f1_micro"].append(train_prf["micro"]["f1"])
    history["train_p_weighted"].append(train_prf["weighted"]["precision"])
    history["train_r_weighted"].append(train_prf["weighted"]["recall"])
    history["train_f1_weighted"].append(train_prf["weighted"]["f1"])

    history["val_p_macro"].append(val_prf["macro"]["precision"])
    history["val_r_macro"].append(val_prf["macro"]["recall"])
    history["val_f1_macro"].append(val_prf["macro"]["f1"])
    history["val_p_micro"].append(val_prf["micro"]["precision"])
    history["val_r_micro"].append(val_prf["micro"]["recall"])
    history["val_f1_micro"].append(val_prf["micro"]["f1"])
    history["val_p_weighted"].append(val_prf["weighted"]["precision"])
    history["val_r_weighted"].append(val_prf["weighted"]["recall"])
    history["val_f1_weighted"].append(val_prf["weighted"]["f1"])

    print(
        f"Epoch {epoch:03d} | lr_lora={lr_lora:.2e} lr_head={lr_head:.2e} | "
        f"train_loss={train_loss:.4f} val_loss={val_loss:.4f} | "
        f"train_acc={train_acc:.4f} val_acc={val_acc:.4f} | "
        f"val_F1(macro/micro/weighted)="
        f"{val_prf['macro']['f1']:.4f}/{val_prf['micro']['f1']:.4f}/{val_prf['weighted']['f1']:.4f}"
    )

    val_f1_macro = float(val_prf["macro"]["f1"])
    if val_f1_macro > best_f1 + 1e-4:
        best_f1 = val_f1_macro
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f"Early stopping at epoch {epoch} (no macro-F1 improvement for {PATIENCE} epochs).")
            break

save_json("history.json", history)


# =======================
# Load best model & BENCHMARK INFERENCE
# =======================
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nLoaded best model with val macro-F1 = {best_f1:.4f}")
else:
    print("\nWarning: no improvement tracked; using last epoch weights.")

print("\n" + "=" * 60)
print("BENCHMARKING INFERENCE SPEED (FULL PIPELINE)")
print("=" * 60)

x_bench = x0[:min(BATCH, int(x0.shape[0]))].contiguous()
bench = benchmark_inference_window(
    model=model,
    example_batch=x_bench,
    device=device,
    n_warmup=25,
    n_runs=100,
)

bench_text = (
    "INFERENCE BENCHMARK (FULL PIPELINE: CWT + Backbone + Aggregation + Head)\n"
    f"Device: {bench['device']}\n"
    f"Input: B={bench['batch_size']} T={bench['T']} C={bench['C']}\n"
    f"Median batch: {bench['median_batch_ms']:.3f} ms | "
    f"Median/window: {bench['median_per_window_ms']:.3f} ms | "
    f"Throughput: {bench['throughput_windows_per_s']:.2f} windows/s\n"
    f"P90/P95 batch: {bench['p90_batch_ms']:.3f}/{bench['p95_batch_ms']:.3f} ms\n"
)
print(bench_text)
save_text("inference_benchmark_summary.txt", bench_text)
save_json("inference_benchmark_window.json", bench)

# Update config
run_config["INFERENCE_BENCHMARK_WINDOW"] = bench
save_json("config.json", run_config)


# =======================
# Save checkpoints
# =======================
SAVE_PATH = os.path.join(NORMWEAR_DIR, "normwear_lora_finetune_best.pt")
SAVE_PATH_LORA_ONLY = os.path.join(NORMWEAR_DIR, "normwear_lora_weights_only.pt")

torch.save(
    {
        "model_state": model.state_dict(),
        "normwear_state": model.normwear.state_dict(),
        "head_state": model.head.state_dict(),
        "lora_config": {
            "r": LORA_R,
            "alpha": LORA_ALPHA,
            "dropout": LORA_DROPOUT,
            "target_modules": LORA_TARGET_MODULES,
            "last_k": LORA_LAST_K,
            "include_cls_fusion": LORA_INCLUDE_CLS_FUSION,
        },
        "config": {
            "SAMPLING_RATE": SAMPLING_RATE,
            "BATCH": BATCH,
            "EPOCHS": EPOCHS,
            "LR_LORA": LR_LORA,
            "LR_HEAD": LR_HEAD,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "n_classes": n_classes,
            "in_dim": in_dim,
            "PATCH_POOL": PATCH_POOL,
            "CHANNEL_POOL": CHANNEL_POOL,
            "DECAY_TARGET_LORA": DECAY_TARGET_LORA,
            "DECAY_TARGET_HEAD": DECAY_TARGET_HEAD,
            "GAMMA_LORA": GAMMA_LORA,
            "GAMMA_HEAD": GAMMA_HEAD,
            "param_summary": param_summary,
            "inference_benchmark_window": bench,
        },
    },
    SAVE_PATH,
)
print("Saved best model to:", SAVE_PATH)

save_lora_weights(model, SAVE_PATH_LORA_ONLY)


# =======================
# Window-level evaluation (VAL + TEST) (UPDATED WITH TIMING)
# =======================
print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION")
print("=" * 60)

val_out = evaluate(val_loader, desc="Val-final")
test_out = evaluate(test_loader, desc="Test")

val_loss, val_acc, val_prf = val_out["loss"], val_out["acc"], val_out["prf"]
test_loss, test_acc, test_prf = test_out["loss"], test_out["acc"], test_out["prf"]
y_val_true, y_val_pred = val_out["y_true"], val_out["y_pred"]
y_test_true, y_test_pred = test_out["y_true"], test_out["y_pred"]

# UPDATED: Add timing from benchmark
save_json("metrics_window.json", {
    "val":  {
        "loss": val_loss,  
        "acc": val_acc,  
        "prf": val_prf,
        "timing": {
            "ms_per_sample": bench["median_per_window_ms"],
            "ms_per_batch": bench["median_batch_ms"],
            "note": "Full pipeline: CWT + Backbone + Aggregation + Head"
        }
    },
    "test": {
        "loss": test_loss, 
        "acc": test_acc, 
        "prf": test_prf,
        "timing": {
            "ms_per_sample": bench["median_per_window_ms"],
            "ms_per_batch": bench["median_batch_ms"],
            "note": "Full pipeline: CWT + Backbone + Aggregation + Head"
        }
    },
})

metrics_summary = (
    f"WINDOW-LEVEL VAL   loss={val_loss:.4f}, acc={val_acc:.4f}, "
    f"F1(macro/micro/weighted)={val_prf['macro']['f1']:.4f}/{val_prf['micro']['f1']:.4f}/{val_prf['weighted']['f1']:.4f}\n"
    f"WINDOW-LEVEL TEST  loss={test_loss:.4f}, acc={test_acc:.4f}, "
    f"F1(macro/micro/weighted)={test_prf['macro']['f1']:.4f}/{test_prf['micro']['f1']:.4f}/{test_prf['weighted']['f1']:.4f}\n"
)
print(metrics_summary)
save_text("metrics_summary_window.txt", metrics_summary)


# =======================
# Training curves
# =======================
print("\n" + "=" * 60)
print("GENERATING TRAINING CURVES")
print("=" * 60)

epochs_hist = history["epoch"]

plt.figure(figsize=(18, 4))

plt.subplot(1, 4, 1)
plt.plot(epochs_hist, history["train_loss"], label="Train loss")
plt.plot(epochs_hist, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Val Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 4, 2)
plt.plot(epochs_hist, history["train_acc"], label="Train acc")
plt.plot(epochs_hist, history["val_acc"], label="Val acc")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Val Accuracy")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 4, 3)
plt.plot(epochs_hist, history["train_f1_macro"], label="Train F1 (macro)")
plt.plot(epochs_hist, history["val_f1_macro"], label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1 (macro)")
plt.title("Train vs Val Macro-F1")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 4, 4)
plt.plot(epochs_hist, history["lr_lora"], label="LR LoRA (exp)")
plt.plot(epochs_hist, history["lr_head"], label="LR Head (exp)")
plt.xlabel("Epoch")
plt.ylabel("Learning rate")
plt.title("Learning rate schedule (per-group exponential)")
plt.legend()
plt.grid(True, alpha=0.3)

save_fig("training_curves.png")


# =======================
# Clip-level aggregation (TEST) (UPDATED WITH TIMING)
# =======================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION")
print("=" * 60)

def early_stop_classwise(seq, k_map=None, default_k=3):
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = k_map.get(v, default_k)
        if run_len >= k_c:
            return v
    return None

test_sw_ds = dm.test_dataset
base_ds = getattr(test_sw_ds, "base", test_sw_ds)
n_clips = len(base_ds)

if hasattr(test_sw_ds, "_index"):
    clip_ids_for_windows = [pair[0] for pair in test_sw_ds._index]
else:
    clip_ids_for_windows = list(range(len(test_sw_ds)))

assert len(clip_ids_for_windows) == len(y_test_true), \
    f"Length mismatch: clip_ids_for_windows={len(clip_ids_for_windows)}, y_test_true={len(y_test_true)}"

clip_to_preds = defaultdict(list)
clip_to_targs = defaultdict(list)

for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
    clip_to_preds[cid].append(int(p))
    clip_to_targs[cid].append(int(t))

clip_targets = []
for cid in range(n_clips):
    if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
        clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
        clip_targets.append(int(base_ds.label_map[clean]))
    else:
        tgts = clip_to_targs.get(cid, [])
        clip_targets.append(max(set(tgts), key=tgts.count) if tgts else 0)

clip_targets = np.array(clip_targets, dtype=int)

clip_preds = []
missed_early = 0
for cid in range(n_clips):
    seq = clip_to_preds.get(cid, [])
    chosen = early_stop_classwise(seq, k_map=CLIP_K_MAP, default_k=CLIP_DEFAULT_K)
    if chosen is None:
        missed_early += 1
        chosen = max(set(seq), key=seq.count) if seq else int(clip_targets[cid])
    clip_preds.append(int(chosen))

clip_preds = np.array(clip_preds, dtype=int)

clip_acc = float((clip_preds == clip_targets).mean())
clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0)

# Timing for aggregation (minimal)
t0 = time.perf_counter()
_ = clip_preds
t1 = time.perf_counter()
clip_agg_ms_total = float(1000.0 * (t1 - t0))
clip_agg_ms_per_clip = float(clip_agg_ms_total / max(1, int(n_clips)))

# UPDATED: Add timing to clip metrics
save_json("metrics_clip.json", {
    "test": {
        "acc": clip_acc, 
        "prf": clip_prf,
        "timing": {
            "window_forward_ms_per_sample": float(bench["median_per_window_ms"]),
            "aggregation_ms_total": float(clip_agg_ms_total),
            "aggregation_ms_per_clip": float(clip_agg_ms_per_clip),
        },
    },
    "early_stop": {
        "default_k": int(CLIP_DEFAULT_K),
        "k_map_size": int(len(CLIP_K_MAP)),
        "missed_early": int(missed_early),
        "n_clips": int(n_clips),
    },
})

clip_summary = (
    f"CLIP-LEVEL TEST  acc={clip_acc:.4f}, "
    f"F1(macro/micro/weighted)={clip_prf['macro']['f1']:.4f}/{clip_prf['micro']['f1']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
    f"Early-stop config: default_k={CLIP_DEFAULT_K}, |k_map|={len(CLIP_K_MAP)}\n"
    f"Clips needing majority fallback (no early stop): {missed_early}/{n_clips}\n"
)
print(clip_summary)
save_text("metrics_summary_clip.txt", clip_summary)


# =======================
# Confusion matrices + reports (STANDARDIZED)
# =======================
class_names = [idx_to_name.get(i, f"Class {i}") for i in range(n_classes)]
if n_classes >= 25 and NEG_ID_EXPECTED < n_classes:
    class_names[NEG_ID_EXPECTED] = NEG_NAME
else:
    class_names[-1] = NEG_NAME

save_json("class_names.json", {"class_names": class_names})

def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title,
    class_names,
    out_png,
    cmap,
    annot=True,
    annot_fmt=".2f",
):
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    png_path = os.path.join(NORMWEAR_DIR, out_png)
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()

    npy_norm_path = os.path.join(NORMWEAR_DIR, out_png.replace(".png", "_norm01.npy"))
    npy_cnt_path = os.path.join(NORMWEAR_DIR, out_png.replace(".png", "_counts.npy"))
    np.save(npy_norm_path, cm_norm)
    np.save(npy_cnt_path, cm_counts)

    return cm_norm, cm_counts, png_path, npy_norm_path, npy_cnt_path


save_json("val_test_support.json", {
    "val_support": np.bincount(y_val_true, minlength=n_classes).tolist(),
    "test_support": np.bincount(y_test_true, minlength=n_classes).tolist(),
})

# Window-level: BLUE
cm_val_win_norm, cm_val_win_counts, _, _, _ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    "Confusion Matrix – VAL (window, LoRA NormWear + Residual MLP, normalized 0–1)",
    class_names, out_png="cm_val_window.png", cmap="Blues", annot=True
)

cm_test_win_norm, cm_test_win_counts, _, _, _ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    "Confusion Matrix – TEST (window, LoRA NormWear + Residual MLP, normalized 0–1)",
    class_names, out_png="cm_test_window.png", cmap="Blues", annot=True
)

# Clip-level: GREEN
cm_clip_norm, cm_clip_counts, _, _, _ = plot_conf_mat_norm01(
    clip_targets, clip_preds,
    "Confusion Matrix – TEST (clip, early-stop, LoRA NormWear + Residual MLP, normalized 0–1)",
    class_names, out_png="cm_test_clip.png", cmap="Greens", annot=True
)

val_report = classification_report(
    y_val_true, y_val_pred,
    labels=np.arange(n_classes),
    target_names=class_names,
    digits=3,
    zero_division=0
)
test_report = classification_report(
    y_test_true, y_test_pred,
    labels=np.arange(n_classes),
    target_names=class_names,
    digits=3,
    zero_division=0
)
clip_report = classification_report(
    clip_targets, clip_preds,
    labels=np.arange(n_classes),
    target_names=class_names,
    digits=3,
    zero_division=0
)

save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)
save_text("report_test_clip.txt", clip_report)


# =======================
# FINAL SUMMARY
# =======================
final_summary = (
    "FINAL SUMMARY (LoRA + HEAD)\n"
    f"LoRA params: {n_lora:,}, Head params: {n_head_train:,}, Backbone: {n_backbone_total:,} (frozen)\n"
    f"Best checkpoint: {SAVE_PATH}\n"
    f"VAL window:  loss={val_loss:.4f}, acc={val_acc:.4f}, "
    f"F1(macro/micro/weighted)={val_prf['macro']['f1']:.4f}/{val_prf['micro']['f1']:.4f}/{val_prf['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_loss:.4f}, acc={test_acc:.4f}, "
    f"F1(macro/micro/weighted)={test_prf['macro']['f1']:.4f}/{test_prf['micro']['f1']:.4f}/{test_prf['weighted']['f1']:.4f}\n"
    f"TEST clip:   acc={clip_acc:.4f}, "
    f"F1(macro/micro/weighted)={clip_prf['macro']['f1']:.4f}/{clip_prf['micro']['f1']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
    f"Inference (median/window ms): {bench['median_per_window_ms']:.3f} | "
    f"Throughput: {bench['throughput_windows_per_s']:.2f} windows/s (device={bench['device']})\n"
    f"Artifacts:   {NORMWEAR_DIR}\n"
)
save_text("final_summary.txt", final_summary)
print(final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", NORMWEAR_DIR)

In [ ]:
# ============================================================
# UCI-HAR Linear Probe: BASE vs LoRA 
# ============================================================

# ============================================================
# 0) OUTPUTS ROOT (prefer your Benchmark tree)
# ============================================================
def default_outputs_root() -> str:
    # You can hard-set this if you want:
    win_default = r"C:\Users\youss\Projects\Code\outputs\Benchmark"
    if os.path.exists(win_default):
        return win_default

    # workspace / cwd / tmp fallbacks
    if os.path.exists("/workspace") and os.access("/workspace", os.W_OK):
        return "/workspace/base/src/outputs/Benchmark"
    if os.access(os.getcwd(), os.W_OK):
        return os.path.join(os.getcwd(), "outputs/Benchmark")
    return "/tmp/outputs/Benchmark"


OUTPUTS_ROOT = os.environ.get("OUTPUTS_ROOT", default_outputs_root())


# ============================================================
# 1) Path Configuration - AUTO-DETECT or MANUAL NORMWEAR_DIR
# ============================================================
def autodetect_latest_lora_run(outputs_root: str) -> str:
    lora_root = os.path.join(outputs_root, "normwear_lora")
    if not os.path.exists(lora_root):
        raise FileNotFoundError(
            f"normwear_lora folder not found under OUTPUTS_ROOT:\n  {outputs_root}\n"
            f"Expected:\n  {lora_root}"
        )

    run_dirs = glob.glob(os.path.join(lora_root, "run_*"))
    if not run_dirs:
        raise FileNotFoundError(
            f"No run_* directories found in:\n  {lora_root}\n"
            "Train NormWear LoRA first or set NORMWEAR_DIR manually."
        )

    # newest first by name is usually OK since run_YYYYMMDD_HHMMSS
    run_dirs.sort(reverse=True)
    return run_dirs[0]


try:
    # If user already defined NORMWEAR_DIR in the notebook kernel, keep it
    NORMWEAR_DIR = NORMWEAR_DIR  # type: ignore[name-defined]
    print(f"✓ Using existing NORMWEAR_DIR: {NORMWEAR_DIR}")
except Exception:
    NORMWEAR_DIR = autodetect_latest_lora_run(OUTPUTS_ROOT)
    print(f"✓ Auto-detected latest NormWear LoRA run: {NORMWEAR_DIR}")

os.makedirs(NORMWEAR_DIR, exist_ok=True)


# ============================================================
# 2) Save helpers
# ============================================================
def save_text(name: str, text: str) -> str:
    path = os.path.join(NORMWEAR_DIR, name)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path


def save_json(name: str, obj) -> str:
    path = os.path.join(NORMWEAR_DIR, name)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    return path


def save_npy(name: str, arr) -> str:
    path = os.path.join(NORMWEAR_DIR, name)
    np.save(path, arr)
    return path


# ============================================================
# 3) CONFIG
# ============================================================
UCI_ROOT = os.environ.get("UCI_ROOT", r"C:/Users/youss/Projects/Code/data/UCI_HAR")
if not os.path.exists(UCI_ROOT):
    for alt in ["/workspace/data/UCI_HAR", "data/UCI_HAR", "../data/UCI_HAR"]:
        if os.path.exists(alt):
            UCI_ROOT = alt
            break

if not os.path.exists(UCI_ROOT):
    raise FileNotFoundError(
        "UCI-HAR dataset not found. Tried:\n"
        f"  - {UCI_ROOT}\n"
        "Set UCI_ROOT env var or place data at one of the fallback paths."
    )

BASE_NORMWEAR_CKPT = os.environ.get(
    "BASE_NORMWEAR_CKPT", "NormWear/normwear_last_checkpoint-15470-correct.pth"
)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BATCH_SIZE = 128
PATCH_POOL = "mean+max"
CHANNEL_POOL = "max"


# ============================================================
# 4) Robust checkpoint resolution
# ============================================================
def resolve_lora_ckpts(preferred_run_dir: str, outputs_root: str):
    """
    Returns:
      (weights_only_path_or_None, full_ckpt_path_or_None, resolution_log_text)
    Logic:
      1) Prefer expected files inside preferred_run_dir
      2) Else search under outputs_root (rglob), pick newest by mtime
    """
    preferred = Path(preferred_run_dir)
    w0 = preferred / "normwear_lora_weights_only.pt"
    f0 = preferred / "normwear_lora_finetune_best.pt"

    log_lines = []
    log_lines.append(f"resolve_lora_ckpts:")
    log_lines.append(f"  preferred_run_dir = {preferred_run_dir}")
    log_lines.append(f"  outputs_root      = {outputs_root}")

    if w0.exists() or f0.exists():
        log_lines.append("  Found at preferred_run_dir.")
        return (str(w0) if w0.exists() else None), (str(f0) if f0.exists() else None), "\n".join(log_lines) + "\n"

    # Fallback: search globally
    root = Path(outputs_root)
    cand_w = list(root.rglob("normwear_lora_weights_only.pt"))
    cand_f = list(root.rglob("normwear_lora_finetune_best.pt"))

    cand_w = sorted(cand_w, key=lambda p: p.stat().st_mtime, reverse=True)
    cand_f = sorted(cand_f, key=lambda p: p.stat().st_mtime, reverse=True)

    log_lines.append(f"  Not found in preferred_run_dir; searching recursively under outputs_root.")
    log_lines.append(f"  candidates weights_only = {len(cand_w)}")
    log_lines.append(f"  candidates full_ckpt    = {len(cand_f)}")

    w = str(cand_w[0]) if cand_w else None
    f = str(cand_f[0]) if cand_f else None

    if w:
        log_lines.append(f"  Selected weights_only (newest): {w}")
    if f:
        log_lines.append(f"  Selected full_ckpt    (newest): {f}")

    return w, f, "\n".join(log_lines) + "\n"


LORA_WEIGHTS_ONLY, LORA_FULL_CKPT, ckpt_resolve_log = resolve_lora_ckpts(NORMWEAR_DIR, OUTPUTS_ROOT)
print(ckpt_resolve_log)
save_text("ucihar_ckpt_resolution_log.txt", ckpt_resolve_log)


# ============================================================
# 5) Load LoRA config (prefer run dir config.json, else defaults)
# ============================================================
config_path = os.path.join(NORMWEAR_DIR, "config.json")
if os.path.exists(config_path):
    with open(config_path, "r", encoding="utf-8") as f:
        saved_config = json.load(f)
    lora_cfg = saved_config.get("LORA", saved_config.get("lora_config", {}))
    LORA_R = int(lora_cfg.get("r", 8))
    LORA_ALPHA = float(lora_cfg.get("alpha", 32))
    LORA_DROPOUT = float(lora_cfg.get("dropout", 0.1))
    LORA_TARGET_MODULES = list(lora_cfg.get("target_modules", ["qkv", "proj"]))
    LORA_LAST_K = int(lora_cfg.get("last_k", 2))
    LORA_INCLUDE_CLS_FUSION = bool(lora_cfg.get("include_cls_fusion", True))
    print("✓ Loaded LoRA config from saved config.json")
else:
    LORA_R = 8
    LORA_ALPHA = 32
    LORA_DROPOUT = 0.1
    LORA_TARGET_MODULES = ["qkv", "proj"]
    LORA_LAST_K = 2
    LORA_INCLUDE_CLS_FUSION = True
    print("⚠️  Using default LoRA config (config.json not found)")

run_meta = {
    "timestamp": time.strftime("%Y%m%d_%H%M%S"),
    "UCI_ROOT": UCI_ROOT,
    "BASE_NORMWEAR_CKPT": BASE_NORMWEAR_CKPT,
    "OUTPUTS_ROOT": OUTPUTS_ROOT,
    "NORMWEAR_DIR": NORMWEAR_DIR,
    "LORA_WEIGHTS_ONLY_RESOLVED": LORA_WEIGHTS_ONLY,
    "LORA_FULL_CKPT_RESOLVED": LORA_FULL_CKPT,
    "DEVICE": str(DEVICE),
    "BATCH_SIZE": BATCH_SIZE,
    "PATCH_POOL": PATCH_POOL,
    "CHANNEL_POOL": CHANNEL_POOL,
    "LORA": {
        "r": LORA_R,
        "alpha": LORA_ALPHA,
        "dropout": LORA_DROPOUT,
        "target_modules": LORA_TARGET_MODULES,
        "last_k": LORA_LAST_K,
        "include_cls_fusion": LORA_INCLUDE_CLS_FUSION,
    },
}
save_json("ucihar_linear_probe_config.json", run_meta)


# ============================================================
# 6) UCI-HAR loader
# ============================================================
def load_ucihar_split(root: str, split: str):
    base = os.path.join(root, split)
    sig_dir = os.path.join(base, "Inertial Signals")

    def read_txt(path: str):
        return np.loadtxt(path)

    acc_x = read_txt(os.path.join(sig_dir, f"body_acc_x_{split}.txt"))
    acc_y = read_txt(os.path.join(sig_dir, f"body_acc_y_{split}.txt"))
    acc_z = read_txt(os.path.join(sig_dir, f"body_acc_z_{split}.txt"))
    gyr_x = read_txt(os.path.join(sig_dir, f"body_gyro_x_{split}.txt"))
    gyr_y = read_txt(os.path.join(sig_dir, f"body_gyro_y_{split}.txt"))
    gyr_z = read_txt(os.path.join(sig_dir, f"body_gyro_z_{split}.txt"))

    X = np.stack([acc_x, acc_y, acc_z, gyr_x, gyr_y, gyr_z], axis=1)   # [N, 6, 128]
    X = np.transpose(X, (0, 2, 1)).astype(np.float32)                 # [N, 128, 6]
    y = np.loadtxt(os.path.join(base, f"y_{split}.txt")).astype(int) - 1
    return X, y


# ============================================================
# 7) LoRA layers (device-safe) + robust loader
# ============================================================
class LoRALayer(torch.nn.Module):
    def __init__(self, original_layer: torch.nn.Linear, r=8, alpha=16, dropout=0.05):
        super().__init__()
        self.original_layer = original_layer
        self.r = r
        self.scaling = alpha / r

        in_features = original_layer.in_features
        out_features = original_layer.out_features

        for p in self.original_layer.parameters():
            p.requires_grad = False

        dev = original_layer.weight.device
        self.lora_A = torch.nn.Parameter(torch.zeros(r, in_features, device=dev))
        self.lora_B = torch.nn.Parameter(torch.zeros(out_features, r, device=dev))

        torch.nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        torch.nn.init.zeros_(self.lora_B)

        self.dropout = torch.nn.Dropout(dropout) if dropout > 0 else torch.nn.Identity()

    def forward(self, x):
        original_output = self.original_layer(x)
        lora_output = self.dropout(x) @ self.lora_A.T @ self.lora_B.T * self.scaling
        return original_output + lora_output


def apply_lora_to_linear_layers(module, target_names, r, alpha, dropout, prefix=""):
    modified = []
    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name
        if isinstance(child, torch.nn.Linear):
            if any(t.lower() in name.lower() for t in target_names):
                setattr(module, name, LoRALayer(child, r=r, alpha=alpha, dropout=dropout))
                modified.append(full_name)
        else:
            modified.extend(apply_lora_to_linear_layers(child, target_names, r, alpha, dropout, full_name))
    return modified


def apply_lora_to_encoder_blocks(backbone, target_names, last_k, r, alpha, dropout, include_cls_fusion=True):
    modified = []
    blocks = getattr(backbone, "encoder_blocks", None)
    if blocks is None:
        print("backbone has no encoder_blocks; no LoRA applied.")
        return modified

    depth = len(blocks)
    if last_k and last_k > 0:
        start = max(0, depth - last_k)
        idxs = range(start, depth)
    else:
        idxs = range(depth)

    for i in idxs:
        layer = blocks[i]
        if hasattr(layer, "variate_encoder"):
            modified.extend(
                apply_lora_to_linear_layers(
                    layer.variate_encoder, target_names, r, alpha, dropout,
                    prefix=f"encoder_blocks.{i}.variate_encoder"
                )
            )
        if include_cls_fusion and getattr(layer, "cls_fusion", None) is not None:
            modified.extend(
                apply_lora_to_linear_layers(
                    layer.cls_fusion, target_names, r, alpha, dropout,
                    prefix=f"encoder_blocks.{i}.cls_fusion"
                )
            )
    return modified


def load_lora_weights_only(model, lora_path: str):
    raw = torch.load(lora_path, map_location="cpu")
    state = {}

    def add_alias(k, v):
        if k not in state:
            state[k] = v

    for k, v in raw.items():
        add_alias(k, v)

        if k.startswith("normwear."):
            add_alias(k[len("normwear."):], v)
        if k.startswith("model."):
            add_alias(k[len("model."):], v)

        k2 = k
        if k2.startswith("normwear."):
            k2 = k2[len("normwear."):]
        if k2.startswith("model."):
            k2 = k2[len("model."):]
        if k2.startswith("backbone."):
            add_alias(k2[len("backbone."):], v)

        if not k2.startswith("backbone."):
            add_alias("backbone." + k2, v)
        if not k2.startswith("normwear.backbone."):
            add_alias("normwear.backbone." + k2, v)

    n_layers = 0
    nA = 0
    nB = 0

    for name, m in model.named_modules():
        if isinstance(m, LoRALayer):
            n_layers += 1

            candidates_A = [
                f"{name}.lora_A",
                f"backbone.{name}.lora_A",
                f"normwear.{name}.lora_A",
                f"normwear.backbone.{name}.lora_A",
            ]
            candidates_B = [
                f"{name}.lora_B",
                f"backbone.{name}.lora_B",
                f"normwear.{name}.lora_B",
                f"normwear.backbone.{name}.lora_B",
            ]

            srcA = next((k for k in candidates_A if k in state), None)
            srcB = next((k for k in candidates_B if k in state), None)

            if srcA is not None:
                m.lora_A.data.copy_(state[srcA].to(m.lora_A.device))
                nA += 1
            if srcB is not None:
                m.lora_B.data.copy_(state[srcB].to(m.lora_B.device))
                nB += 1

    msg = (
        f"Loaded LoRA weights-only from: {lora_path}\n"
        f"LoRA layers found in model: {n_layers}\n"
        f"lora_A tensors loaded: {nA} | lora_B tensors loaded: {nB}\n"
    )
    print(msg)
    save_text("ucihar_lora_load_log.txt", msg)

    if nA == 0 and nB == 0:
        head = "No matches found. First 50 keys in file:\n"
        keys = "\n".join([str(k) for k in list(raw.keys())[:50]]) + "\n"
        print(head + keys)
        save_text("ucihar_lora_load_keys_head.txt", head + keys)


def try_load_from_full_ckpt(model, full_ckpt_path: str) -> bool:
    ckpt = torch.load(full_ckpt_path, map_location="cpu")
    if isinstance(ckpt, dict) and "model_state" in ckpt:
        missing, unexpected = model.load_state_dict(ckpt["model_state"], strict=False)
        msg = (
            f"Loaded full model_state from: {full_ckpt_path} | "
            f"missing={len(missing)} unexpected={len(unexpected)}\n"
        )
        print(msg)
        save_text("ucihar_full_ckpt_load_log.txt", msg)
        return True
    return False


# ============================================================
# 8) Embedding extraction
# ============================================================
@torch.no_grad()
def extract_normwear_embeddings(
    X,
    normwear,
    device,
    batch_size=128,
    patch_pool="mean",
    channel_pool="mean",
    desc="Embedding",
):
    def aggregate(out):
        if patch_pool == "mean":
            emb = out.mean(dim=2)
        elif patch_pool == "max":
            emb, _ = out.max(dim=2)
        elif patch_pool == "mean+max":
            mean_p = out.mean(dim=2)
            max_p, _ = out.max(dim=2)
            emb = torch.cat([mean_p, max_p], dim=-1)
        elif patch_pool == "cls":
            emb = out[:, :, 0, :]
        else:
            raise ValueError(f"Unknown patch_pool={patch_pool}")

        if channel_pool == "mean":
            emb = emb.mean(dim=1)
        elif channel_pool == "max":
            emb, _ = emb.max(dim=1)
        elif channel_pool == "mean+max":
            mean_c = emb.mean(dim=1)
            max_c, _ = emb.max(dim=1)
            emb = torch.cat([mean_c, max_c], dim=-1)
        elif channel_pool == "concat":
            emb = emb.reshape(emb.shape[0], -1)
        else:
            raise ValueError(f"Unknown channel_pool={channel_pool}")
        return emb

    normwear.eval()
    N = X.shape[0]
    feats = []

    for i in tqdm(range(0, N, batch_size), desc=desc):
        xb = torch.from_numpy(X[i : i + batch_size]).to(device)  # [B, T, C]
        x_nw = xb.permute(0, 2, 1).contiguous()                   # [B, C, T]
        spec = normwear.calc_cwt(x_nw, device=device).float()

        fn = normwear.backbone.get_signal_embedding
        if hasattr(fn, "__wrapped__"):
            out = fn.__wrapped__(normwear.backbone, spec, hidden_out=False, device=device)
        else:
            out = fn(spec, hidden_out=False, device=device)

        feats.append(aggregate(out).cpu().numpy())

    return np.concatenate(feats, axis=0)


# ============================================================
# 9) Metrics helpers
# ============================================================
def prf_all_averages(y_true, y_pred, zero_division=0):
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out


def compute_all_metrics(y_true, y_pred, y_prob=None, n_classes=None):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    acc = float((y_true == y_pred).mean())
    prf = prf_all_averages(y_true, y_pred, zero_division=0)

    out = {"acc": acc, "prf": prf}

    if y_prob is not None:
        try:
            out["auc_ovr"] = float(roc_auc_score(y_true, y_prob, multi_class="ovr"))
        except Exception as e:
            out["auc_ovr"] = None
            out["auc_ovr_error"] = str(e)
    else:
        out["auc_ovr"] = None

    if n_classes is None:
        n_classes = int(max(y_true.max(), y_pred.max())) + 1
    out["support"] = np.bincount(y_true, minlength=n_classes).astype(int).tolist()
    return out


# ============================================================
# 10) Confusion matrix plotting (row-normalized 0..1)
# ============================================================
def plot_conf_mat_norm01_from_labels(
    y_true,
    y_pred,
    title,
    class_names,
    out_png,
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
):
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(8, 0.85 * len(class_names)), max(6, 0.75 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    png_path = os.path.join(NORMWEAR_DIR, out_png)
    plt.tight_layout()
    plt.savefig(png_path, dpi=200, bbox_inches="tight")
    plt.close()

    npy_norm_path = os.path.join(NORMWEAR_DIR, out_png.replace(".png", "_norm01.npy"))
    npy_cnt_path = os.path.join(NORMWEAR_DIR, out_png.replace(".png", "_counts.npy"))
    np.save(npy_norm_path, cm_norm)
    np.save(npy_cnt_path, cm_counts)

    return cm_norm, cm_counts, png_path, npy_norm_path, npy_cnt_path


# ============================================================
# 11) Linear probe
# ============================================================
def train_eval_linear_probe(Z_train, y_train, Z_test, y_test):
    scaler = StandardScaler()
    Z_train_s = scaler.fit_transform(Z_train)
    Z_test_s = scaler.transform(Z_test)

    base = LogisticRegression(
        max_iter=5000,
        solver="lbfgs",
        class_weight="balanced",
        random_state=42,
    )
    clf = OneVsRestClassifier(base)
    clf.fit(Z_train_s, y_train)

    y_pred = clf.predict(Z_test_s)

    y_prob = None
    if hasattr(clf, "predict_proba"):
        y_prob = clf.predict_proba(Z_test_s)

    f1_macro = float(f1_score(y_test, y_pred, average="macro"))
    auc_ovr = None
    if y_prob is not None:
        try:
            auc_ovr = float(roc_auc_score(y_test, y_prob, multi_class="ovr"))
        except Exception:
            auc_ovr = None

    return f1_macro, auc_ovr, y_pred, y_prob


# ============================================================
# 12) Load backbone helper
# ============================================================
def load_normwear_backbone(ckpt_path, device):
    from NormWear.main_model import NormWearModel

    normwear = NormWearModel(weight_path="", optimized_cwt=True).to(device)
    ckpt = torch.load(ckpt_path, map_location="cpu", weights_only=False)

    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt

    missing, unexpected = normwear.backbone.load_state_dict(state_dict, strict=False)
    print(f"Loaded backbone from {ckpt_path} | missing={len(missing)} unexpected={len(unexpected)}")
    return normwear


# ============================================================
# 13) Main
# ============================================================
def main():
    print("\n" + "=" * 80)
    print("UCI-HAR LINEAR PROBE: BASE vs LoRA (ROBUST CKPT RESOLUTION)")
    print("=" * 80)
    print(f"DEVICE: {DEVICE}")
    print(f"UCI_ROOT: {UCI_ROOT}")
    print(f"OUTPUTS_ROOT: {OUTPUTS_ROOT}")
    print(f"NORMWEAR_DIR: {NORMWEAR_DIR}")
    print(f"Resolved LoRA weights-only: {LORA_WEIGHTS_ONLY}")
    print(f"Resolved LoRA full ckpt   : {LORA_FULL_CKPT}")
    print("=" * 80 + "\n")

    has_weights_only = (LORA_WEIGHTS_ONLY is not None) and os.path.exists(LORA_WEIGHTS_ONLY)
    has_full_ckpt = (LORA_FULL_CKPT is not None) and os.path.exists(LORA_FULL_CKPT)

    if not has_weights_only and not has_full_ckpt:
        error_msg = (
            "Neither LoRA checkpoint file exists.\n"
            f"Resolved paths:\n  weights_only={LORA_WEIGHTS_ONLY}\n  full_ckpt={LORA_FULL_CKPT}\n\n"
            f"Search root was:\n  {OUTPUTS_ROOT}\n"
            "Fix training paths or point OUTPUTS_ROOT/NORMWEAR_DIR correctly."
        )
        print(error_msg)
        save_text("ucihar_error_log.txt", error_msg)
        raise FileNotFoundError(error_msg)

    print("✓ LoRA checkpoints found:")
    print(f"  - Weights-only: {has_weights_only}")
    print(f"  - Full checkpoint: {has_full_ckpt}\n")

    # Load UCI-HAR
    X_train, y_train = load_ucihar_split(UCI_ROOT, "train")
    X_test, y_test = load_ucihar_split(UCI_ROOT, "test")
    print(f"Train windows: {X_train.shape}, Test windows: {X_test.shape}\n")

    activity_names = ["WALKING", "WALK_UP", "WALK_DOWN", "SITTING", "STANDING", "LAYING"]
    n_classes = len(activity_names)
    save_json("ucihar_class_names.json", {"class_names": activity_names})

    # ---------------- BASE ----------------
    print("=" * 60)
    print("EVALUATING BASE MODEL")
    print("=" * 60)

    base_model = NormWearModel(weight_path="", optimized_cwt=True).to(device)

    ckpt = torch.load(NORMWEAR_ENCODER_CKPT, map_location="cpu", weights_only=False)
    if isinstance(ckpt, dict) and "model" in ckpt:
        state_dict = ckpt["model"]
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        state_dict = ckpt["state_dict"]
    else:
        state_dict = ckpt
    
    missing, unexpected = base_model.backbone.load_state_dict(state_dict, strict=False)
    print("Loaded NormWear pretrained backbone.")
    print("Missing keys:", len(missing))
    print("Unexpected keys:", len(unexpected))
    
    # Freeze entire backbone
    for p in base_model.backbone.parameters():
        p.requires_grad = False
    
    base_model.eval()  # Set to eval mode permanently

    Ztr_base = extract_normwear_embeddings(
        X_train, base_model, DEVICE, BATCH_SIZE, PATCH_POOL, CHANNEL_POOL, desc="Embed train (BASE)"
    )
    Zte_base = extract_normwear_embeddings(
        X_test, base_model, DEVICE, BATCH_SIZE, PATCH_POOL, CHANNEL_POOL, desc="Embed test  (BASE)"
    )
    _, _, pred_base, prob_base = train_eval_linear_probe(Ztr_base, y_train, Zte_base, y_test)

    save_npy("ucihar_Ztr_base.npy", Ztr_base)
    save_npy("ucihar_Zte_base.npy", Zte_base)
    save_npy("ucihar_pred_base.npy", pred_base)
    if prob_base is not None:
        save_npy("ucihar_prob_base.npy", prob_base)

    metrics_base = compute_all_metrics(y_test, pred_base, y_prob=prob_base, n_classes=n_classes)
    rep_base = classification_report(y_test, pred_base, target_names=activity_names, digits=4, zero_division=0)
    save_text("ucihar_report_base.txt", rep_base)
    save_json("ucihar_metrics_base.json", metrics_base)

    cm_base_norm, cm_base_counts, cm_base_png, cm_base_npy_norm, cm_base_npy_counts = plot_conf_mat_norm01_from_labels(
        y_test, pred_base,
        title="UCI-HAR Confusion Matrix - BASE (row-normalized 0-1)",
        class_names=activity_names,
        out_png="ucihar_cm_base.png",
        cmap="Blues",
        annot=True,
        annot_fmt=".2f",
    )

    print(f"✓ BASE: acc={metrics_base['acc']:.4f}, F1(macro)={metrics_base['prf']['macro']['f1']:.4f}\n")

    # ---------------- LoRA ----------------
    print("=" * 60)
    print("EVALUATING LoRA MODEL")
    print("=" * 60)

    lora_model = load_normwear_backbone(BASE_NORMWEAR_CKPT, DEVICE)

    modified = apply_lora_to_encoder_blocks(
        lora_model.backbone,
        target_names=LORA_TARGET_MODULES,
        last_k=LORA_LAST_K,
        r=LORA_R,
        alpha=LORA_ALPHA,
        dropout=LORA_DROPOUT,
        include_cls_fusion=LORA_INCLUDE_CLS_FUSION
    )
    print(f"Applied LoRA to {len(modified)} linear layers.")
    save_json("ucihar_lora_injected_modules.json", {"modified": modified})

    # Ensure LoRA params live on DEVICE
    lora_model = lora_model.to(DEVICE)

    loaded = False
    load_mode = None
    if os.path.exists(LORA_WEIGHTS_ONLY):
        load_lora_weights_only(lora_model, LORA_WEIGHTS_ONLY)
        loaded = True
        load_mode = "weights_only"
    elif os.path.exists(LORA_FULL_CKPT):
        loaded = try_load_from_full_ckpt(lora_model, LORA_FULL_CKPT)
        load_mode = "full_ckpt" if loaded else None

    if not loaded:
        raise FileNotFoundError("Could not load LoRA weights from the paths in NORMWEAR_DIR.")

    for p in lora_model.parameters():
        p.requires_grad = False
    lora_model.eval()

    Ztr_lora = extract_normwear_embeddings(
        X_train, lora_model, DEVICE, BATCH_SIZE, PATCH_POOL, CHANNEL_POOL, desc="Embed train (LoRA)"
    )
    Zte_lora = extract_normwear_embeddings(
        X_test, lora_model, DEVICE, BATCH_SIZE, PATCH_POOL, CHANNEL_POOL, desc="Embed test  (LoRA)"
    )
    _, _, pred_lora, prob_lora = train_eval_linear_probe(Ztr_lora, y_train, Zte_lora, y_test)

    save_npy("ucihar_Ztr_lora.npy", Ztr_lora)
    save_npy("ucihar_Zte_lora.npy", Zte_lora)
    save_npy("ucihar_pred_lora.npy", pred_lora)
    if prob_lora is not None:
        save_npy("ucihar_prob_lora.npy", prob_lora)

    metrics_lora = compute_all_metrics(y_test, pred_lora, y_prob=prob_lora, n_classes=n_classes)
    rep_lora = classification_report(y_test, pred_lora, target_names=activity_names, digits=4, zero_division=0)
    save_text("ucihar_report_lora.txt", rep_lora)
    save_json("ucihar_metrics_lora.json", metrics_lora)

    cm_lora_norm, cm_lora_counts, cm_lora_png, cm_lora_npy_norm, cm_lora_npy_counts = plot_conf_mat_norm01_from_labels(
        y_test, pred_lora,
        title="UCI-HAR Confusion Matrix - LoRA (row-normalized 0-1)",
        class_names=activity_names,
        out_png="ucihar_cm_lora.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
    )

    print(f"✓ LoRA: acc={metrics_lora['acc']:.4f}, F1(macro)={metrics_lora['prf']['macro']['f1']:.4f}\n")

    # ---------------- Summary ----------------
    print("\n" + "=" * 80)
    print("FINAL RESULTS")
    print("=" * 80)

    def fmt_auc(x):
        return f"{x:.4f}" if isinstance(x, (float, int)) else "NA"

    summary = (
        "UCI-HAR Linear Probe Results\n"
        f"BASE | acc={metrics_base['acc']:.4f} | "
        f"F1(macro/micro/weighted)={metrics_base['prf']['macro']['f1']:.4f}/"
        f"{metrics_base['prf']['micro']['f1']:.4f}/{metrics_base['prf']['weighted']['f1']:.4f} | "
        f"AUC(ovr)={fmt_auc(metrics_base.get('auc_ovr'))}\n"
        f"LoRA | acc={metrics_lora['acc']:.4f} | "
        f"F1(macro/micro/weighted)={metrics_lora['prf']['macro']['f1']:.4f}/"
        f"{metrics_lora['prf']['micro']['f1']:.4f}/{metrics_lora['prf']['weighted']['f1']:.4f} | "
        f"AUC(ovr)={fmt_auc(metrics_lora.get('auc_ovr'))}\n"
        f"Delta acc         = {metrics_lora['acc'] - metrics_base['acc']:+.4f}\n"
        f"Delta F1(macro)   = {metrics_lora['prf']['macro']['f1'] - metrics_base['prf']['macro']['f1']:+.4f}\n"
        f"Delta F1(micro)   = {metrics_lora['prf']['micro']['f1'] - metrics_base['prf']['micro']['f1']:+.4f}\n"
        f"Delta F1(weighted)= {metrics_lora['prf']['weighted']['f1'] - metrics_base['prf']['weighted']['f1']:+.4f}\n"
        f"LoRA load mode    = {load_mode}\n"
        f"Resolved ckpts    = weights_only={LORA_WEIGHTS_ONLY}, full_ckpt={LORA_FULL_CKPT}\n"
        f"Saved CMs         = {os.path.basename(cm_base_png)}, {os.path.basename(cm_lora_png)}\n"
    )

    print(summary)
    save_text("ucihar_linear_probe_summary.txt", summary)

    save_json(
        "ucihar_linear_probe_metrics.json",
        {
            "base": metrics_base,
            "lora": metrics_lora,
            "delta": {
                "acc": float(metrics_lora["acc"] - metrics_base["acc"]),
                "f1_macro": float(metrics_lora["prf"]["macro"]["f1"] - metrics_base["prf"]["macro"]["f1"]),
                "f1_micro": float(metrics_lora["prf"]["micro"]["f1"] - metrics_base["prf"]["micro"]["f1"]),
                "f1_weighted": float(metrics_lora["prf"]["weighted"]["f1"] - metrics_base["prf"]["weighted"]["f1"]),
                "auc_ovr": (
                    float(metrics_lora["auc_ovr"] - metrics_base["auc_ovr"])
                    if (metrics_lora.get("auc_ovr") is not None and metrics_base.get("auc_ovr") is not None)
                    else None
                ),
            },
            "load_mode": load_mode,
            "artifacts": {
                "cm_base_png": cm_base_png,
                "cm_lora_png": cm_lora_png,
                "cm_base_counts_npy": cm_base_npy_counts,
                "cm_base_norm01_npy": cm_base_npy_norm,
                "cm_lora_counts_npy": cm_lora_npy_counts,
                "cm_lora_norm01_npy": cm_lora_npy_norm,
            },
        },
    )

    print("\n" + "=" * 80)
    print(f"✓ All results saved to: {NORMWEAR_DIR}")
    print("=" * 80 + "\n")


if __name__ == "__main__":
    main()

## Transformer Model

In [ ]:
# ===== Models (TCN, CNN+Transformer, Lightning wrapper) =====
# ---------------- TCN primitives ----------------
class Chomp1d(nn.Module):
    def __init__(self, chomp_size):
        super().__init__()
        self.chomp_size = chomp_size
    def forward(self, x):
        return x[:, :, :-self.chomp_size].contiguous()

class TemporalBlock(nn.Module):
    def __init__(self, n_inputs, n_outputs, kernel_size, stride, dilation, padding, dropout=0.2):
        super().__init__()
        self.conv1 = weight_norm(nn.Conv1d(n_inputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp1 = Chomp1d(padding)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(dropout)

        self.conv2 = weight_norm(nn.Conv1d(n_outputs, n_outputs, kernel_size,
                                           stride=stride, padding=padding, dilation=dilation))
        self.chomp2 = Chomp1d(padding)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(dropout)

        self.net = nn.Sequential(self.conv1, self.chomp1, self.relu1, self.dropout1,
                                 self.conv2, self.chomp2, self.relu2, self.dropout2)
        self.downsample = nn.Conv1d(n_inputs, n_outputs, 1) if n_inputs != n_outputs else None
        self.relu = nn.ReLU()
        self.init_weights()

    def init_weights(self):
        self.conv1.weight.data.normal_(0, 0.01)
        self.conv2.weight.data.normal_(0, 0.01)
        if self.downsample is not None:
            self.downsample.weight.data.normal_(0, 0.01)

    def forward(self, x):
        out = self.net(x)
        res = x if self.downsample is None else self.downsample(x)
        return self.relu(out + res)

class TemporalConvNet(nn.Module):
    def __init__(self, num_inputs, num_channels, kernel_size=2, dropout=0.2):
        super().__init__()
        layers = []
        for i in range(len(num_channels)):
            dilation_size = 2 ** i
            in_channels = num_inputs if i == 0 else num_channels[i-1]
            out_channels = num_channels[i]
            layers += [TemporalBlock(in_channels, out_channels, kernel_size, stride=1,
                                     dilation=dilation_size,
                                     padding=(kernel_size-1) * dilation_size,
                                     dropout=dropout)]
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x)

class TimeSeriesCNNTransformer(nn.Module):
    def __init__(self, input_length, **kwargs):
        super().__init__()
        args = {
            "in_channels": 6,           # CNN branch channels (will become 24)
            "stats_in_channels": None,  # stats/Transformer channels (we'll set to 6)
            "n_classes": 10,
            "conv_channels": [16, 32],
            "kernel_sizes": [7, 5],
            "strides": [1, 1],
            "pool_sizes": [2, 2],
            "dropout": 0.4,
            "transformer_layers": 1,
            "transformer_heads": 2,
            "transformer_dim_feedforward": 64,
            "embed_dim": 32,
            "max_seq_len": 300,
        }
        args.update(kwargs)

        # ---------- channel configs ----------
        self.in_channels = args["in_channels"]          # CNN input (e.g. 24)
        self.stats_in_channels = (
            args["stats_in_channels"] or self.in_channels
        )                                               # stats input (e.g. 6)

        self.n_classes = args["n_classes"]
        self.dropout_rate = args["dropout"]
        self.embed_dim = args["embed_dim"]

        # ---------- Per-channel residual conv stack (CNN branch) ----------
        conv_layers, relu_layers, residual_convs = [], [], []
        for i in range(len(args["conv_channels"])):
            in_ch = 1 if i == 0 else args["conv_channels"][i - 1]
            out_ch = args["conv_channels"][i]
            k = args["kernel_sizes"][i]
            s = args["strides"][i]

            conv_layers.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=s, padding=k // 2)
            )
            relu_layers.append(nn.ReLU())
            residual_convs.append(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=s)
                if in_ch != out_ch else nn.Identity()
            )

        self.conv_layers = nn.ModuleList(conv_layers)
        self.relu_layers = nn.ModuleList(relu_layers)
        self.residual_convs = nn.ModuleList(residual_convs)
        self.conv_dropout = nn.Dropout(self.dropout_rate)

        # ---------- Transformer (stats stream only) ----------
        # input_proj kept for parity (still unused)
        self.input_proj = nn.Conv1d(self.in_channels, self.embed_dim, kernel_size=1)
        # IMPORTANT: stats_proj now uses stats_in_channels (raw 6 ch), not in_channels
        self.stats_proj = nn.Conv1d(
            self.stats_in_channels, self.embed_dim, kernel_size=1
        )

        self.register_buffer(
            "pos_encoding",
            self._generate_pos_encoding(self.embed_dim, args["max_seq_len"])
        )
        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=args["transformer_heads"],
            dim_feedforward=args["transformer_dim_feedforward"],
            dropout=self.dropout_rate,
            batch_first=False,
        )
        self.transformer = nn.TransformerEncoder(
            enc_layer, num_layers=args["transformer_layers"]
        )

        self.query = nn.Parameter(torch.randn(1, 1, self.embed_dim))
        self.mha = nn.MultiheadAttention(
            self.embed_dim, args["transformer_heads"], batch_first=False
        )

        # ---------- Fusion heads ----------
        # local_dim = (#CNN channels) * (last conv channels)
        local_dim = self.in_channels * args["conv_channels"][-1]
        fusion_dim = local_dim + self.embed_dim

        self.local_head = nn.Sequential(
            nn.Linear(local_dim, 64),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )
        self.attn_head = nn.Sequential(
            nn.Linear(self.embed_dim, 64),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )
        self.fused_head = nn.Sequential(
            nn.Linear(fusion_dim, 64),
            nn.ReLU(),
            nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )

        self.w = nn.Parameter(torch.ones(3))
        self.temporal_pool = nn.Conv1d(
            in_channels=args["conv_channels"][-1],
            out_channels=args["conv_channels"][-1],
            kernel_size=input_length,  # full-seq pooling
            bias=False,
        )

    def _generate_pos_encoding(self, d_model, max_len):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(1)  # (max_len, 1, d_model)

    def forward(self, x, x_stats):
        """
        x:       (B, C_cnn,   T)       e.g. C_cnn = 24 (6 axes × 4 bands)
        x_stats: (B, C_stats, T_stats) e.g. C_stats = 6 (raw accel+gyro only)
        """
        B, C, T = x.size()
        B_s, C_s, Tstats = x_stats.size()

        # sanity checks
        assert C == self.in_channels, \
            f"Expected x with {self.in_channels} channels, got {C}"
        assert C_s == self.stats_in_channels, \
            f"Expected x_stats with {self.stats_in_channels} channels, got {C_s}"
        assert B_s == B, "Batch size mismatch between x and x_stats"

        # ---------- Per-channel conv branch (over filterbank channels) ----------
        local_feats = []
        for ch in range(C):
            xi = x[:, ch:ch + 1, :]  # (B, 1, T)
            out = xi
            for conv, relu, res_conv in zip(
                self.conv_layers, self.relu_layers, self.residual_convs
            ):
                res = res_conv(out)
                out = relu(conv(out) + res)

            pooled = self.temporal_pool(out).squeeze(-1)  # (B, conv_channels[-1])
            local_feats.append(pooled)

        local_feat = torch.cat(local_feats, dim=1)  # (B, C_cnn * last_conv_ch)
        local_feat = self.conv_dropout(local_feat)

        # ---------- Stats → Transformer (raw-only stats branch) ----------
        stats_proj = self.stats_proj(x_stats).permute(2, 0, 1)  # (Tstats, B, D)
        pe = self.pos_encoding[:Tstats].to(x.device)
        attn_in = stats_proj + pe
        attn_out = self.transformer(attn_in)

        # ---------- Attention pooling with learnable query ----------
        q = self.query.expand(-1, B, -1)  # (1, B, D)
        attn_pooled, _ = self.mha(q, attn_out, attn_out)
        attn_pooled = attn_pooled.squeeze(0)  # (B, D)

        # ---------- Heads & fusion ----------
        fusion = torch.cat([local_feat, attn_pooled], dim=1)
        pred_local = self.local_head(local_feat)
        pred_attn  = self.attn_head(attn_pooled)
        pred_fused = self.fused_head(fusion)

        w = torch.softmax(self.w, dim=0)
        return w[0] * pred_local + w[1] * pred_attn + w[2] * pred_fused

# -------------- Pure Transformer (optional – kept for completeness) --------------
from torchmetrics import Accuracy, Precision, Recall, F1Score, MetricCollection

class GestureClassifier(pl.LightningModule):
    def __init__(
        self,
        input_length,
        n_classes,
        in_channels,
        stats_in_channels: int | None = None,   # NEW
        class_weights: torch.Tensor | None = None,
        learning_rate: float = 5e-4,
        weight_decay: float = 1e-3,
        **model_kwargs
    ):
        super().__init__()
        self.save_hyperparameters()
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay

        # if stats_in_channels not given, default: same as CNN input
        if stats_in_channels is None:
            stats_in_channels = in_channels

        self.model = TimeSeriesCNNTransformer(
            input_length=input_length,
            n_classes=n_classes,
            in_channels=in_channels,              # CNN branch (e.g. 24)
            stats_in_channels=stats_in_channels,  # stats branch (e.g. 6)
            **model_kwargs,
        )

        if class_weights is not None:
            self.register_buffer("ce_weights", class_weights.float())
            self.criterion = nn.CrossEntropyLoss(weight=self.ce_weights)
        else:
            self.ce_weights = None
            self.criterion = nn.CrossEntropyLoss()
           
        task = dict(task="multiclass", num_classes=n_classes)

        def coll():
            return MetricCollection({
                "acc_overall": Accuracy(average="micro", **task),
                "acc_macro":   Accuracy(average="macro", **task),
                "precision":   Precision(average="macro", **task, zero_division=0),
                "recall":      Recall(average="macro", **task, zero_division=0),
                "f1":          F1Score(average="macro", **task, zero_division=0),
                "f1_weighted": F1Score(average="weighted", **task, zero_division=0),
            })

        self.train_metrics = coll()
        self.val_metrics   = coll()
        self.test_metrics  = coll()

    def forward(self, x, x_stats):
        return self.model(x, x_stats)

    def _step(self, batch):
        x, x_stats, y = batch
        # x:       (B, T, C_cnn)    e.g. (B, 100, 24)
        # x_stats: (B, T_stats, C_stats) e.g. (B, 100, 6)
        x       = x.permute(0, 2, 1)       # → (B, C_cnn,   T)
        x_stats = x_stats.permute(0, 2, 1) # → (B, C_stats, T_stats)
        logits  = self(x, x_stats)
        loss    = self.criterion(logits, y)
        preds   = torch.argmax(logits, dim=1)
        return loss, preds, y


    def training_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.train_metrics.update(preds, y)
        # epoch metrics only (clean CSV), lr on bar
        self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)
        opt = self.optimizers()
        if opt: self.log("lr", opt.param_groups[0]["lr"], on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.val_metrics.update(preds, y)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)

    def test_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.test_metrics.update(preds, y)
        self.log("test/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)

    def on_train_epoch_end(self):
        m = self.train_metrics.compute()
        for k, v in m.items():
            self.log(f"train/{k}", v, prog_bar=(k in ["acc_overall","f1","f1_weighted"]), sync_dist=True)
        self.train_metrics.reset()
        # also log the fusion weights once per epoch
        w = self.model.w.detach().cpu()
        self.log_dict({ "epoch/w_0": w[0].item(), "epoch/w_1": w[1].item(), "epoch/w_2": w[2].item() }, sync_dist=True)

    def on_validation_epoch_end(self):
        m = self.val_metrics.compute()
        for k, v in m.items():
            self.log(f"val/{k}", v, prog_bar=(k in ["acc_overall","f1","f1_weighted"]), sync_dist=True)
        self.val_metrics.reset()

    def on_test_epoch_end(self):
        m = self.test_metrics.compute()
        for k, v in m.items():
            self.log(f"test/{k}", v, prog_bar=(k in ["acc_overall","f1","f1_weighted"]), sync_dist=True)
        self.test_metrics.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=5, factor=0.5)
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val/loss", "frequency": 1, "strict": True}}

    # Peek one sample to infer shapes
sample_x, sample_x_stats, sample_y = dm_transformer.train_dataset[0]
seq_len          = sample_x.shape[0]   # T
in_channels      = sample_x.shape[1]   # CNN input channels (now includes filterbank)
stats_in_channels = sample_x_stats.shape[1]  # Transformer stats channels (raw-only)

# Use label_map directly on train_dataset (works for both ETHGestureDataset and SlidingWindowETHDataset)
label_map = dm_transformer.train_dataset.label_map
n_classes = len(set(label_map.values()))

print(f"\nSequence length: {seq_len}, CNN channels: {in_channels}, "
      f"Stats channels: {stats_in_channels}, Classes: {n_classes}")

print("\nClass Mapping:")
for name, cid in label_map.items():
    print(f"  {name} -> Class {cid}")

print(f"\nTrain samples: {len(dm_transformer.train_dataset)}")
print(f"Val samples:   {len(dm_transformer.val_dataset)}")
print(f"Test samples:  {len(dm_transformer.test_dataset)}")

# Count samples per class for weights (still based on base clips via .samples)
train_labels = []
for s in dm_transformer.train_dataset.samples:
    clean = re.sub(r'\(\d+\)$', '', s['label'])
    train_labels.append(label_map[clean])
counts = Counter(train_labels)
freq = torch.tensor([counts.get(i, 0) for i in range(n_classes)], dtype=torch.float)
class_weights = (freq.sum() / (n_classes * freq.clamp(min=1.0))).float()
print("\nUsing class weights:", [round(x, 4) for x in class_weights.tolist()])

# Hyperparams (reuse globals if set)
LR = globals().get("LR", 5e-4)
WEIGHT_DECAY = globals().get("WEIGHT_DECAY", 1e-3)
MAX_EPOCHS = 1

# Build model pass both in_channels and stats_in_channels
input_length = seq_len

standard_model = GestureClassifier(
    input_length=input_length,
    n_classes=n_classes,
    in_channels=in_channels,              # e.g. 24 when IMU filterbank is enabled
    stats_in_channels=stats_in_channels,  # e.g. 6 (raw accel+gyro)
    class_weights=class_weights,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
)
print(f"\nModel params: {sum(p.numel() for p in standard_model.parameters()):,}")


# ============================================================
# 0) CONFIG
# ============================================================
SEED = int(globals().get("SEED", 42))
BATCH_SIZE = int(globals().get("BATCH_SIZE", 16))  # if your DM uses it
MAX_EPOCHS = int(globals().get("MAX_EPOCHS", 100))
PATIENCE = int(globals().get("PATIENCE", 5))

LR = float(globals().get("LR", 5e-4))
LR = 0.001
WEIGHT_DECAY = float(globals().get("WEIGHT_DECAY", 1e-3))

# Exponential LR scheduler target: lr_end ~= lr0 * DECAY_TARGET by last epoch
DECAY_TARGET = float(globals().get("DECAY_TARGET", 0.01))  # decay to ~1% over MAX_EPOCHS
GAMMA = float(math.exp(math.log(DECAY_TARGET) / max(1, int(MAX_EPOCHS))))

# Clip-level aggregation config (optional)
CLIP_DEFAULT_K = int(globals().get("CLIP_DEFAULT_K", 3))
CLIP_K_MAP = globals().get("CLIP_K_MAP", None)  # dict {class_id: k} or None

# Output root
OUTPUTS_ROOT = globals().get("OUTPUTS", "./outputs")


# ============================================================
# 1) REPRODUCIBILITY
# ============================================================
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


# ============================================================
# 2) RUN DIRECTORY + SAVE HELPERS
# ============================================================
TRANSFORMER_ROOT = os.path.join(OUTPUTS_ROOT, "transformer")
Path(TRANSFORMER_ROOT).mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
TRANSFORMER_DIR = os.path.join(TRANSFORMER_ROOT, f"run_{RUN_ID}")
Path(TRANSFORMER_DIR).mkdir(parents=True, exist_ok=True)

def tpath(filename: str) -> str:
    return os.path.join(TRANSFORMER_DIR, filename)

def save_text(filename: str, text: str) -> str:
    path = tpath(filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

def save_json(filename: str, obj) -> str:
    path = tpath(filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    return path

def save_csv(filename: str, df: pd.DataFrame, index: bool = False) -> str:
    path = tpath(filename)
    df.to_csv(path, index=index)
    return path

def save_npy(filename: str, arr: np.ndarray) -> str:
    path = tpath(filename)
    np.save(path, arr)
    return path

def save_fig(filename: str, dpi: int = 300) -> str:
    path = tpath(filename)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path

print("Transformer run directory:", TRANSFORMER_DIR)


# ============================================================
# 3) MODEL: CNN+Transformer
# ============================================================
class TimeSeriesCNNTransformer(nn.Module):
    def __init__(self, input_length, **kwargs):
        super().__init__()
        args = {
            "in_channels": 6,
            "stats_in_channels": None,
            "n_classes": 10,
            "conv_channels": [16, 32],
            "kernel_sizes": [7, 5],
            "strides": [1, 1],
            "pool_sizes": [2, 2],
            "dropout": 0.4,
            "transformer_layers": 1,
            "transformer_heads": 2,
            "transformer_dim_feedforward": 64,
            "embed_dim": 32,
            "max_seq_len": 300,
        }
        args.update(kwargs)

        self.in_channels = int(args["in_channels"])
        self.stats_in_channels = int(args["stats_in_channels"] or self.in_channels)
        self.n_classes = int(args["n_classes"])
        self.dropout_rate = float(args["dropout"])
        self.embed_dim = int(args["embed_dim"])

        # ---- CNN branch (per-channel conv stack) ----
        conv_layers, relu_layers, residual_convs = [], [], []
        for i in range(len(args["conv_channels"])):
            in_ch = 1 if i == 0 else int(args["conv_channels"][i - 1])
            out_ch = int(args["conv_channels"][i])
            k = int(args["kernel_sizes"][i])
            s = int(args["strides"][i])

            conv_layers.append(nn.Conv1d(in_ch, out_ch, kernel_size=k, stride=s, padding=k // 2))
            relu_layers.append(nn.ReLU())
            residual_convs.append(nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=s) if in_ch != out_ch else nn.Identity())

        self.conv_layers = nn.ModuleList(conv_layers)
        self.relu_layers = nn.ModuleList(relu_layers)
        self.residual_convs = nn.ModuleList(residual_convs)
        self.conv_dropout = nn.Dropout(self.dropout_rate)

        # ---- Transformer branch (stats stream only) ----
        self.input_proj = nn.Conv1d(self.in_channels, self.embed_dim, kernel_size=1)  # kept for parity
        self.stats_proj = nn.Conv1d(self.stats_in_channels, self.embed_dim, kernel_size=1)

        self.register_buffer("pos_encoding", self._generate_pos_encoding(self.embed_dim, int(args["max_seq_len"])))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=self.embed_dim,
            nhead=int(args["transformer_heads"]),
            dim_feedforward=int(args["transformer_dim_feedforward"]),
            dropout=self.dropout_rate,
            batch_first=False,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=int(args["transformer_layers"]))

        self.query = nn.Parameter(torch.randn(1, 1, self.embed_dim))
        self.mha = nn.MultiheadAttention(self.embed_dim, int(args["transformer_heads"]), batch_first=False)

        # ---- Fusion heads ----
        local_dim = self.in_channels * int(args["conv_channels"][-1])
        fusion_dim = local_dim + self.embed_dim

        self.local_head = nn.Sequential(
            nn.Linear(local_dim, 64), nn.ReLU(), nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )
        self.attn_head = nn.Sequential(
            nn.Linear(self.embed_dim, 64), nn.ReLU(), nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )
        self.fused_head = nn.Sequential(
            nn.Linear(fusion_dim, 64), nn.ReLU(), nn.Dropout(self.dropout_rate),
            nn.Linear(64, self.n_classes),
        )

        self.w = nn.Parameter(torch.ones(3))
        self.temporal_pool = nn.Conv1d(
            in_channels=int(args["conv_channels"][-1]),
            out_channels=int(args["conv_channels"][-1]),
            kernel_size=int(input_length),
            bias=False,
        )

    def _generate_pos_encoding(self, d_model, max_len):
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        return pe.unsqueeze(1)  # (max_len, 1, d_model)

    def forward(self, x, x_stats):
        """
        x:       (B, C_cnn,   T)
        x_stats: (B, C_stats, T_stats)
        """
        B, C, T = x.size()
        B_s, C_s, Tstats = x_stats.size()

        assert C == self.in_channels, f"Expected x with {self.in_channels} channels, got {C}"
        assert C_s == self.stats_in_channels, f"Expected x_stats with {self.stats_in_channels} channels, got {C_s}"
        assert B_s == B, "Batch size mismatch between x and x_stats"

        # ---- per-channel conv branch ----
        local_feats = []
        for ch in range(C):
            xi = x[:, ch:ch + 1, :]  # (B, 1, T)
            out = xi
            for conv, relu, res_conv in zip(self.conv_layers, self.relu_layers, self.residual_convs):
                res = res_conv(out)
                out = relu(conv(out) + res)

            pooled = self.temporal_pool(out).squeeze(-1)  # (B, last_conv_ch)
            local_feats.append(pooled)

        local_feat = torch.cat(local_feats, dim=1)  # (B, C * last_conv_ch)
        local_feat = self.conv_dropout(local_feat)

        # ---- stats -> transformer ----
        stats_proj = self.stats_proj(x_stats).permute(2, 0, 1)  # (Tstats, B, D)
        pe = self.pos_encoding[:Tstats].to(x.device)
        attn_in = stats_proj + pe
        attn_out = self.transformer(attn_in)

        # ---- attention pooling ----
        q = self.query.expand(-1, B, -1)  # (1, B, D)
        attn_pooled, _ = self.mha(q, attn_out, attn_out)
        attn_pooled = attn_pooled.squeeze(0)  # (B, D)

        # ---- heads & fusion ----
        fusion = torch.cat([local_feat, attn_pooled], dim=1)
        pred_local = self.local_head(local_feat)
        pred_attn  = self.attn_head(attn_pooled)
        pred_fused = self.fused_head(fusion)

        w = torch.softmax(self.w, dim=0)
        return w[0] * pred_local + w[1] * pred_attn + w[2] * pred_fused


class GestureClassifier(pl.LightningModule):
    def __init__(
        self,
        input_length,
        n_classes,
        in_channels,
        stats_in_channels: int | None = None,
        class_weights: torch.Tensor | None = None,
        learning_rate: float = 5e-4,
        weight_decay: float = 1e-3,
        gamma: float = 0.99,
        **model_kwargs
    ):
        super().__init__()
        self.save_hyperparameters()

        self.learning_rate = float(learning_rate)
        self.weight_decay = float(weight_decay)
        self.gamma = float(gamma)

        if stats_in_channels is None:
            stats_in_channels = in_channels

        self.model = TimeSeriesCNNTransformer(
            input_length=input_length,
            n_classes=n_classes,
            in_channels=in_channels,
            stats_in_channels=stats_in_channels,
            **model_kwargs,
        )

        if class_weights is not None:
            self.register_buffer("ce_weights", class_weights.float())
            self.criterion = nn.CrossEntropyLoss(weight=self.ce_weights)
        else:
            self.ce_weights = None
            self.criterion = nn.CrossEntropyLoss()

        task = dict(task="multiclass", num_classes=int(n_classes))

        def coll():
            return MetricCollection({
                "acc_overall": Accuracy(average="micro", **task),
                "acc_macro":   Accuracy(average="macro", **task),
                "precision":   Precision(average="macro", **task, zero_division=0),
                "recall":      Recall(average="macro", **task, zero_division=0),
                "f1":          F1Score(average="macro", **task, zero_division=0),
                "f1_weighted": F1Score(average="weighted", **task, zero_division=0),
            })

        self.train_metrics = coll()
        self.val_metrics   = coll()
        self.test_metrics  = coll()

    def forward(self, x, x_stats):
        return self.model(x, x_stats)

    def _step(self, batch):
        x, x_stats, y = batch[0], batch[1], batch[2]
        x = x.permute(0, 2, 1)             # (B, C_cnn, T)
        x_stats = x_stats.permute(0, 2, 1) # (B, C_stats, T_stats)
        logits = self(x, x_stats)
        loss = self.criterion(logits, y)
        preds = torch.argmax(logits, dim=1)
        return loss, preds, y

    def training_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.train_metrics.update(preds, y)
        self.log("train/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)

        opt = self.optimizers()
        if opt:
            self.log("lr", float(opt.param_groups[0]["lr"]), on_step=False, on_epoch=True, prog_bar=True, sync_dist=True)
        return loss

    def validation_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.val_metrics.update(preds, y)
        self.log("val/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)

    def test_step(self, batch, _):
        loss, preds, y = self._step(batch)
        self.test_metrics.update(preds, y)
        self.log("test/loss", loss, on_step=False, on_epoch=True, prog_bar=False, sync_dist=True)

    def on_train_epoch_end(self):
        m = self.train_metrics.compute()
        for k, v in m.items():
            self.log(f"train/{k}", v, prog_bar=(k in ["acc_overall", "f1", "f1_weighted"]), sync_dist=True)
        self.train_metrics.reset()
        w = self.model.w.detach().cpu()
        self.log_dict({"epoch/w_0": w[0].item(), "epoch/w_1": w[1].item(), "epoch/w_2": w[2].item()}, sync_dist=True)

    def on_validation_epoch_end(self):
        m = self.val_metrics.compute()
        for k, v in m.items():
            self.log(f"val/{k}", v, prog_bar=(k in ["acc_overall", "f1", "f1_weighted"]), sync_dist=True)
        self.val_metrics.reset()

    def on_test_epoch_end(self):
        m = self.test_metrics.compute()
        for k, v in m.items():
            self.log(f"test/{k}", v, prog_bar=(k in ["acc_overall", "f1", "f1_weighted"]), sync_dist=True)
        self.test_metrics.reset()

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=self.weight_decay)
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=self.gamma)
        return {
            "optimizer": optimizer,
            "lr_scheduler": {
                "scheduler": scheduler,
                "interval": "epoch",
                "frequency": 1,
                "name": "exp_lr",
            }
        }


# ============================================================
# 4) INFER SHAPES / CLASSES / CLASS WEIGHTS
# ============================================================
# Peek one sample to infer shapes
sample_x, sample_x_stats, sample_y = dm_transformer.train_dataset[0]
seq_len = int(sample_x.shape[0])
in_channels = int(sample_x.shape[1])
stats_in_channels = int(sample_x_stats.shape[1])

# Determine n_classes and label_map
label_map = getattr(getattr(dm_transformer, "train_dataset", None), "label_map", None)
if not isinstance(label_map, dict):
    base = getattr(dm_transformer.train_dataset, "base", dm_transformer.train_dataset)
    label_map = getattr(base, "label_map", {})

if isinstance(label_map, dict) and len(label_map) > 0:
    n_classes_local = int(max(int(v) for v in label_map.values()) + 1)
else:
    # fallback: inspect one batch labels
    b = next(iter(dm_transformer.train_dataloader()))
    y_tmp = b[2]
    n_classes_local = int(y_tmp.max().item()) + 1

def build_class_labels(dm, n_classes_local: int):
    td = dm.train_dataset
    base = getattr(td, "base", td)
    lm = getattr(base, "label_map", getattr(td, "label_map", None))

    labels = [None] * int(n_classes_local)
    neg_id = None
    if "SELECTED_CLASSES" in globals() and "ADD_NEGATIVE_CLASS" in globals() and globals().get("ADD_NEGATIVE_CLASS", False):
        try:
            neg_id = len(globals()["SELECTED_CLASSES"])
        except Exception:
            neg_id = None

    if isinstance(lm, dict):
        for name, cid in lm.items():
            try:
                cid = int(cid)
            except Exception:
                continue
            if 0 <= cid < n_classes_local and labels[cid] is None:
                labels[cid] = "Negative" if (neg_id is not None and cid == neg_id) else str(name)

    for i in range(int(n_classes_local)):
        if labels[i] is None:
            labels[i] = "Negative" if (neg_id is not None and i == neg_id) else f"Class {i}"
    return labels

class_labels = build_class_labels(dm_transformer, n_classes_local)

print(f"Sequence length: {seq_len}, CNN channels: {in_channels}, Stats channels: {stats_in_channels}, Classes: {n_classes_local}")
print("Class labels:", class_labels)

# Class weights (based on base clips via .samples if available)
train_labels = []
td = dm_transformer.train_dataset
base_td = getattr(td, "base", td)
base_samples = getattr(base_td, "samples", None)
base_label_map = getattr(base_td, "label_map", label_map)

if isinstance(base_samples, list) and isinstance(base_label_map, dict) and len(base_label_map) > 0:
    for s in base_samples:
        clean = re.sub(r"\(\d+\)$", "", s["label"])
        if clean in base_label_map:
            train_labels.append(int(base_label_map[clean]))
else:
    # fallback: iterate a few batches
    for i, b in enumerate(dm_transformer.train_dataloader()):
        yb = b[2].detach().cpu().numpy().tolist()
        train_labels.extend([int(v) for v in yb])
        if i >= 50:
            break

counts = Counter(train_labels)
freq = torch.tensor([counts.get(i, 0) for i in range(int(n_classes_local))], dtype=torch.float)
class_weights = (freq.sum() / (int(n_classes_local) * freq.clamp(min=1.0))).float()
print("Using class weights:", [round(float(x), 4) for x in class_weights.tolist()])

# Save run context/config
save_json("run_context.json", {
    "run_id": RUN_ID,
    "outputs_root": OUTPUTS_ROOT,
    "transformer_dir": TRANSFORMER_DIR,
    "device": str(device),
})
save_json("config.json", {
    "SEED": int(SEED),
    "LR": float(LR),
    "WEIGHT_DECAY": float(WEIGHT_DECAY),
    "MAX_EPOCHS": int(MAX_EPOCHS),
    "PATIENCE": int(PATIENCE),
    "DECAY_TARGET": float(DECAY_TARGET),
    "GAMMA": float(GAMMA),
    "seq_len": int(seq_len),
    "in_channels": int(in_channels),
    "stats_in_channels": int(stats_in_channels),
    "n_classes": int(n_classes_local),
    "CLIP_DEFAULT_K": int(CLIP_DEFAULT_K),
    "CLIP_K_MAP": CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else None,
})
save_json("class_labels.json", {"class_labels": class_labels, "n_classes": int(n_classes_local)})

# Build model
standard_model = GestureClassifier(
    input_length=int(seq_len),
    n_classes=int(n_classes_local),
    in_channels=int(in_channels),
    stats_in_channels=int(stats_in_channels),
    class_weights=class_weights,
    learning_rate=float(LR),
    weight_decay=float(WEIGHT_DECAY),
    gamma=float(GAMMA),
)

n_total = sum(p.numel() for p in standard_model.parameters())
n_trainable = sum(p.numel() for p in standard_model.parameters() if p.requires_grad)
save_text("parameter_stats.txt", f"Total params: {n_total}\nTrainable params: {n_trainable}\n")
print(f"Model params: {n_total:,} (trainable {n_trainable:,})")


# ============================================================
# 5) TRAINER (CSVLogger -> TRANSFORMER_DIR)
# ============================================================
checkpoint_callback = ModelCheckpoint(
    monitor="val/f1",
    mode="max",
    save_top_k=1,
    dirpath=TRANSFORMER_DIR,
    filename="gesture-classifier-best-{epoch:02d}",
)
lr_monitor = LearningRateMonitor(logging_interval="epoch")
early_stop = EarlyStopping(monitor="val/f1", mode="max", patience=int(PATIENCE), min_delta=0.0)

csv_logger = CSVLogger(save_dir=TRANSFORMER_DIR, name="pl_logs")

use_cuda = torch.cuda.is_available() or torch.backends.mps.is_available()
precision = "16-mixed" if torch.cuda.is_available() else 32

trainer = pl.Trainer(
    max_epochs=int(MAX_EPOCHS),
    accelerator="auto",
    devices=1 if use_cuda else "auto",
    callbacks=[checkpoint_callback, lr_monitor, early_stop],
    logger=[csv_logger],
    log_every_n_steps=min(10, max(1, len(dm_transformer.train_dataloader()))),
    gradient_clip_val=1.0,
    precision=precision,
    deterministic=False,
)

print(f"Trainer ready -> epochs={int(MAX_EPOCHS)}, precision={precision}")


# ============================================================
# 6) TRAIN + TEST (Lightning)
# ============================================================
print("Training...")
trainer.fit(standard_model, datamodule=dm_transformer)

print("Testing on best checkpoint...")
trainer.test(ckpt_path="best", datamodule=dm_transformer)

best_ckpt = checkpoint_callback.best_model_path
save_text("best_checkpoint_path.txt", best_ckpt if best_ckpt else "NONE")
print("Best checkpoint:", best_ckpt if best_ckpt else "NONE")


# ============================================================
# 7) LOAD BEST MODEL FOR MANUAL EVAL
# ============================================================
if best_ckpt and os.path.isfile(best_ckpt):
    print("Loading best checkpoint for manual eval:", best_ckpt)
    best_model = type(standard_model).load_from_checkpoint(best_ckpt)
else:
    print("No best checkpoint found; using current model weights for manual eval.")
    best_model = standard_model


# ============================================================
# 8) COMPREHENSIVE METRICS HELPERS (EXTRACTED FROM NORMWEAR)
# ============================================================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out

def per_class_f1(y_true, y_pred, n_classes_local: int, zero_division=0):
    """Compute per-class precision, recall, F1, and support"""
    labels = np.arange(int(n_classes_local))
    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=zero_division
    )
    return {
        "precision": [float(v) for v in p],
        "recall":    [float(v) for v in r],
        "f1":        [float(v) for v in f1],
        "support":   [int(v) for v in s],
    }


# ============================================================
# 9) STANDARDIZED CONFUSION MATRIX PLOTTING (EXTRACTED FROM NORMWEAR)
# ============================================================
def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str,
    annot: bool = True,
    annot_fmt: str = ".2f",
    dpi: int = 200,
):
    """Plot row-normalized confusion matrix (0-1 scale)"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_norm01.npy"), cm_norm.astype(np.float32))
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)

    return cm_norm, cm_counts

def plot_conf_mat_counts(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str = "Blues",
    annot: bool = True,
    dpi: int = 200,
    max_annot: int = 9999,
):
    """Plot count-based confusion matrix"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_counts,
        annot=annot,
        fmt="d",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
        vmin=0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    if annot and max_annot is not None:
        for t in ax.texts:
            try:
                v = int(t.get_text())
                if v > int(max_annot):
                    t.set_text(f">{max_annot}")
            except Exception:
                pass

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)
    return cm_counts


# ============================================================
# 10) WINDOW-LEVEL EVAL WITH COMPREHENSIVE METRICS
# ============================================================
@torch.no_grad()
def eval_split_window_comprehensive(loader, model, device, criterion=None):
    """Evaluate with comprehensive metrics matching NormWear structure"""
    model.eval().to(device)
    total_loss = 0.0
    preds_all, targs_all = [], []
    n_seen = 0

    for batch in loader:
        if isinstance(batch, (list, tuple)) and len(batch) >= 3:
            x, x_stats, y = batch[0], batch[1], batch[2]
        else:
            raise ValueError("Unexpected batch format; expected (x, x_stats, y, ...).")

        x = x.permute(0, 2, 1).to(device)
        x_stats = x_stats.permute(0, 2, 1).to(device)
        y = y.to(device)

        logits = model(x, x_stats)
        pred = torch.argmax(logits, dim=1)

        if criterion is not None:
            loss = criterion(logits, y)
            total_loss += float(loss.item()) * int(y.size(0))

        preds_all.append(pred.detach().cpu())
        targs_all.append(y.detach().cpu())
        n_seen += int(y.size(0))

    y_pred = torch.cat(preds_all).numpy() if preds_all else np.array([], dtype=int)
    y_true = torch.cat(targs_all).numpy() if targs_all else np.array([], dtype=int)

    loss_avg = (total_loss / max(1, n_seen)) if criterion is not None else float("nan")
    
    if y_true.size:
        acc = float((y_pred == y_true).mean())
        prf = prf_all_averages(y_true, y_pred, zero_division=0)
        per_cls = per_class_f1(y_true, y_pred, n_classes_local=int(n_classes_local), zero_division=0)
    else:
        acc = float("nan")
        prf = {
            "macro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "micro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "weighted": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
        }
        per_cls = {"precision": [], "recall": [], "f1": [], "support": []}
    
    metrics = {
        "loss": float(loss_avg),
        "acc": float(acc),
        "prf": prf,
        "per_class": per_cls,
    }
    
    return metrics, y_true, y_pred


print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION WITH COMPREHENSIVE METRICS")
print("=" * 60)

val_loader = dm_transformer.val_dataloader()
test_loader = dm_transformer.test_dataloader()

val_metrics, y_val_true, y_val_pred = eval_split_window_comprehensive(
    val_loader, best_model, device, criterion=best_model.criterion
)
test_metrics, y_test_true, y_test_pred = eval_split_window_comprehensive(
    test_loader, best_model, device, criterion=best_model.criterion
)

# Save comprehensive metrics
window_summary = {
    "val": val_metrics,
    "test": test_metrics,
    "best_ckpt": str(best_ckpt) if best_ckpt else None,
}
save_json("metrics_window.json", window_summary)

# Detailed summary text
save_text(
    "metrics_summary_window.txt",
    "WINDOW-LEVEL METRICS (COMPREHENSIVE)\n"
    f"VAL  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f}\n"
    f"VAL  macro    P/R/F1={val_metrics['prf']['macro']['precision']:.4f}/"
    f"{val_metrics['prf']['macro']['recall']:.4f}/{val_metrics['prf']['macro']['f1']:.4f}\n"
    f"VAL  micro    P/R/F1={val_metrics['prf']['micro']['precision']:.4f}/"
    f"{val_metrics['prf']['micro']['recall']:.4f}/{val_metrics['prf']['micro']['f1']:.4f}\n"
    f"VAL  weighted P/R/F1={val_metrics['prf']['weighted']['precision']:.4f}/"
    f"{val_metrics['prf']['weighted']['recall']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f}\n"
    f"TEST macro    P/R/F1={test_metrics['prf']['macro']['precision']:.4f}/"
    f"{test_metrics['prf']['macro']['recall']:.4f}/{test_metrics['prf']['macro']['f1']:.4f}\n"
    f"TEST micro    P/R/F1={test_metrics['prf']['micro']['precision']:.4f}/"
    f"{test_metrics['prf']['micro']['recall']:.4f}/{test_metrics['prf']['micro']['f1']:.4f}\n"
    f"TEST weighted P/R/F1={test_metrics['prf']['weighted']['precision']:.4f}/"
    f"{test_metrics['prf']['weighted']['recall']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
)

# Save support counts
save_json(
    "val_test_support.json",
    {
        "val_support": np.bincount(y_val_true, minlength=int(n_classes_local)).tolist(),
        "test_support": np.bincount(y_test_true, minlength=int(n_classes_local)).tolist(),
    },
)


# ============================================================
# 11) CONFUSION MATRICES - WINDOW LEVEL (BLUES COLORMAP)
# ============================================================
print("\n" + "=" * 60)
print("GENERATING CONFUSION MATRICES (WINDOW-LEVEL)")
print("=" * 60)

# Validation - normalized
_ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_val_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

# Test - normalized
_ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_test_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

# Validation - counts
_ = plot_conf_mat_counts(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, counts)",
    class_names=class_labels,
    out_png="cm_val_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Test - counts
_ = plot_conf_mat_counts(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, counts)",
    class_names=class_labels,
    out_png="cm_test_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Classification reports
labels_idx = np.arange(int(n_classes_local))
val_report = classification_report(
    y_val_true, y_val_pred, labels=labels_idx, target_names=class_labels, digits=3, zero_division=0
)
test_report = classification_report(
    y_test_true, y_test_pred, labels=labels_idx, target_names=class_labels, digits=3, zero_division=0
)
save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)


# ============================================================
# 12) CLIP-LEVEL AGGREGATION (TEST) WITH COMPREHENSIVE METRICS
# ============================================================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION WITH COMPREHENSIVE METRICS")
print("=" * 60)

def early_stop_classwise(seq, k_map=None, default_k=3):
    """Early-stop aggregation with class-wise k values"""
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        v = int(v)
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = int(k_map.get(v, default_k))
        if run_len >= k_c:
            return v
    return None

k_map = CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else {}

test_ds = dm_transformer.test_dataset
base_ds = getattr(test_ds, "base", test_ds)
n_clips = len(base_ds)

if hasattr(test_ds, "_index"):
    clip_ids_for_windows = [pair[0] for pair in test_ds._index]
else:
    clip_ids_for_windows = list(range(len(test_ds)))

if len(clip_ids_for_windows) != len(y_test_true):
    save_text(
        "clip_eval_note.txt",
        f"Skipping clip-level: clip_ids_for_windows={len(clip_ids_for_windows)} != n_windows={len(y_test_true)}"
    )
else:
    clip_to_preds = defaultdict(list)
    clip_to_targs = defaultdict(list)
    for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
        clip_to_preds[int(cid)].append(int(p))
        clip_to_targs[int(cid)].append(int(t))

    # Clip targets
    clip_targets = []
    for cid in range(int(n_clips)):
        if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
            clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
            clip_targets.append(int(base_ds.label_map[clean]))
        else:
            tgts = clip_to_targs.get(cid, [])
            clip_targets.append(int(max(set(tgts), key=tgts.count)) if tgts else 0)
    clip_targets = np.array(clip_targets, dtype=int)

    # Clip predictions with early-stop
    clip_preds = []
    missed_early = 0
    for cid in range(int(n_clips)):
        seq = clip_to_preds.get(cid, [])
        chosen = early_stop_classwise(seq, k_map=k_map, default_k=int(CLIP_DEFAULT_K))
        if chosen is None:
            missed_early += 1
            chosen = int(max(set(seq), key=seq.count) if seq else clip_targets[cid])
        clip_preds.append(int(chosen))
    clip_preds = np.array(clip_preds, dtype=int)

    # Compute comprehensive clip metrics
    clip_acc = float((clip_preds == clip_targets).mean()) if clip_targets.size else float("nan")
    clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0) if clip_targets.size else None
    clip_per_cls = per_class_f1(clip_targets, clip_preds, n_classes_local=int(n_classes_local), zero_division=0) if clip_targets.size else None

    clip_metrics = {
        "acc": float(clip_acc),
        "prf": clip_prf,
        "per_class": clip_per_cls,
        "early_stop": {
            "default_k": int(CLIP_DEFAULT_K),
            "k_map_size": int(len(k_map)),
            "missed_early": int(missed_early),
            "n_clips": int(n_clips),
        },
    }
    save_json("metrics_clip.json", clip_metrics)

    # Detailed summary
    save_text(
        "metrics_summary_clip.txt",
        "CLIP-LEVEL METRICS (COMPREHENSIVE)\n"
        f"TEST acc={clip_acc:.4f}\n"
        f"TEST macro    P/R/F1={clip_prf['macro']['precision']:.4f}/"
        f"{clip_prf['macro']['recall']:.4f}/{clip_prf['macro']['f1']:.4f}\n"
        f"TEST micro    P/R/F1={clip_prf['micro']['precision']:.4f}/"
        f"{clip_prf['micro']['recall']:.4f}/{clip_prf['micro']['f1']:.4f}\n"
        f"TEST weighted P/R/F1={clip_prf['weighted']['precision']:.4f}/"
        f"{clip_prf['weighted']['recall']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
        f"Early-stop default_k={int(CLIP_DEFAULT_K)} k_map_size={len(k_map)} majority_fallback={missed_early}/{n_clips}\n"
    )

    # Confusion matrices - CLIP LEVEL (GREENS COLORMAP)
    _ = plot_conf_mat_norm01(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, normalized 0-1)",
        class_names=class_labels,
        out_png="cm_test_clip_norm01.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
        dpi=200,
    )

    _ = plot_conf_mat_counts(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, counts)",
        class_names=class_labels,
        out_png="cm_test_clip_counts.png",
        cmap="Greens",
        annot=True,
        dpi=200,
    )

    # Classification report
    clip_report = classification_report(
        clip_targets, clip_preds, labels=labels_idx, target_names=class_labels, digits=3, zero_division=0
    )
    save_text("report_test_clip.txt", clip_report)


# ============================================================
# 13) CURVES FROM metrics.csv (loss/acc/f1/lr)
# ============================================================
def _find_metrics_csv(trainer_obj: pl.Trainer) -> str:
    if hasattr(trainer_obj, "loggers") and trainer_obj.loggers:
        for lg in trainer_obj.loggers:
            if hasattr(lg, "log_dir"):
                cand = os.path.join(lg.log_dir, "metrics.csv")
                if os.path.isfile(cand):
                    return cand
    for root, _, files in os.walk(TRANSFORMER_DIR):
        if "metrics.csv" in files:
            return os.path.join(root, "metrics.csv")
    raise FileNotFoundError("metrics.csv not found under Transformer run directory.")

def _pick_col(df, must_have=(), any_of=(), avoid=("step", "time", "wall", "grad", "ema")):
    cols = []
    for c in df.columns:
        if c == "epoch":
            continue
        c_low = c.lower()
        if any(a in c_low for a in avoid):
            continue
        if must_have and not all(m in c_low for m in must_have):
            continue
        if any_of and not any(a in c_low for a in any_of):
            continue
        cols.append(c)
    cols = sorted(cols, key=lambda x: (len(x), x))
    return cols[0] if cols else None

def _last_series(grouped, col):
    if (col is None) or (col not in grouped.obj.columns):
        return None
    return grouped[col].last().astype(float).values

metrics_csv = _find_metrics_csv(trainer)
save_text("metrics_csv_path.txt", metrics_csv)

dfm = pd.read_csv(metrics_csv)

# remove sanity-check rows if present
for col in ("sanity_check", "stage"):
    if col in dfm.columns:
        if col == "sanity_check":
            dfm = dfm[dfm[col] == 0]
        else:
            dfm = dfm[~dfm[col].astype(str).str.contains("sanity", case=False, na=False)]

if "epoch" not in dfm.columns:
    raise ValueError("metrics.csv missing 'epoch' column.")

g = dfm.groupby("epoch", sort=True)
epochs = g.size().index.values

# pick columns
col_train_loss = _pick_col(dfm, must_have=("train",), any_of=("loss",))
col_val_loss   = _pick_col(dfm, must_have=("val",),   any_of=("loss",))

col_train_acc = _pick_col(dfm, must_have=("train",), any_of=("acc_overall", "acc",))
col_val_acc   = _pick_col(dfm, must_have=("val",),   any_of=("acc_overall", "acc",))

col_train_f1 = _pick_col(dfm, must_have=("train",), any_of=("f1",))
col_val_f1   = _pick_col(dfm, must_have=("val",),   any_of=("f1",))

# if weighted was chosen, attempt to prefer non-weighted
def _prefer_non_weighted(df, col_candidate, prefix):
    if col_candidate is None:
        return None
    if "weighted" in col_candidate.lower():
        alt = _pick_col(df, must_have=(prefix,), any_of=("f1",), avoid=("weighted", "step", "time", "wall", "grad", "ema"))
        return alt or col_candidate
    return col_candidate

col_train_f1 = _prefer_non_weighted(dfm, col_train_f1, "train")
col_val_f1   = _prefer_non_weighted(dfm, col_val_f1, "val")

col_lr = _pick_col(dfm, must_have=(), any_of=("lr",), avoid=("step", "time", "wall", "grad", "ema", "train", "val"))

train_loss_s = _last_series(g, col_train_loss)
val_loss_s   = _last_series(g, col_val_loss)
train_acc_s  = _last_series(g, col_train_acc)
val_acc_s    = _last_series(g, col_val_acc)
train_f1_s   = _last_series(g, col_train_f1)
val_f1_s     = _last_series(g, col_val_f1)
lr_s         = _last_series(g, col_lr)

save_json("picked_plot_columns.json", {
    "metrics_csv": metrics_csv,
    "train_loss_col": col_train_loss,
    "val_loss_col": col_val_loss,
    "train_acc_col": col_train_acc,
    "val_acc_col": col_val_acc,
    "train_f1_col": col_train_f1,
    "val_f1_col": col_val_f1,
    "lr_col": col_lr,
})

# Loss curve
plt.figure(figsize=(9, 5))
plotted_any = False
if train_loss_s is not None:
    plt.plot(epochs, train_loss_s, label="Train loss"); plotted_any = True
if val_loss_s is not None:
    plt.plot(epochs, val_loss_s, label="Val loss"); plotted_any = True
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.title("Train vs Validation Loss")
plt.grid(True, alpha=0.3)
if plotted_any: plt.legend()
save_fig("curve_loss_train_val.png", dpi=300)

# Accuracy curve
plt.figure(figsize=(9, 5))
plotted_any = False
if train_acc_s is not None:
    plt.plot(epochs, train_acc_s, label="Train accuracy"); plotted_any = True
if val_acc_s is not None:
    plt.plot(epochs, val_acc_s, label="Val accuracy"); plotted_any = True
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title("Train vs Validation Accuracy")
plt.grid(True, alpha=0.3)
if plotted_any: plt.legend()
save_fig("curve_acc_train_val.png", dpi=300)

# F1 curve
plt.figure(figsize=(9, 5))
plotted_any = False
if train_f1_s is not None:
    plt.plot(epochs, train_f1_s, label="Train F1"); plotted_any = True
if val_f1_s is not None:
    plt.plot(epochs, val_f1_s, label="Val F1"); plotted_any = True
plt.xlabel("Epoch"); plt.ylabel("F1"); plt.title("Train vs Validation F1")
plt.grid(True, alpha=0.3)
if plotted_any: plt.legend()
save_fig("curve_f1_train_val.png", dpi=300)

# LR curve
if lr_s is not None:
    plt.figure(figsize=(9, 5))
    plt.plot(epochs, lr_s, label="Learning rate")
    plt.xlabel("Epoch"); plt.ylabel("LR"); plt.title("Learning Rate Schedule (ExponentialLR)")
    plt.grid(True, alpha=0.3)
    plt.legend()
    save_fig("curve_lr.png", dpi=300)
else:
    save_text("curve_lr_note.txt", "No LR column found in metrics.csv; LR plot not created.")


# ============================================================
# 14) FINAL SUMMARY
# ============================================================
final_summary = (
    "FINAL SUMMARY (COMPREHENSIVE METRICS)\n"
    f"Params total: {n_total}\n"
    f"Best checkpoint: {best_ckpt if best_ckpt else 'NONE'}\n"
    f"VAL window:  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={val_metrics['prf']['macro']['f1']:.4f}/"
    f"{val_metrics['prf']['micro']['f1']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={test_metrics['prf']['macro']['f1']:.4f}/"
    f"{test_metrics['prf']['micro']['f1']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
    + (
        f"TEST clip:   acc={clip_metrics['acc']:.4f} "
        f"F1(macro/micro/weighted)={clip_metrics['prf']['macro']['f1']:.4f}/"
        f"{clip_metrics['prf']['micro']['f1']:.4f}/{clip_metrics['prf']['weighted']['f1']:.4f}\n"
        if 'clip_metrics' in locals() and clip_metrics is not None and clip_metrics.get("prf") is not None else
        "TEST clip:   SKIPPED\n"
    ) +
    f"Artifacts:   {TRANSFORMER_DIR}\n"
)
print(final_summary)
save_text("final_summary.txt", final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", TRANSFORMER_DIR)

In [ ]:
# ============================================================
# INFERENCE TIMING BENCHMARK
# ============================================================
import time
import torch
import numpy as np

print("\n" + "=" * 60)
print("INFERENCE TIMING BENCHMARK")
print("=" * 60)

@torch.no_grad()
def benchmark_inference_time(model, dataloader, device, n_warmup=10, n_iterations=100):
    """
    Benchmark inference time per window/sample.
    
    Args:
        model: The trained model
        dataloader: DataLoader to get sample batches from
        device: torch device
        n_warmup: Number of warmup iterations (not timed)
        n_iterations: Number of timed iterations
    
    Returns:
        dict with timing statistics
    """
    model.eval().to(device)
    
    # Get a single batch for benchmarking
    batch = next(iter(dataloader))
    x, x_stats, y = batch[0], batch[1], batch[2]
    
    # Prepare single-sample input (batch size = 1)
    x_single = x[:1].permute(0, 2, 1).to(device)
    x_stats_single = x_stats[:1].permute(0, 2, 1).to(device)
    
    # Warmup (not timed) - important for GPU to reach steady state
    print(f"Warming up with {n_warmup} iterations...")
    for _ in range(n_warmup):
        _ = model(x_single, x_stats_single)
    
    # Synchronize if using CUDA
    if device.type == "cuda":
        torch.cuda.synchronize()
    
    # Timed iterations
    print(f"Running {n_iterations} timed iterations...")
    times_ms = []
    
    for _ in range(n_iterations):
        if device.type == "cuda":
            torch.cuda.synchronize()
        
        t_start = time.perf_counter()
        _ = model(x_single, x_stats_single)
        
        if device.type == "cuda":
            torch.cuda.synchronize()
        
        t_end = time.perf_counter()
        times_ms.append((t_end - t_start) * 1000)  # Convert to ms
    
    times_ms = np.array(times_ms)
    
    stats = {
        "n_warmup": n_warmup,
        "n_iterations": n_iterations,
        "device": str(device),
        "batch_size": 1,
        "input_shape_x": list(x_single.shape),
        "input_shape_x_stats": list(x_stats_single.shape),
        "time_per_window_ms": {
            "mean": float(np.mean(times_ms)),
            "std": float(np.std(times_ms)),
            "min": float(np.min(times_ms)),
            "max": float(np.max(times_ms)),
            "median": float(np.median(times_ms)),
            "p95": float(np.percentile(times_ms, 95)),
            "p99": float(np.percentile(times_ms, 99)),
        },
        "throughput_windows_per_sec": float(1000.0 / np.mean(times_ms)),
    }
    
    return stats, times_ms


# Run benchmark
timing_stats, timing_raw = benchmark_inference_time(
    model=best_model,
    dataloader=dm_transformer.test_dataloader(),
    device=device,
    n_warmup=10,
    n_iterations=100,
)

# Print results
print(f"\nInference Timing Results:")
print(f"  Device: {timing_stats['device']}")
print(f"  Input shape (x):       {timing_stats['input_shape_x']}")
print(f"  Input shape (x_stats): {timing_stats['input_shape_x_stats']}")
print(f"  Time per window:")
print(f"    Mean:   {timing_stats['time_per_window_ms']['mean']:.3f} ms")
print(f"    Std:    {timing_stats['time_per_window_ms']['std']:.3f} ms")
print(f"    Min:    {timing_stats['time_per_window_ms']['min']:.3f} ms")
print(f"    Max:    {timing_stats['time_per_window_ms']['max']:.3f} ms")
print(f"    Median: {timing_stats['time_per_window_ms']['median']:.3f} ms")
print(f"    P95:    {timing_stats['time_per_window_ms']['p95']:.3f} ms")
print(f"    P99:    {timing_stats['time_per_window_ms']['p99']:.3f} ms")
print(f"  Throughput: {timing_stats['throughput_windows_per_sec']:.1f} windows/sec")

# Save to TRANSFORMER_DIR
save_json("inference_timing.json", timing_stats)
save_npy("inference_timing_raw_ms.npy", timing_raw)

print(f"\nTiming artifacts saved to: {TRANSFORMER_DIR}")

## Apple Model

In [ ]:
# =============================================================================
# CONFIG
# =============================================================================
SEED = 42
SAMPLING_RATE = 100.0

BATCH_SIZE = 64
EPOCHS = 100
PATIENCE = 8

LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.05

# Model architecture params
N_IMU_AXES = 6
N_FILTERBANK = 4
SEQ_LEN = 100

MODEL_DROPOUT = 0.0  # keep 0.0 if you want the same as the diagram; neutral name

# Clip-level aggregation config
CLIP_K_MAP = None     # example: {0: 3, 1: 3, 2: 3}
CLIP_DEFAULT_K = 3

# =============================================================================
# 1) Run directory under OUTPUTS/apple/run_<timestamp>
# =============================================================================
OUTPUTS_ROOT = OUTPUTS
APPLE_ROOT = os.path.join(OUTPUTS_ROOT, "apple")
Path(APPLE_ROOT).mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
APPLE_DIR = os.path.join(APPLE_ROOT, f"run_{RUN_ID}")
Path(APPLE_DIR).mkdir(parents=True, exist_ok=True)

def apath(filename: str) -> str:
    return os.path.join(APPLE_DIR, filename)

def save_text(filename: str, text: str) -> str:
    path = apath(filename)
    with open(path, "w", encoding="utf-8") as f:
        f.write(text)
    return path

def save_json(filename: str, obj) -> str:
    path = apath(filename)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    return path

def save_csv(filename: str, df: pd.DataFrame, index: bool = False) -> str:
    path = apath(filename)
    df.to_csv(path, index=index)
    return path

def save_npy(filename: str, arr: np.ndarray) -> str:
    path = apath(filename)
    np.save(path, arr)
    return path

def save_fig(filename: str, dpi: int = 300) -> str:
    path = apath(filename)
    plt.tight_layout()
    plt.savefig(path, dpi=dpi, bbox_inches="tight")
    plt.close()
    return path

print("Apple run directory:", APPLE_DIR)

save_json("run_context.json", {
    "run_id": RUN_ID,
    "outputs_root": OUTPUTS_ROOT,
    "apple_root": APPLE_ROOT,
    "apple_dir": APPLE_DIR,
})

save_json("config.json", {
    "SEED": SEED,
    "SAMPLING_RATE": SAMPLING_RATE,
    "BATCH_SIZE": BATCH_SIZE,
    "EPOCHS": EPOCHS,
    "PATIENCE": PATIENCE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "LABEL_SMOOTHING": LABEL_SMOOTHING,
    "N_IMU_AXES": N_IMU_AXES,
    "N_FILTERBANK": N_FILTERBANK,
    "SEQ_LEN": SEQ_LEN,
    "MODEL_DROPOUT": MODEL_DROPOUT,
    "CLIP_DEFAULT_K": CLIP_DEFAULT_K,
    "CLIP_K_MAP": CLIP_K_MAP,
    "SELECTED_CLASSES": SELECTED_CLASSES,
    "ADD_NEGATIVE_CLASS": bool(ADD_NEGATIVE_CLASS),
    "NEGATIVE_CLASS_IDS": list(NEGATIVE_CLASS_IDS) if NEGATIVE_CLASS_IDS is not None else None,
})

# =============================================================================
# RESET + SEEDS
# =============================================================================
for name in ["model", "optimizer", "scheduler", "history", "best_state"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================================================================
# MODEL DEFINITION - Filterbank IMU CNN
# =============================================================================
def _pad_1d(L_in: int, kernel: int, stride: int, dilation: int = 1):
    L_out = math.ceil(L_in / stride)
    pad_total = max((L_out - 1) * stride + (kernel - 1) * dilation + 1 - L_in, 0)
    pad_left = pad_total // 2
    pad_right = pad_total - pad_left
    return pad_left, pad_right, L_out

class Conv1d(nn.Module):
    def __init__(self, in_ch, out_ch, kernel_size, stride=1, dilation=1, groups=1, bias=False):
        super().__init__()
        self.kernel_size = int(kernel_size)
        self.stride = int(stride)
        self.dilation = int(dilation)
        self.conv = nn.Conv1d(
            in_ch, out_ch,
            kernel_size=self.kernel_size,
            stride=self.stride,
            padding=0,
            dilation=self.dilation,
            groups=groups,
            bias=bias,
        )

    def forward(self, x):
        T = x.size(-1)
        pl, pr, _ = _pad_1d(T, self.kernel_size, self.stride, self.dilation)
        if pl or pr:
            x = F.pad(x, (pl, pr))
        return self.conv(x)

class Detrend1d(nn.Module):
    def forward(self, x):
        B, C, T = x.shape
        t = torch.arange(T, device=x.device, dtype=x.dtype).view(1, 1, T)
        t_mean = t.mean(dim=-1, keepdim=True)
        x_mean = x.mean(dim=-1, keepdim=True)
        t_center = t - t_mean
        x_center = x - x_mean
        var_t = (t_center * t_center).sum(dim=-1, keepdim=True) + 1e-8
        cov_tx = (t_center * x_center).sum(dim=-1, keepdim=True)
        slope = cov_tx / var_t
        intercept = x_mean - slope * t_mean
        trend = slope * t + intercept
        return x - trend

class MBConv1d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=10, expand_ratio=1, stride=1, act=nn.SiLU):
        super().__init__()
        hidden_dim = in_channels * int(expand_ratio)
        layers = []
        if expand_ratio > 1:
            layers += [
                nn.Conv1d(in_channels, hidden_dim, kernel_size=1, bias=False),
                nn.BatchNorm1d(hidden_dim),
                act(inplace=True),
            ]
        else:
            hidden_dim = in_channels

        layers += [
            Conv1d(hidden_dim, hidden_dim, kernel_size=kernel_size, stride=stride,
                       groups=hidden_dim, bias=False),
            nn.BatchNorm1d(hidden_dim),
            act(inplace=True),
        ]
        layers += [
            nn.Conv1d(hidden_dim, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm1d(out_channels),
        ]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

class SeparableConv1d(nn.Module):
    def __init__(self, in_channels, out_channels=24, kernel_size=10, act=nn.SiLU):
        super().__init__()
        self.depthwise = nn.Conv1d(
            in_channels, in_channels,
            kernel_size=kernel_size,
            stride=1,
            padding=0,
            groups=in_channels,
            bias=False,
        )
        self.pointwise = nn.Conv1d(in_channels, out_channels, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(out_channels)
        self.act = act(inplace=True)

    def forward(self, x):
        x = self.depthwise(x)
        x = self.pointwise(x)
        x = self.bn(x)
        x = self.act(x)
        return x

class AxisBranch(nn.Module):
    def __init__(self, n_filterbank=4, act=nn.SiLU):
        super().__init__()
        self.detrend = Detrend1d()
        self.mbconv1 = MBConv1d(n_filterbank, 16, kernel_size=10, expand_ratio=1, stride=1, act=act)
        self.mbconv6 = MBConv1d(16, 24, kernel_size=10, expand_ratio=6, stride=2, act=act)

    def forward(self, x):
        x = self.detrend(x)
        x = self.mbconv1(x)
        x = self.mbconv6(x)
        return x

class AppleModel(nn.Module):
    def __init__(self, n_classes, n_axes=6, n_filterbank=4, seq_len=100, dropout=0.0, act=nn.SiLU):
        super().__init__()
        self.n_axes = int(n_axes)
        self.n_filterbank = int(n_filterbank)
        self.seq_len = int(seq_len)

        self.axis_branches = nn.ModuleList([
            AxisBranch(n_filterbank=self.n_filterbank, act=act) for _ in range(self.n_axes)
        ])

        self.separable_conv = SeparableConv1d(
            in_channels=self.n_axes * 24,
            out_channels=24,
            kernel_size=10,
            act=act,
        )
        self.maxpool = nn.MaxPool1d(kernel_size=8, stride=8)

        drop = nn.Dropout(dropout) if dropout and dropout > 0 else nn.Identity()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(120, 80),
            act(inplace=True),
            drop,
            nn.Linear(80, 40),
            act(inplace=True),
            drop,
            nn.Linear(40, 20),
            act(inplace=True),
            drop,
            nn.Linear(20, 10),
            act(inplace=True),
            nn.Linear(10, n_classes),
        )
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.trunc_normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def reorganize_filterbank_channels(self, x):
        B, T, C = x.shape
        expected = self.n_axes * self.n_filterbank
        assert C == expected, f"Expected {expected} channels, got {C}"
        x = x.permute(0, 2, 1).contiguous()              # [B,C,T]
        x = x.view(B, self.n_filterbank, self.n_axes, T) # [B,4,6,T]
        x = x.permute(0, 2, 1, 3).contiguous()           # [B,6,4,T]
        return x

    def forward(self, x):
        x = self.reorganize_filterbank_channels(x)   # [B,6,4,100]
        outs = [br(x[:, i, :, :]) for i, br in enumerate(self.axis_branches)]
        x = torch.cat(outs, dim=1)                   # [B,144,50]
        x = self.separable_conv(x)                   # [B,24,41]
        x = self.maxpool(x)                          # [B,24,5]
        return self.classifier(x)

    @torch.no_grad()
    def check_expected_shapes(self, x):
        assert x.dim() == 3 and x.shape[1] == 100 and x.shape[2] == 24, f"Expected [B,100,24], got {tuple(x.shape)}"
        xg = self.reorganize_filterbank_channels(x)  # [B,6,4,100]
        outs = []
        for i, br in enumerate(self.axis_branches):
            y = br(xg[:, i, :, :])
            assert y.shape[1] == 24 and y.shape[2] == 50, f"Axis {i}: expected [B,24,50], got {tuple(y.shape)}"
            outs.append(y)
        xc = torch.cat(outs, dim=1)
        assert xc.shape[1] == 144 and xc.shape[2] == 50, f"Concat: expected [B,144,50], got {tuple(xc.shape)}"
        xs = self.separable_conv(xc)
        assert xs.shape[1] == 24 and xs.shape[2] == 41, f"Separable: expected [B,24,41], got {tuple(xs.shape)}"
        xp = self.maxpool(xs)
        assert xp.shape[1] == 24 and xp.shape[2] == 5, f"Pool: expected [B,24,5], got {tuple(xp.shape)}"
        xf = xp.flatten(1)
        assert xf.shape[1] == 120, f"Flatten: expected [B,120], got {tuple(xf.shape)}"
        return True

# =============================================================================
# DATAMODULE SETUP
# =============================================================================
print("\n" + "=" * 60)
print("SETTING UP DATAMODULE WITH FILTERBANK")
print("=" * 60)

train_loader = dm_apple.train_dataloader()
val_loader = dm_apple.val_dataloader()
test_loader = dm_apple.test_dataloader()

x0, _, y0 = next(iter(train_loader))
T, C = x0.shape[1], x0.shape[2]
expected_C = N_IMU_AXES * N_FILTERBANK
assert C == expected_C, f"Expected {expected_C} channels, got {C}. Ensure enable_imu_filterbank=True"

train_labels = dm_apple.train_dataset.get_label_list()
unique_labels = sorted(set(train_labels))
n_classes = len(unique_labels)

print("Input shape:", tuple(x0.shape))
print("n_classes:", n_classes, "unique labels:", unique_labels)


save_json("data_overview.json", {
    "selected_classes": SELECTED_CLASSES,
    "add_negative_class": bool(ADD_NEGATIVE_CLASS),
    "negative_class_ids": list(NEGATIVE_CLASS_IDS) if NEGATIVE_CLASS_IDS is not None else None,
    "input_T": int(T),
    "input_C": int(C),
    "n_classes": int(n_classes),
    "unique_labels": unique_labels,
})

# -------------------------------------------------------------------------
# ADD THIS BLOCK HERE (right before class_names construction)
# -------------------------------------------------------------------------
idx_to_name = {
    0:  "double_clench",
    1:  "double_pinch",
    2:  "pinch_down",
    3:  "pinch_up",
    4:  "slide",
    5:  "Negative",   # only if class id 5 exists in this run
}
NEG_NAME = "Negative"
NEG_ID_EXPECTED = 5

# Prefer a provided mapping if you already have one
if "idx_to_name" in globals() and isinstance(idx_to_name, dict):
    try:
        _tmp = {int(k): str(v) for k, v in idx_to_name.items()}
        class_names = [str(_tmp.get(i, f"Class {i}")) for i in range(n_classes)]
    except Exception:
        class_names = [f"Class {i}" for i in range(n_classes)]
else:
    class_names = [f"Class {i}" for i in range(n_classes)]
save_json("class_names.json", {"class_names": class_names})

# Prefer a provided mapping if you already have one
if "idx_to_name" in globals() and isinstance(idx_to_name, dict):
    try:
        _tmp = {int(k): str(v) for k, v in idx_to_name.items()}
        class_names = [str(_tmp.get(i, f"Class {i}")) for i in range(n_classes)]
    except Exception:
        class_names = [f"Class {i}" for i in range(n_classes)]
else:
    class_names = [f"Class {i}" for i in range(n_classes)]
save_json("class_names.json", {"class_names": class_names})

# =============================================================================
# BUILD MODEL
# =============================================================================
model = AppleModel(
    n_classes=n_classes,
    n_axes=N_IMU_AXES,
    n_filterbank=N_FILTERBANK,
    seq_len=SEQ_LEN,
    dropout=MODEL_DROPOUT,
).to(device)

with torch.no_grad():
    model.check_expected_shapes(x0.to(device))
print("Shape checks passed.")

# =============================================================================
# PARAMETER COUNT
# =============================================================================
n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_text = (
    "PARAMETER STATISTICS\n"
    f"Total params: {n_total:,}\n"
    f"Trainable params: {n_trainable:,}\n"
)
print(param_text)
save_text("parameter_stats.txt", param_text)

# =============================================================================
# CLASS WEIGHTS & LOSS
# =============================================================================
class_counts = Counter(train_labels)
class_weights = torch.zeros(n_classes, dtype=torch.float32)
for cls_id, count in class_counts.items():
    class_weights[int(cls_id)] = 1.0 / (float(count) + 1e-8)
class_weights = class_weights * (n_classes / class_weights.sum())

save_json("train_class_counts.json", {str(k): int(v) for k, v in class_counts.items()})
save_json("train_class_weights.json", {"weights": class_weights.tolist()})

criterion = nn.CrossEntropyLoss(
    weight=class_weights.to(device),
    label_smoothing=float(LABEL_SMOOTHING),
)

# =============================================================================
# OPTIMIZER + EXPONENTIAL LR SCHEDULER
# =============================================================================
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(LR),
    weight_decay=float(WEIGHT_DECAY),
)

# Exponential decay: lr = lr0 * gamma^epoch, gamma chosen to decay ~100x over EPOCHS
gamma = float(math.exp(math.log(0.01) / max(1, int(EPOCHS))))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=gamma)

save_json("optimizer_scheduler.json", {
    "optimizer": "AdamW",
    "lr0": float(LR),
    "weight_decay": float(WEIGHT_DECAY),
    "scheduler": "ExponentialLR",
    "gamma": gamma,
})

# =============================================================================
# STANDARDIZED CONFUSION MATRIX PLOTTING (FROM TRANSFORMER)
# =============================================================================
def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str,
    annot: bool = True,
    annot_fmt: str = ".2f",
    dpi: int = 200,
):
    """Plot row-normalized confusion matrix (0-1 scale)"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_norm01.npy"), cm_norm.astype(np.float32))
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)

    return cm_norm, cm_counts

def plot_conf_mat_counts(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str = "Blues",
    annot: bool = True,
    dpi: int = 200,
    max_annot: int = 9999,
):
    """Plot count-based confusion matrix"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_counts,
        annot=annot,
        fmt="d",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
        vmin=0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    if annot and max_annot is not None:
        for t in ax.texts:
            try:
                v = int(t.get_text())
                if v > int(max_annot):
                    t.set_text(f">{max_annot}")
            except Exception:
                pass

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)
    return cm_counts

# =============================================================================
# COMPREHENSIVE METRICS HELPERS (FROM TRANSFORMER)
# =============================================================================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out

def per_class_f1(y_true, y_pred, n_classes_local: int, zero_division=0):
    """Compute per-class precision, recall, F1, and support"""
    labels = np.arange(int(n_classes_local))
    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=zero_division
    )
    return {
        "precision": [float(v) for v in p],
        "recall":    [float(v) for v in r],
        "f1":        [float(v) for v in f1],
        "support":   [int(v) for v in s],
    }

def _cuda_sync_if_needed(dev: torch.device):
    """Synchronize CUDA if needed for accurate timing"""
    if str(dev).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()

# =============================================================================
# ENHANCED EVALUATION WITH TIMING + COMPREHENSIVE METRICS
# =============================================================================
@torch.no_grad()
def evaluate_comprehensive(loader, desc="Eval", warmup_batches=5):
    """
    Comprehensive evaluation with timing and all metrics.
    Returns: metrics dict, y_true, y_pred
    """
    model.eval()
    total_loss = 0.0
    all_y, all_p = [], []
    n_seen = 0
    
    times = []
    did_sync_after_warmup = False
    
    for bi, (xb, _, yb) in enumerate(tqdm(loader, desc=desc, leave=False)):
        xb, yb = xb.to(device), yb.to(device)
        
        do_time = (bi >= int(warmup_batches))
        
        if do_time and (not did_sync_after_warmup):
            _cuda_sync_if_needed(device)
            did_sync_after_warmup = True
        
        if do_time:
            t0 = time.perf_counter()
            logits = model(xb)
            _cuda_sync_if_needed(device)
            t1 = time.perf_counter()
            times.append((t1 - t0))
        else:
            logits = model(xb)
        
        loss = criterion(logits, yb)
        total_loss += float(loss.item()) * xb.size(0)
        
        preds = logits.argmax(dim=1).detach().cpu()
        all_y.append(yb.detach().cpu())
        all_p.append(preds)
        n_seen += int(yb.size(0))
    
    y_true = torch.cat(all_y).numpy()
    y_pred = torch.cat(all_p).numpy()
    
    loss_avg = total_loss / max(1, n_seen)
    
    if y_true.size:
        acc = float((y_pred == y_true).mean())
        prf = prf_all_averages(y_true, y_pred, zero_division=0)
        per_cls = per_class_f1(y_true, y_pred, n_classes_local=int(n_classes), zero_division=0)
    else:
        acc = float("nan")
        prf = {
            "macro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "micro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "weighted": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
        }
        per_cls = {"precision": [], "recall": [], "f1": [], "support": []}
    
    # Timing stats
    if len(times) > 0 and n_seen > 0:
        total_t = float(np.sum(times))
        ms_per_batch = float(1000.0 * np.mean(times))
        ms_per_sample = float(1000.0 * (total_t / float(n_seen)))
        n_timed_batches = int(len(times))
    else:
        total_t = 0.0
        ms_per_batch = float("nan")
        ms_per_sample = float("nan")
        n_timed_batches = 0
    
    timing = {
        "warmup_batches": int(warmup_batches),
        "timed_batches": int(n_timed_batches),
        "timed_total_seconds": float(total_t),
        "ms_per_batch": float(ms_per_batch),
        "ms_per_sample": float(ms_per_sample),
    }
    
    metrics = {
        "loss": float(loss_avg),
        "acc": float(acc),
        "prf": prf,
        "per_class": per_cls,
        "timing": timing,
    }
    
    return metrics, y_true, y_pred

# =============================================================================
# TRAINING LOOP WITH COMPREHENSIVE METRICS
# =============================================================================
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1_macro": [],
    "train_f1_micro": [],
    "train_f1_weighted": [],
    "val_f1_macro": [],
    "val_f1_micro": [],
    "val_f1_weighted": [],
    "train_precision_macro": [],
    "train_recall_macro": [],
    "val_precision_macro": [],
    "val_recall_macro": [],
    "lr": [],
}

best_state = None
best_f1 = -1.0
no_improve = 0

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60 + "\n")

for epoch in range(1, int(EPOCHS) + 1):
    model.train()
    running_loss = 0.0
    all_y_train, all_p_train = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{int(EPOCHS)}", leave=False)
    for xb, _, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        running_loss += float(loss.item()) * xb.size(0)
        preds = logits.argmax(dim=1).detach().cpu()
        all_p_train.append(preds)
        all_y_train.append(yb.detach().cpu())
        pbar.set_postfix(loss=float(loss.item()))

    scheduler.step()
    current_lr = float(optimizer.param_groups[0]["lr"])

    # Train metrics
    train_loss = running_loss / max(1, len(train_loader.dataset))
    y_train_true = torch.cat(all_y_train).numpy()
    y_train_pred = torch.cat(all_p_train).numpy()
    train_acc = float((y_train_true == y_train_pred).mean())
    train_prf = prf_all_averages(y_train_true, y_train_pred, zero_division=0)

    # Val metrics (comprehensive)
    val_metrics, _, _ = evaluate_comprehensive(val_loader, desc="Val", warmup_batches=5)

    # Record history
    history["epoch"].append(int(epoch))
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_metrics["loss"]))
    history["train_acc"].append(float(train_acc))
    history["val_acc"].append(float(val_metrics["acc"]))
    
    history["train_f1_macro"].append(float(train_prf["macro"]["f1"]))
    history["train_f1_micro"].append(float(train_prf["micro"]["f1"]))
    history["train_f1_weighted"].append(float(train_prf["weighted"]["f1"]))
    history["val_f1_macro"].append(float(val_metrics["prf"]["macro"]["f1"]))
    history["val_f1_micro"].append(float(val_metrics["prf"]["micro"]["f1"]))
    history["val_f1_weighted"].append(float(val_metrics["prf"]["weighted"]["f1"]))
    
    history["train_precision_macro"].append(float(train_prf["macro"]["precision"]))
    history["train_recall_macro"].append(float(train_prf["macro"]["recall"]))
    history["val_precision_macro"].append(float(val_metrics["prf"]["macro"]["precision"]))
    history["val_recall_macro"].append(float(val_metrics["prf"]["macro"]["recall"]))
    
    history["lr"].append(float(current_lr))

    print(
        f"Epoch {epoch:03d} | lr={current_lr:.3e} | "
        f"train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} | "
        f"train_acc={train_acc:.4f} val_acc={val_metrics['acc']:.4f} | "
        f"train_f1={train_prf['macro']['f1']:.4f} val_f1={val_metrics['prf']['macro']['f1']:.4f}"
    )

    val_f1 = val_metrics["prf"]["macro"]["f1"]
    if val_f1 > best_f1 + 1e-4:
        best_f1 = float(val_f1)
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= int(PATIENCE):
            print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {int(PATIENCE)} epochs).")
            break

save_json("history.json", history)

# =============================================================================
# LOAD BEST MODEL & SAVE CHECKPOINT
# =============================================================================
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model with val macro-F1 = {best_f1:.4f}")
else:
    print("No improvement tracked; using last epoch weights.")

SAVE_PATH = apath("filterbank_imu_cnn_best.pt")
torch.save(
    {
        "model_state": model.state_dict(),
        "config": {
            "SAMPLING_RATE": SAMPLING_RATE,
            "BATCH_SIZE": BATCH_SIZE,
            "EPOCHS": EPOCHS,
            "PATIENCE": PATIENCE,
            "LR": LR,
            "WEIGHT_DECAY": WEIGHT_DECAY,
            "LABEL_SMOOTHING": LABEL_SMOOTHING,
            "n_classes": n_classes,
            "n_axes": N_IMU_AXES,
            "n_filterbank": N_FILTERBANK,
            "seq_len": SEQ_LEN,
            "dropout": MODEL_DROPOUT,
            "SELECTED_CLASSES": SELECTED_CLASSES,
            "NEGATIVE_CLASS_IDS": list(NEGATIVE_CLASS_IDS) if NEGATIVE_CLASS_IDS is not None else None,
        },
        "class_weights": class_weights.cpu(),
        "history": history,
        "best_val_f1": best_f1,
    },
    SAVE_PATH,
)
save_text("best_checkpoint_path.txt", SAVE_PATH)

# =============================================================================
# WINDOW-LEVEL EVALUATION (VAL + TEST) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION")
print("=" * 60)

val_metrics, y_val_true, y_val_pred = evaluate_comprehensive(val_loader, desc="Val-final", warmup_batches=5)
test_metrics, y_test_true, y_test_pred = evaluate_comprehensive(test_loader, desc="Test", warmup_batches=5)

save_json(
    "metrics_window.json",
    {"val": val_metrics, "test": test_metrics, "best_ckpt": str(SAVE_PATH)},
)

# Detailed summary with all metrics
save_text(
    "metrics_summary_window.txt",
    "WINDOW-LEVEL METRICS\n"
    f"VAL  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} ms/sample={val_metrics['timing']['ms_per_sample']:.4f}\n"
    f"VAL  macro    P/R/F1={val_metrics['prf']['macro']['precision']:.4f}/{val_metrics['prf']['macro']['recall']:.4f}/{val_metrics['prf']['macro']['f1']:.4f}\n"
    f"VAL  micro    P/R/F1={val_metrics['prf']['micro']['precision']:.4f}/{val_metrics['prf']['micro']['recall']:.4f}/{val_metrics['prf']['micro']['f1']:.4f}\n"
    f"VAL  weighted P/R/F1={val_metrics['prf']['weighted']['precision']:.4f}/{val_metrics['prf']['weighted']['recall']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} ms/sample={test_metrics['timing']['ms_per_sample']:.4f}\n"
    f"TEST macro    P/R/F1={test_metrics['prf']['macro']['precision']:.4f}/{test_metrics['prf']['macro']['recall']:.4f}/{test_metrics['prf']['macro']['f1']:.4f}\n"
    f"TEST micro    P/R/F1={test_metrics['prf']['micro']['precision']:.4f}/{test_metrics['prf']['micro']['recall']:.4f}/{test_metrics['prf']['micro']['f1']:.4f}\n"
    f"TEST weighted P/R/F1={test_metrics['prf']['weighted']['precision']:.4f}/{test_metrics['prf']['weighted']['recall']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
)

save_json(
    "val_test_support.json",
    {
        "val_support": np.bincount(y_val_true, minlength=int(n_classes)).tolist(),
        "test_support": np.bincount(y_test_true, minlength=int(n_classes)).tolist(),
    },
)

# Confusion matrices - WINDOW LEVEL (Blues for window, as per Transformer)
_ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, normalized 0-1)",
    class_names=class_names,
    out_png="cm_val_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, normalized 0-1)",
    class_names=class_names,
    out_png="cm_test_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, counts)",
    class_names=class_names,
    out_png="cm_val_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, counts)",
    class_names=class_names,
    out_png="cm_test_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Classification reports
labels = np.arange(n_classes)
val_report = classification_report(y_val_true, y_val_pred, labels=labels, target_names=class_names, digits=3, zero_division=0)
test_report = classification_report(y_test_true, y_test_pred, labels=labels, target_names=class_names, digits=3, zero_division=0)
save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)

# =============================================================================
# CLIP-LEVEL EVALUATION (EARLY-STOP) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION")
print("=" * 60)

def early_stop_classwise(seq, k_map=None, default_k=3):
    """Early-stop aggregation with class-wise k values"""
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        v = int(v)
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = int(k_map.get(v, default_k))
        if run_len >= k_c:
            return v
    return None

test_ds = dm_apple.test_dataset
base_ds = getattr(test_ds, "base", test_ds)
n_clips = len(base_ds)

if hasattr(test_ds, "_index"):
    clip_ids_for_windows = [pair[0] for pair in test_ds._index]
else:
    clip_ids_for_windows = list(range(len(test_ds)))

clip_metrics = None
clip_targets = None
clip_preds = None

if len(clip_ids_for_windows) != len(y_test_true):
    save_text(
        "clip_eval_note.txt",
        f"Skipping clip-level: clip_ids_for_windows={len(clip_ids_for_windows)} != n_windows={len(y_test_true)}"
    )
else:
    clip_to_preds = defaultdict(list)
    clip_to_targs = defaultdict(list)
    for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
        clip_to_preds[int(cid)].append(int(p))
        clip_to_targs[int(cid)].append(int(t))

    # Get clip targets
    clip_targets_list = []
    for cid in range(int(n_clips)):
        if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
            clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
            clip_targets_list.append(int(base_ds.label_map[clean]))
        else:
            tgts = clip_to_targs.get(cid, [])
            clip_targets_list.append(int(max(set(tgts), key=tgts.count)) if tgts else 0)
    clip_targets = np.array(clip_targets_list, dtype=int)

    # Get clip predictions with early-stop
    k_map = CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else {}
    clip_preds_list = []
    missed_early = 0
    for cid in range(int(n_clips)):
        seq = clip_to_preds.get(cid, [])
        chosen = early_stop_classwise(seq, k_map=k_map, default_k=int(CLIP_DEFAULT_K))
        if chosen is None:
            missed_early += 1
            chosen = int(max(set(seq), key=seq.count) if seq else clip_targets[cid])
        clip_preds_list.append(int(chosen))
    clip_preds = np.array(clip_preds_list, dtype=int)

    # Compute comprehensive clip metrics
    clip_acc = float((clip_preds == clip_targets).mean()) if clip_targets.size else float("nan")
    clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0) if clip_targets.size else None
    clip_per_cls = per_class_f1(clip_targets, clip_preds, n_classes_local=int(n_classes), zero_division=0) if clip_targets.size else None

    # Timing for aggregation (minimal)
    t0 = time.perf_counter()
    _ = clip_preds
    t1 = time.perf_counter()
    clip_agg_ms_total = float(1000.0 * (t1 - t0))
    clip_agg_ms_per_clip = float(clip_agg_ms_total / max(1, int(n_clips)))

    clip_metrics = {
        "acc": float(clip_acc),
        "prf": clip_prf,
        "per_class": clip_per_cls,
        "timing": {
            "window_forward_ms_per_sample": float(test_metrics["timing"]["ms_per_sample"]),
            "aggregation_ms_total": float(clip_agg_ms_total),
            "aggregation_ms_per_clip": float(clip_agg_ms_per_clip),
        },
        "early_stop": {
            "default_k": int(CLIP_DEFAULT_K),
            "k_map_size": int(len(k_map)),
            "missed_early": int(missed_early),
            "n_clips": int(n_clips),
        },
    }
    save_json("metrics_clip.json", clip_metrics)

    # Detailed summary
    save_text(
        "metrics_summary_clip.txt",
        "CLIP-LEVEL METRICS\n"
        f"TEST acc={clip_acc:.4f}\n"
        f"TEST macro    P/R/F1={clip_prf['macro']['precision']:.4f}/{clip_prf['macro']['recall']:.4f}/{clip_prf['macro']['f1']:.4f}\n"
        f"TEST micro    P/R/F1={clip_prf['micro']['precision']:.4f}/{clip_prf['micro']['recall']:.4f}/{clip_prf['micro']['f1']:.4f}\n"
        f"TEST weighted P/R/F1={clip_prf['weighted']['precision']:.4f}/{clip_prf['weighted']['recall']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
        f"Forward ms/sample (from window eval)={test_metrics['timing']['ms_per_sample']:.4f}\n"
        f"Aggregation ms/clip={clip_agg_ms_per_clip:.4f}\n"
        f"Early-stop default_k={int(CLIP_DEFAULT_K)} k_map_size={len(k_map)} majority_fallback={missed_early}/{n_clips}\n"
    )

    # Confusion matrices - CLIP LEVEL (Greens for clip/aggregation, as per Transformer)
    _ = plot_conf_mat_norm01(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, normalized 0-1)",
        class_names=class_names,
        out_png="cm_test_clip_norm01.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
        dpi=200,
    )

    _ = plot_conf_mat_counts(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, counts)",
        class_names=class_names,
        out_png="cm_test_clip_counts.png",
        cmap="Greens",
        annot=True,
        dpi=200,
    )

    # Classification report
    clip_report = classification_report(clip_targets, clip_preds, labels=labels, target_names=class_names, digits=3, zero_division=0)
    save_text("report_test_clip.txt", clip_report)

# =============================================================================
# TRAINING CURVES (Loss, Accuracy, F1, LR)
# =============================================================================
print("\n" + "=" * 60)
print("GENERATING TRAINING CURVES")
print("=" * 60)

epochs_hist = history["epoch"]

# Loss curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_loss"], label="Train loss")
plt.plot(epochs_hist, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_loss_train_val.png", dpi=300)

# Accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_acc"], label="Train accuracy")
plt.plot(epochs_hist, history["val_acc"], label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_acc_train_val.png", dpi=300)

# F1 curve (macro)
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["train_f1_macro"], label="Train F1 (macro)")
plt.plot(epochs_hist, history["val_f1_macro"], label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Train vs Validation F1 (macro)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_f1_train_val.png", dpi=300)

# LR curve
plt.figure(figsize=(9, 5))
plt.plot(epochs_hist, history["lr"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule (ExponentialLR)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_lr.png", dpi=300)

# =============================================================================
# FINAL SUMMARY
# =============================================================================
final_summary = (
    "FINAL SUMMARY\n"
    f"Params total: {n_total:,}\n"
    f"Best checkpoint: {SAVE_PATH}\n"
    f"VAL window:  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={val_metrics['prf']['macro']['f1']:.4f}/"
    f"{val_metrics['prf']['micro']['f1']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={test_metrics['prf']['macro']['f1']:.4f}/"
    f"{test_metrics['prf']['micro']['f1']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
    + (
        f"TEST clip:   acc={clip_metrics['acc']:.4f} "
        f"F1(macro/micro/weighted)={clip_metrics['prf']['macro']['f1']:.4f}/"
        f"{clip_metrics['prf']['micro']['f1']:.4f}/{clip_metrics['prf']['weighted']['f1']:.4f}\n"
        if clip_metrics is not None and clip_metrics.get("prf") is not None else
        "TEST clip:   SKIPPED\n"
    ) +
    f"Artifacts:   {APPLE_DIR}\n"
)
print(final_summary)
save_text("final_summary.txt", final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", APPLE_DIR)

## InceptionTime Model

In [ ]:
# =============================================================================
# INCEPTIONTIME
# =============================================================================

# =============================================================================
# CONFIG
# =============================================================================
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE = 8

LR = 1e-3
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.0  # set >0 if you want smoothing

# InceptionTime model params
NB_FILTERS = 32
DEPTH = 6                 # must be multiple of 3
KERNEL_SIZES = [9, 19, 39]
USE_RESIDUAL = True
USE_BOTTLENECK = True
BOTTLENECK_SIZE = 32
DROPOUT_RATE = 0.5

# Exponential LR scheduler
# lr = lr0 * gamma^epoch
DECAY_TARGET = 0.01       # decay to ~1% over MAX_EPOCHS
GAMMA = float(math.exp(math.log(DECAY_TARGET) / max(1, int(MAX_EPOCHS))))

# Clip-level aggregation config
CLIP_K_MAP = None         # example: {0: 3, 1: 3, 2: 3}
CLIP_DEFAULT_K = 3

# =============================================================================
# Run directory under OUTPUTS/inceptiontime/run_<timestamp>
# =============================================================================
INCEPTION_ROOT = os.path.join(OUTPUTS, "inceptiontime")
Path(INCEPTION_ROOT).mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = os.path.join(INCEPTION_ROOT, f"run_{RUN_ID}")
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

def rpath(name: str) -> str:
    return os.path.join(RUN_DIR, name)

def save_text(name: str, text: str) -> str:
    p = rpath(name)
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    return p

def save_json(name: str, obj) -> str:
    p = rpath(name)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    return p

def save_csv(name: str, df: pd.DataFrame, index: bool = False) -> str:
    p = rpath(name)
    df.to_csv(p, index=index)
    return p

def save_npy(name: str, arr: np.ndarray) -> str:
    p = rpath(name)
    np.save(p, arr)
    return p

def save_fig(name: str, dpi: int = 300) -> str:
    p = rpath(name)
    plt.tight_layout()
    plt.savefig(p, dpi=dpi, bbox_inches="tight")
    plt.close()
    return p

print("Run directory:", RUN_DIR)

# =============================================================================
# Reproducibility
# =============================================================================
for name in ["model", "optimizer", "scheduler", "trainer_state", "history"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================================================================
# Model definition
# =============================================================================
class InceptionBlock(nn.Module):
    def __init__(
        self,
        in_channels: int,
        nb_filters: int = 32,
        kernel_sizes=(9, 19, 39),
        bottleneck_size: int = 32,
        use_bottleneck: bool = True,
    ):
        super().__init__()
        self.use_bottleneck = bool(use_bottleneck)

        if self.use_bottleneck:
            self.bottleneck = nn.Conv1d(in_channels, bottleneck_size, kernel_size=1, bias=False)
            conv_in = bottleneck_size
        else:
            conv_in = in_channels

        self.convs = nn.ModuleList([
            nn.Conv1d(conv_in, nb_filters, int(k), padding=int(k) // 2, bias=False)
            for k in kernel_sizes
        ])

        self.maxpool = nn.MaxPool1d(3, stride=1, padding=1)
        self.maxpool_conv = nn.Conv1d(in_channels, nb_filters, kernel_size=1, bias=False)

        total = nb_filters * (len(kernel_sizes) + 1)
        self.proj = nn.Conv1d(total, nb_filters, kernel_size=1, bias=False)
        self.bn = nn.BatchNorm1d(nb_filters)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        inp = x
        if self.use_bottleneck:
            x = self.bottleneck(x)
        outs = [conv(x) for conv in self.convs]
        outs.append(self.maxpool_conv(self.maxpool(inp)))
        x = torch.cat(outs, dim=1)
        x = self.proj(x)
        x = self.bn(x)
        return F.relu(x)

class InceptionTimeModel(nn.Module):
    def __init__(
        self,
        input_channels: int,
        num_classes: int,
        nb_filters: int = 32,
        depth: int = 6,
        kernel_sizes=(9, 19, 39),
        use_residual: bool = True,
        use_bottleneck: bool = True,
        bottleneck_size: int = 32,
        dropout_rate: float = 0.5,
    ):
        super().__init__()
        assert int(depth) % 3 == 0, "depth must be a multiple of 3"
        self.use_residual = bool(use_residual)
        self.nb_filters = int(nb_filters)

        blocks = []
        in_ch = int(input_channels)
        for _ in range(int(depth)):
            blocks.append(
                InceptionBlock(
                    in_channels=in_ch,
                    nb_filters=self.nb_filters,
                    kernel_sizes=kernel_sizes,
                    bottleneck_size=int(bottleneck_size),
                    use_bottleneck=bool(use_bottleneck),
                )
            )
            in_ch = self.nb_filters
        self.blocks = nn.ModuleList(blocks)

        self.num_groups = int(depth) // 3
        self.res_projs = nn.ModuleList()
        for g in range(self.num_groups):
            in_c = int(input_channels) if g == 0 else self.nb_filters
            self.res_projs.append(
                nn.Sequential(
                    nn.Conv1d(in_c, self.nb_filters, kernel_size=1, bias=False),
                    nn.BatchNorm1d(self.nb_filters),
                )
            )

        self.gap = nn.AdaptiveAvgPool1d(1)
        self.dropout = nn.Dropout(float(dropout_rate))
        self.fc = nn.Linear(self.nb_filters, int(num_classes))
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv1d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm1d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, C) -> (B, C, T)
        x = x.transpose(1, 2)
        group_idx = 0
        res = x
        for i, blk in enumerate(self.blocks):
            x = blk(x)
            if self.use_residual and (i + 1) % 3 == 0:
                x = F.relu(x + self.res_projs[group_idx](res))
                res = x
                group_idx += 1
        x = self.gap(x).squeeze(-1)
        x = self.dropout(x)
        return self.fc(x)

# =============================================================================
# setup + infer shapes/classes + class labels
# =============================================================================
train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

x0, _, y0 = next(iter(train_loader))
seq_len = int(x0.shape[1])
n_channels = int(x0.shape[2])

# Robust n_classes: prefer label_map max+1, else from observed labels
label_map = getattr(getattr(dm, "train_dataset", None), "label_map", None)
if isinstance(label_map, dict) and len(label_map) > 0:
    n_classes = int(max(int(v) for v in label_map.values()) + 1)
else:
    # fallback from first batch
    n_classes = int(y0.max().item() + 1)

def build_class_labels(dm, n_classes_local: int):
    td = dm.train_dataset
    base = getattr(td, "base", td)
    lm = getattr(base, "label_map", getattr(td, "label_map", None))

    labels = [None] * n_classes_local
    neg_id = None
    if "SELECTED_CLASSES" in globals() and "ADD_NEGATIVE_CLASS" in globals() and ADD_NEGATIVE_CLASS:
        try:
            neg_id = len(SELECTED_CLASSES)
        except Exception:
            neg_id = None

    if isinstance(lm, dict):
        for name, cid in lm.items():
            try:
                cid = int(cid)
            except Exception:
                continue
            if 0 <= cid < n_classes_local and labels[cid] is None:
                if neg_id is not None and cid == neg_id:
                    labels[cid] = "Negative"
                else:
                    labels[cid] = str(name)

    for i in range(n_classes_local):
        if labels[i] is None:
            labels[i] = "Negative" if (neg_id is not None and i == neg_id) else f"Class {i}"
    return labels

class_labels = build_class_labels(dm, n_classes)

print(f"Sequence length: {seq_len}, Channels: {n_channels}, Classes: {n_classes}")
print("Class labels:", class_labels)

save_json("run_context.json", {
    "run_id": RUN_ID,
    "outputs_root": OUTPUTS,
    "run_dir": RUN_DIR,
    "device": str(device),
})
save_json("config.json", {
    "SEED": SEED,
    "BATCH_SIZE": BATCH_SIZE,
    "MAX_EPOCHS": MAX_EPOCHS,
    "PATIENCE": PATIENCE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "LABEL_SMOOTHING": LABEL_SMOOTHING,
    "NB_FILTERS": NB_FILTERS,
    "DEPTH": DEPTH,
    "KERNEL_SIZES": KERNEL_SIZES,
    "USE_RESIDUAL": USE_RESIDUAL,
    "USE_BOTTLENECK": USE_BOTTLENECK,
    "BOTTLENECK_SIZE": BOTTLENECK_SIZE,
    "DROPOUT_RATE": DROPOUT_RATE,
    "GAMMA": GAMMA,
    "CLIP_DEFAULT_K": CLIP_DEFAULT_K,
    "CLIP_K_MAP": CLIP_K_MAP,
    "seq_len": seq_len,
    "n_channels": n_channels,
    "n_classes": n_classes,
})
save_json("class_labels.json", {"class_labels": class_labels})

# =============================================================================
# Build model + optimizer + scheduler + loss
# =============================================================================
model = InceptionTimeModel(
    input_channels=n_channels,
    num_classes=n_classes,
    nb_filters=NB_FILTERS,
    depth=DEPTH,
    kernel_sizes=KERNEL_SIZES,
    use_residual=USE_RESIDUAL,
    use_bottleneck=USE_BOTTLENECK,
    bottleneck_size=BOTTLENECK_SIZE,
    dropout_rate=DROPOUT_RATE,
).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_text = (
    "PARAMETER STATISTICS\n"
    f"Total params: {n_total:,}\n"
    f"Trainable params: {n_trainable:,}\n"
)
print(param_text)
save_text("parameter_stats.txt", param_text)

optimizer = torch.optim.AdamW(model.parameters(), lr=float(LR), weight_decay=float(WEIGHT_DECAY))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=float(GAMMA))

criterion = nn.CrossEntropyLoss(label_smoothing=float(LABEL_SMOOTHING))

BEST_CKPT_PATH = rpath("best_inceptiontime.pt")

# =============================================================================
# STANDARDIZED CONFUSION MATRIX PLOTTING (FROM TRANSFORMER)
# =============================================================================
def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str,
    annot: bool = True,
    annot_fmt: str = ".2f",
    dpi: int = 200,
):
    """Plot row-normalized confusion matrix (0-1 scale)"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_norm01.npy"), cm_norm.astype(np.float32))
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)

    return cm_norm, cm_counts

def plot_conf_mat_counts(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str = "Blues",
    annot: bool = True,
    dpi: int = 200,
    max_annot: int = 9999,
):
    """Plot count-based confusion matrix"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_counts,
        annot=annot,
        fmt="d",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
        vmin=0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    if annot and max_annot is not None:
        for t in ax.texts:
            try:
                v = int(t.get_text())
                if v > int(max_annot):
                    t.set_text(f">{max_annot}")
            except Exception:
                pass

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)
    return cm_counts

# =============================================================================
# COMPREHENSIVE METRICS HELPERS (FROM TRANSFORMER)
# =============================================================================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out

def per_class_f1(y_true, y_pred, n_classes_local: int, zero_division=0):
    """Compute per-class precision, recall, F1, and support"""
    labels = np.arange(int(n_classes_local))
    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=zero_division
    )
    return {
        "precision": [float(v) for v in p],
        "recall":    [float(v) for v in r],
        "f1":        [float(v) for v in f1],
        "support":   [int(v) for v in s],
    }

def _cuda_sync_if_needed(dev: torch.device):
    """Synchronize CUDA if needed for accurate timing"""
    if str(dev).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()

# =============================================================================
# ENHANCED EVALUATION WITH TIMING + COMPREHENSIVE METRICS
# =============================================================================
@torch.no_grad()
def evaluate_comprehensive(loader, desc="Eval", warmup_batches=5):
    """
    Comprehensive evaluation with timing and all metrics.
    Returns: metrics dict, y_true, y_pred
    """
    model.eval()
    total_loss = 0.0
    preds_all, targs_all = [], []
    n_seen = 0
    
    times = []
    did_sync_after_warmup = False
    
    for bi, (xb, _, yb) in enumerate(tqdm(loader, desc=desc, leave=False)):
        xb, yb = xb.to(device), yb.to(device)
        
        do_time = (bi >= int(warmup_batches))
        
        if do_time and (not did_sync_after_warmup):
            _cuda_sync_if_needed(device)
            did_sync_after_warmup = True
        
        if do_time:
            t0 = time.perf_counter()
            logits = model(xb)
            _cuda_sync_if_needed(device)
            t1 = time.perf_counter()
            times.append((t1 - t0))
        else:
            logits = model(xb)
        
        loss = criterion(logits, yb)
        total_loss += float(loss.item()) * xb.size(0)
        
        preds = torch.argmax(logits, dim=1).detach().cpu()
        targs_all.append(yb.detach().cpu())
        preds_all.append(preds)
        n_seen += int(yb.size(0))
    
    y_pred = torch.cat(preds_all).numpy() if preds_all else np.array([], dtype=int)
    y_true = torch.cat(targs_all).numpy() if targs_all else np.array([], dtype=int)
    
    loss_avg = total_loss / max(1, n_seen)
    
    if y_true.size:
        acc = float((y_pred == y_true).mean())
        prf = prf_all_averages(y_true, y_pred, zero_division=0)
        per_cls = per_class_f1(y_true, y_pred, n_classes_local=int(n_classes), zero_division=0)
    else:
        acc = float("nan")
        prf = {
            "macro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "micro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "weighted": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
        }
        per_cls = {"precision": [], "recall": [], "f1": [], "support": []}
    
    # Timing stats
    if len(times) > 0 and n_seen > 0:
        total_t = float(np.sum(times))
        ms_per_batch = float(1000.0 * np.mean(times))
        ms_per_sample = float(1000.0 * (total_t / float(n_seen)))
        n_timed_batches = int(len(times))
    else:
        total_t = 0.0
        ms_per_batch = float("nan")
        ms_per_sample = float("nan")
        n_timed_batches = 0
    
    timing = {
        "warmup_batches": int(warmup_batches),
        "timed_batches": int(n_timed_batches),
        "timed_total_seconds": float(total_t),
        "ms_per_batch": float(ms_per_batch),
        "ms_per_sample": float(ms_per_sample),
    }
    
    metrics = {
        "loss": float(loss_avg),
        "acc": float(acc),
        "prf": prf,
        "per_class": per_cls,
        "timing": timing,
    }
    
    return metrics, y_true, y_pred

def early_stop_classwise(seq, k_map=None, default_k=3):
    """Early-stop aggregation with class-wise k values"""
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        v = int(v)
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = int(k_map.get(v, default_k))
        if run_len >= k_c:
            return v
    return None

# =============================================================================
# TRAINING LOOP WITH COMPREHENSIVE METRICS
# =============================================================================
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1_macro": [],
    "train_f1_micro": [],
    "train_f1_weighted": [],
    "val_f1_macro": [],
    "val_f1_micro": [],
    "val_f1_weighted": [],
    "train_precision_macro": [],
    "train_recall_macro": [],
    "val_precision_macro": [],
    "val_recall_macro": [],
    "lr": [],
}

best_val_f1 = None
best_state = None
no_improve = 0

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60 + "\n")

for epoch in range(1, int(MAX_EPOCHS) + 1):
    model.train()
    total_loss = 0.0
    preds_all, targs_all = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{int(MAX_EPOCHS)}", leave=False)
    for xb, _, yb in pbar:
        xb, yb = xb.to(device), yb.to(device)

        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += float(loss.item()) * xb.size(0)
        preds_all.append(torch.argmax(logits, dim=1).detach().cpu())
        targs_all.append(yb.detach().cpu())
        pbar.set_postfix(loss=float(loss.item()))

    scheduler.step()
    lr_now = float(optimizer.param_groups[0]["lr"])

    # Train metrics
    y_train_pred = torch.cat(preds_all).numpy()
    y_train_true = torch.cat(targs_all).numpy()

    train_loss = total_loss / max(1, len(train_loader.dataset))
    train_acc = float((y_train_pred == y_train_true).mean())
    train_prf = prf_all_averages(y_train_true, y_train_pred, zero_division=0)

    # Val metrics (comprehensive)
    val_metrics, _, _ = evaluate_comprehensive(val_loader, desc="Val", warmup_batches=5)

    # Record history
    history["epoch"].append(int(epoch))
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_metrics["loss"]))
    history["train_acc"].append(float(train_acc))
    history["val_acc"].append(float(val_metrics["acc"]))
    
    history["train_f1_macro"].append(float(train_prf["macro"]["f1"]))
    history["train_f1_micro"].append(float(train_prf["micro"]["f1"]))
    history["train_f1_weighted"].append(float(train_prf["weighted"]["f1"]))
    history["val_f1_macro"].append(float(val_metrics["prf"]["macro"]["f1"]))
    history["val_f1_micro"].append(float(val_metrics["prf"]["micro"]["f1"]))
    history["val_f1_weighted"].append(float(val_metrics["prf"]["weighted"]["f1"]))
    
    history["train_precision_macro"].append(float(train_prf["macro"]["precision"]))
    history["train_recall_macro"].append(float(train_prf["macro"]["recall"]))
    history["val_precision_macro"].append(float(val_metrics["prf"]["macro"]["precision"]))
    history["val_recall_macro"].append(float(val_metrics["prf"]["macro"]["recall"]))
    
    history["lr"].append(float(lr_now))

    print(
        f"Epoch {epoch:03d} | lr={lr_now:.3e} | "
        f"train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} | "
        f"train_acc={train_acc:.4f} val_acc={val_metrics['acc']:.4f} | "
        f"train_f1={train_prf['macro']['f1']:.4f} val_f1={val_metrics['prf']['macro']['f1']:.4f}"
    )

    # persist metrics each epoch
    save_csv("inceptiontime_metrics.csv", pd.DataFrame(history), index=False)
    save_json("history.json", history)

    # early stopping on val macro-f1
    val_f1 = val_metrics["prf"]["macro"]["f1"]
    improved = (best_val_f1 is None) or (val_f1 > best_val_f1 + 1e-4)
    if improved:
        best_val_f1 = float(val_f1)
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_CKPT_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= int(PATIENCE):
            print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {int(PATIENCE)} epochs).")
            break

# Load best for evaluation
if os.path.isfile(BEST_CKPT_PATH):
    model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device))
    model.to(device).eval()
save_text("best_checkpoint_path.txt", BEST_CKPT_PATH)

# =============================================================================
# TRAINING CURVES (Loss, Accuracy, F1, LR)
# =============================================================================
print("\n" + "=" * 60)
print("GENERATING TRAINING CURVES")
print("=" * 60)

epochs = history["epoch"]

# Loss curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_loss_train_val.png", dpi=300)

# Accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_acc"], label="Train accuracy")
plt.plot(epochs, history["val_acc"], label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_acc_train_val.png", dpi=300)

# F1 curve (macro)
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_f1_macro"], label="Train F1 (macro)")
plt.plot(epochs, history["val_f1_macro"], label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Train vs Validation F1 (macro)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_f1_train_val.png", dpi=300)

# LR curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["lr"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule (ExponentialLR)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_lr.png", dpi=300)

# =============================================================================
# WINDOW-LEVEL EVALUATION (VAL + TEST) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION")
print("=" * 60)

val_metrics, y_val_true, y_val_pred = evaluate_comprehensive(val_loader, desc="Val-final", warmup_batches=5)
test_metrics, y_test_true, y_test_pred = evaluate_comprehensive(test_loader, desc="Test", warmup_batches=5)

save_json(
    "metrics_window.json",
    {"val": val_metrics, "test": test_metrics, "best_ckpt": str(BEST_CKPT_PATH)},
)

# Detailed summary with all metrics
save_text(
    "metrics_summary_window.txt",
    "WINDOW-LEVEL METRICS\n"
    f"VAL  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} ms/sample={val_metrics['timing']['ms_per_sample']:.4f}\n"
    f"VAL  macro    P/R/F1={val_metrics['prf']['macro']['precision']:.4f}/{val_metrics['prf']['macro']['recall']:.4f}/{val_metrics['prf']['macro']['f1']:.4f}\n"
    f"VAL  micro    P/R/F1={val_metrics['prf']['micro']['precision']:.4f}/{val_metrics['prf']['micro']['recall']:.4f}/{val_metrics['prf']['micro']['f1']:.4f}\n"
    f"VAL  weighted P/R/F1={val_metrics['prf']['weighted']['precision']:.4f}/{val_metrics['prf']['weighted']['recall']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} ms/sample={test_metrics['timing']['ms_per_sample']:.4f}\n"
    f"TEST macro    P/R/F1={test_metrics['prf']['macro']['precision']:.4f}/{test_metrics['prf']['macro']['recall']:.4f}/{test_metrics['prf']['macro']['f1']:.4f}\n"
    f"TEST micro    P/R/F1={test_metrics['prf']['micro']['precision']:.4f}/{test_metrics['prf']['micro']['recall']:.4f}/{test_metrics['prf']['micro']['f1']:.4f}\n"
    f"TEST weighted P/R/F1={test_metrics['prf']['weighted']['precision']:.4f}/{test_metrics['prf']['weighted']['recall']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
)

save_json(
    "val_test_support.json",
    {
        "val_support": np.bincount(y_val_true, minlength=int(n_classes)).tolist(),
        "test_support": np.bincount(y_test_true, minlength=int(n_classes)).tolist(),
    },
)

# Confusion matrices - WINDOW LEVEL (Blues for window, as per Transformer)
_ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_val_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_test_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, counts)",
    class_names=class_labels,
    out_png="cm_val_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, counts)",
    class_names=class_labels,
    out_png="cm_test_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Classification reports
labels = np.arange(n_classes)
val_report = classification_report(y_val_true, y_val_pred, labels=labels, target_names=class_labels, digits=3, zero_division=0)
test_report = classification_report(y_test_true, y_test_pred, labels=labels, target_names=class_labels, digits=3, zero_division=0)
save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)

# =============================================================================
# CLIP-LEVEL EVALUATION (EARLY-STOP) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION")
print("=" * 60)

test_ds = dm.test_dataset
base_ds = getattr(test_ds, "base", test_ds)
n_clips = len(base_ds)

# Map each window to a clip id (preferred: dataset._index)
if hasattr(test_ds, "_index"):
    clip_ids_for_windows = [pair[0] for pair in test_ds._index]
else:
    # fallback: each sample is its own clip (not ideal, but safe)
    clip_ids_for_windows = list(range(len(test_ds)))

clip_metrics = None
clip_targets = None
clip_preds = None

if len(clip_ids_for_windows) != len(y_test_true):
    save_text(
        "clip_eval_note.txt",
        f"Skipping clip-level: clip_ids_for_windows={len(clip_ids_for_windows)} != n_windows={len(y_test_true)}"
    )
else:
    clip_to_preds = defaultdict(list)
    clip_to_targs = defaultdict(list)
    for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
        clip_to_preds[int(cid)].append(int(p))
        clip_to_targs[int(cid)].append(int(t))

    # Clip targets: prefer base_ds.samples label if available; else majority of window targets
    clip_targets_list = []
    for cid in range(int(n_clips)):
        if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
            clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
            clip_targets_list.append(int(base_ds.label_map[clean]))
        else:
            tgts = clip_to_targs.get(cid, [])
            clip_targets_list.append(int(max(set(tgts), key=tgts.count)) if tgts else 0)
    clip_targets = np.array(clip_targets_list, dtype=int)

    # Get clip predictions with early-stop
    k_map = CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else {}
    clip_preds_list = []
    missed_early = 0
    for cid in range(int(n_clips)):
        seq = clip_to_preds.get(cid, [])
        chosen = early_stop_classwise(seq, k_map=k_map, default_k=int(CLIP_DEFAULT_K))
        if chosen is None:
            missed_early += 1
            chosen = int(max(set(seq), key=seq.count) if seq else clip_targets[cid])
        clip_preds_list.append(int(chosen))
    clip_preds = np.array(clip_preds_list, dtype=int)

    # Compute comprehensive clip metrics
    clip_acc = float((clip_preds == clip_targets).mean()) if clip_targets.size else float("nan")
    clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0) if clip_targets.size else None
    clip_per_cls = per_class_f1(clip_targets, clip_preds, n_classes_local=int(n_classes), zero_division=0) if clip_targets.size else None

    # Timing for aggregation (minimal)
    t0 = time.perf_counter()
    _ = clip_preds
    t1 = time.perf_counter()
    clip_agg_ms_total = float(1000.0 * (t1 - t0))
    clip_agg_ms_per_clip = float(clip_agg_ms_total / max(1, int(n_clips)))

    clip_metrics = {
        "acc": float(clip_acc),
        "prf": clip_prf,
        "per_class": clip_per_cls,
        "timing": {
            "window_forward_ms_per_sample": float(test_metrics["timing"]["ms_per_sample"]),
            "aggregation_ms_total": float(clip_agg_ms_total),
            "aggregation_ms_per_clip": float(clip_agg_ms_per_clip),
        },
        "early_stop": {
            "default_k": int(CLIP_DEFAULT_K),
            "k_map_size": int(len(k_map)),
            "missed_early": int(missed_early),
            "n_clips": int(n_clips),
        },
    }
    save_json("metrics_clip.json", clip_metrics)

    # Detailed summary
    save_text(
        "metrics_summary_clip.txt",
        "CLIP-LEVEL METRICS\n"
        f"TEST acc={clip_acc:.4f}\n"
        f"TEST macro    P/R/F1={clip_prf['macro']['precision']:.4f}/{clip_prf['macro']['recall']:.4f}/{clip_prf['macro']['f1']:.4f}\n"
        f"TEST micro    P/R/F1={clip_prf['micro']['precision']:.4f}/{clip_prf['micro']['recall']:.4f}/{clip_prf['micro']['f1']:.4f}\n"
        f"TEST weighted P/R/F1={clip_prf['weighted']['precision']:.4f}/{clip_prf['weighted']['recall']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
        f"Forward ms/sample (from window eval)={test_metrics['timing']['ms_per_sample']:.4f}\n"
        f"Aggregation ms/clip={clip_agg_ms_per_clip:.4f}\n"
        f"Early-stop default_k={int(CLIP_DEFAULT_K)} k_map_size={len(k_map)} majority_fallback={missed_early}/{n_clips}\n"
    )

    # Confusion matrices - CLIP LEVEL (Greens for clip/aggregation, as per Transformer)
    _ = plot_conf_mat_norm01(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, normalized 0-1)",
        class_names=class_labels,
        out_png="cm_test_clip_norm01.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
        dpi=200,
    )

    _ = plot_conf_mat_counts(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, counts)",
        class_names=class_labels,
        out_png="cm_test_clip_counts.png",
        cmap="Greens",
        annot=True,
        dpi=200,
    )

    # Classification report
    clip_report = classification_report(clip_targets, clip_preds, labels=labels, target_names=class_labels, digits=3, zero_division=0)
    save_text("report_test_clip.txt", clip_report)

# =============================================================================
# FINAL SUMMARY
# =============================================================================
final_summary = (
    "FINAL SUMMARY\n"
    f"Params total: {n_total:,}\n"
    f"Best checkpoint: {BEST_CKPT_PATH}\n"
    f"VAL window:  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={val_metrics['prf']['macro']['f1']:.4f}/"
    f"{val_metrics['prf']['micro']['f1']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={test_metrics['prf']['macro']['f1']:.4f}/"
    f"{test_metrics['prf']['micro']['f1']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
    + (
        f"TEST clip:   acc={clip_metrics['acc']:.4f} "
        f"F1(macro/micro/weighted)={clip_metrics['prf']['macro']['f1']:.4f}/"
        f"{clip_metrics['prf']['micro']['f1']:.4f}/{clip_metrics['prf']['weighted']['f1']:.4f}\n"
        if clip_metrics is not None and clip_metrics.get("prf") is not None else
        "TEST clip:   SKIPPED\n"
    ) +
    f"Artifacts:   {RUN_DIR}\n"
)
print(final_summary)
save_text("final_summary.txt", final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", RUN_DIR)

## Hydra Model

In [ ]:
# =============================================================================
# HYDRA
# =============================================================================

# =============================================================================
# CONFIG
# =============================================================================
SEED = 42
BATCH_SIZE = 64
MAX_EPOCHS = 100
PATIENCE = 8

LR = 1e-3
WEIGHT_DECAY = 1e-4

# Hydra params
K = 8
G = 64
DROPOUT_1 = 0.3
DROPOUT_2 = 0.5
HIDDEN = 16

# Exponential LR scheduler
DECAY_TARGET = 0.01
GAMMA = float(math.exp(math.log(DECAY_TARGET) / max(1, int(MAX_EPOCHS))))

# Clip-level aggregation config
CLIP_K_MAP = None         # example: {0: 3, 1: 3, 2: 3}
CLIP_DEFAULT_K = 3

# =============================================================================
# Run directory under OUTPUTS/hydra/run_<timestamp>
# =============================================================================
HYDRA_ROOT = os.path.join(OUTPUTS, "hydra")
Path(HYDRA_ROOT).mkdir(parents=True, exist_ok=True)

RUN_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_DIR = os.path.join(HYDRA_ROOT, f"run_{RUN_ID}")
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)

def rpath(name: str) -> str:
    return os.path.join(RUN_DIR, name)

def save_text(name: str, text: str) -> str:
    p = rpath(name)
    with open(p, "w", encoding="utf-8") as f:
        f.write(text)
    return p

def save_json(name: str, obj) -> str:
    p = rpath(name)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2)
    return p

def save_csv(name: str, df: pd.DataFrame, index: bool = False) -> str:
    p = rpath(name)
    df.to_csv(p, index=index)
    return p

def save_npy(name: str, arr: np.ndarray) -> str:
    p = rpath(name)
    np.save(p, arr)
    return p

def save_fig(name: str, dpi: int = 300) -> str:
    p = rpath(name)
    plt.tight_layout()
    plt.savefig(p, dpi=dpi, bbox_inches="tight")
    plt.close()
    return p

print("Run directory:", RUN_DIR)

# =============================================================================
# Reproducibility
# =============================================================================
for name in ["model", "optimizer", "scheduler", "history", "best_state"]:
    if name in globals():
        del globals()[name]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =============================================================================
# Hydra model definition
# =============================================================================
class HydraFeatureExtractor(nn.Module):
    def __init__(self, input_length=300, num_channels=6, k=4, g=32):
        super().__init__()
        self.k = int(k)
        self.g = int(g)
        self.input_length = int(input_length)
        self.num_channels = int(num_channels)

        max_exponent = np.log2((self.input_length - 1) / (9 - 1))
        self.dilations = 2 ** torch.arange(int(max_exponent) + 1)
        self.num_dilations = len(self.dilations)
        self.paddings = torch.div((9 - 1) * self.dilations, 2, rounding_mode="floor").int()

        self.divisor = min(2, self.g)
        self.h = self.g // self.divisor
        self.register_buffer("W", self._init_weights())

    def _init_weights(self):
        W = torch.randn(self.num_dilations, self.divisor, self.k * self.h, self.num_channels, 9)
        W = W - W.mean(-1, keepdims=True)
        W = W / W.abs().sum(-1, keepdims=True)
        return W

    def forward(self, X):
        # X: (B, C, T)
        num_examples = X.shape[0]
        if self.divisor > 1:
            diff_X = torch.diff(X, dim=-1)
            diff_X = torch.nn.functional.pad(diff_X, (0, 1), mode="replicate")

        Z = []
        for d_idx in range(self.num_dilations):
            d = int(self.dilations[d_idx].item())
            p = int(self.paddings[d_idx].item())
            for diff_index in range(self.divisor):
                inp = X if diff_index == 0 else diff_X
                _Z = torch.nn.functional.conv1d(inp, self.W[d_idx, diff_index], dilation=d, padding=p)
                _Z = _Z.view(num_examples, self.h, self.k, -1)

                max_vals, max_idx = _Z.max(2)
                min_vals, min_idx = _Z.min(2)

                count_max = torch.zeros(num_examples, self.h, self.k, device=X.device, dtype=max_vals.dtype)
                count_min = torch.zeros(num_examples, self.h, self.k, device=X.device, dtype=min_vals.dtype)

                count_max.scatter_add_(-1, max_idx, max_vals)
                count_min.scatter_add_(-1, min_idx, torch.ones_like(min_vals))
                Z.append(count_max)
                Z.append(count_min)

        Z = torch.cat(Z, 1).view(num_examples, -1)
        return Z

class HydraNet(nn.Module):
    def __init__(self, input_length=300, num_channels=6, num_classes=5, k=4, g=32, class_weights=None,
                 dropout_1=0.3, dropout_2=0.5, hidden=16):
        super().__init__()
        self.hydra = HydraFeatureExtractor(input_length, num_channels, k, g)

        with torch.no_grad():
            dummy = torch.randn(1, num_channels, input_length)
            num_features = int(self.hydra(dummy).shape[1])

        self.f_mean = nn.Parameter(torch.zeros(num_features), requires_grad=False)
        self.f_std = nn.Parameter(torch.ones(num_features), requires_grad=False)
        self.features_fitted = False

        self.classifier = nn.Sequential(
            nn.Dropout(float(dropout_1)),
            nn.Linear(num_features, int(hidden)),
            nn.ReLU(),
            nn.Dropout(float(dropout_2)),
            nn.Linear(int(hidden), int(num_classes)),
        )

        w = None
        if class_weights is not None:
            w = torch.tensor(class_weights, dtype=torch.float32)
        self.ce = nn.CrossEntropyLoss(weight=w)

    def extract_and_norm(self, x, fit=False):
        # x: (B, C, T)
        f = self.hydra(x)
        if fit and (not self.features_fitted):
            with torch.no_grad():
                self.f_mean.copy_(f.mean(dim=0))
                self.f_std.copy_(f.std(dim=0) + 1e-8)
            self.features_fitted = True
        f = (f - self.f_mean) / self.f_std
        return f

    def forward(self, x, fit_norm=False):
        # input: (B, T, C) -> (B, C, T)
        x = x.transpose(1, 2)
        f = self.extract_and_norm(x, fit=fit_norm)
        return self.classifier(f)

# =============================================================================
# infer shapes/classes + class labels + class weights
# =============================================================================
train_loader = dm.train_dataloader()
val_loader = dm.val_dataloader()
test_loader = dm.test_dataloader()

x0, _, y0 = next(iter(train_loader))
seq_len = int(x0.shape[1])
n_channels = int(x0.shape[2])

# Robust n_classes
label_map = getattr(getattr(dm, "train_dataset", None), "label_map", None)
if isinstance(label_map, dict) and len(label_map) > 0:
    n_classes = int(max(int(v) for v in label_map.values()) + 1)
else:
    n_classes = int(y0.max().item() + 1)

def build_class_labels(dm, n_classes_local: int):
    td = dm.train_dataset
    base = getattr(td, "base", td)
    lm = getattr(base, "label_map", getattr(td, "label_map", None))

    labels = [None] * n_classes_local
    neg_id = None
    if "SELECTED_CLASSES" in globals() and "ADD_NEGATIVE_CLASS" in globals() and ADD_NEGATIVE_CLASS:
        try:
            neg_id = len(SELECTED_CLASSES)
        except Exception:
            neg_id = None

    if isinstance(lm, dict):
        for name, cid in lm.items():
            try:
                cid = int(cid)
            except Exception:
                continue
            if 0 <= cid < n_classes_local and labels[cid] is None:
                labels[cid] = "Negative" if (neg_id is not None and cid == neg_id) else str(name)

    for i in range(n_classes_local):
        if labels[i] is None:
            labels[i] = "Negative" if (neg_id is not None and i == neg_id) else f"Class {i}"
    return labels

class_labels = build_class_labels(dm, n_classes)

# Class weights: inverse frequency over train label list (robust)
train_labels = getattr(dm.train_dataset, "get_label_list", None)
if callable(train_labels):
    train_labels = dm.train_dataset.get_label_list()
else:
    # fallback from a pass over train_loader
    tmp = []
    for _, _, yb in train_loader:
        tmp.append(yb.detach().cpu().numpy())
    train_labels = np.concatenate(tmp) if tmp else np.array([], dtype=int)

cnt = pd.Series(train_labels).value_counts().reindex(range(n_classes), fill_value=0)
class_weights = (cnt.sum() / (len(cnt) * cnt.clip(lower=1))).tolist()

print(f"Sequence length: {seq_len}, Channels: {n_channels}, Classes: {n_classes}")
print("Class labels:", class_labels)

save_json("run_context.json", {"run_id": RUN_ID, "outputs_root": OUTPUTS, "run_dir": RUN_DIR, "device": str(device)})
save_json("config.json", {
    "SEED": SEED,
    "BATCH_SIZE": BATCH_SIZE,
    "MAX_EPOCHS": MAX_EPOCHS,
    "PATIENCE": PATIENCE,
    "LR": LR,
    "WEIGHT_DECAY": WEIGHT_DECAY,
    "K": K,
    "G": G,
    "HIDDEN": HIDDEN,
    "DROPOUT_1": DROPOUT_1,
    "DROPOUT_2": DROPOUT_2,
    "GAMMA": GAMMA,
    "CLIP_DEFAULT_K": CLIP_DEFAULT_K,
    "CLIP_K_MAP": CLIP_K_MAP,
    "seq_len": seq_len,
    "n_channels": n_channels,
    "n_classes": n_classes,
})
save_json("class_labels.json", {"class_labels": class_labels})
save_json("train_class_counts.json", {str(i): int(v) for i, v in cnt.items()})
save_json("train_class_weights.json", {"weights": [float(w) for w in class_weights]})

# =============================================================================
# Build model + optimizer + scheduler
# =============================================================================
model = HydraNet(
    input_length=seq_len,
    num_channels=n_channels,
    num_classes=n_classes,
    k=K,
    g=G,
    class_weights=class_weights,
    dropout_1=DROPOUT_1,
    dropout_2=DROPOUT_2,
    hidden=HIDDEN,
).to(device)

n_total = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

param_text = (
    "PARAMETER STATISTICS\n"
    f"Total params: {n_total:,}\n"
    f"Trainable params: {n_trainable:,}\n"
)
print(param_text)
save_text("parameter_stats.txt", param_text)

optimizer = torch.optim.AdamW(model.parameters(), lr=float(LR), weight_decay=float(WEIGHT_DECAY))
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=float(GAMMA))

BEST_CKPT_PATH = rpath("best_hydra.pt")

# =============================================================================
# STANDARDIZED CONFUSION MATRIX PLOTTING (FROM TRANSFORMER)
# =============================================================================
def plot_conf_mat_norm01(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str,
    annot: bool = True,
    annot_fmt: str = ".2f",
    dpi: int = 200,
):
    """Plot row-normalized confusion matrix (0-1 scale)"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    row_sums = cm_counts.sum(axis=1, keepdims=True)
    cm_norm = cm_counts / np.maximum(row_sums, 1)
    cm_norm = np.nan_to_num(cm_norm, nan=0.0, posinf=0.0, neginf=0.0)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_norm,
        annot=annot,
        fmt=annot_fmt,
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Proportion (0–1)"},
        vmin=0.0,
        vmax=1.0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_norm01.npy"), cm_norm.astype(np.float32))
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)

    return cm_norm, cm_counts

def plot_conf_mat_counts(
    y_true,
    y_pred,
    title: str,
    class_names: list[str],
    out_png: str,
    cmap: str = "Blues",
    annot: bool = True,
    dpi: int = 200,
    max_annot: int = 9999,
):
    """Plot count-based confusion matrix"""
    labels = np.arange(len(class_names))
    cm_counts = confusion_matrix(y_true, y_pred, labels=labels).astype(int)

    plt.figure(figsize=(max(10, 0.55 * len(class_names)), max(8, 0.45 * len(class_names))))
    ax = sns.heatmap(
        cm_counts,
        annot=annot,
        fmt="d",
        cmap=cmap,
        xticklabels=class_names,
        yticklabels=class_names,
        cbar_kws={"label": "Count"},
        vmin=0,
    )
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(title, pad=12)
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)

    if annot and max_annot is not None:
        for t in ax.texts:
            try:
                v = int(t.get_text())
                if v > int(max_annot):
                    t.set_text(f">{max_annot}")
            except Exception:
                pass

    save_fig(out_png, dpi=dpi)
    save_npy(out_png.replace(".png", "_counts.npy"), cm_counts)
    return cm_counts

# =============================================================================
# COMPREHENSIVE METRICS HELPERS (FROM TRANSFORMER)
# =============================================================================
def prf_all_averages(y_true, y_pred, zero_division=0):
    """Compute precision, recall, F1 for macro, micro, weighted"""
    out = {}
    for avg in ["macro", "micro", "weighted"]:
        p, r, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average=avg, zero_division=zero_division
        )
        out[avg] = {"precision": float(p), "recall": float(r), "f1": float(f1)}
    return out

def per_class_f1(y_true, y_pred, n_classes_local: int, zero_division=0):
    """Compute per-class precision, recall, F1, and support"""
    labels = np.arange(int(n_classes_local))
    p, r, f1, s = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, average=None, zero_division=zero_division
    )
    return {
        "precision": [float(v) for v in p],
        "recall":    [float(v) for v in r],
        "f1":        [float(v) for v in f1],
        "support":   [int(v) for v in s],
    }

def _cuda_sync_if_needed(dev: torch.device):
    """Synchronize CUDA if needed for accurate timing"""
    if str(dev).startswith("cuda") and torch.cuda.is_available():
        torch.cuda.synchronize()

# =============================================================================
# ENHANCED EVALUATION WITH TIMING + COMPREHENSIVE METRICS
# =============================================================================
@torch.no_grad()
def evaluate_comprehensive(loader, desc="Eval", warmup_batches=5):
    """
    Comprehensive evaluation with timing and all metrics.
    Returns: metrics dict, y_true, y_pred
    """
    model.eval()
    total_loss = 0.0
    preds_all, targs_all = [], []
    n_seen = 0
    
    times = []
    did_sync_after_warmup = False
    
    for bi, (bx, _, by) in enumerate(tqdm(loader, desc=desc, leave=False)):
        bx, by = bx.to(device), by.to(device)
        
        do_time = (bi >= int(warmup_batches))
        
        if do_time and (not did_sync_after_warmup):
            _cuda_sync_if_needed(device)
            did_sync_after_warmup = True
        
        if do_time:
            t0 = time.perf_counter()
            logits = model(bx, fit_norm=False)
            _cuda_sync_if_needed(device)
            t1 = time.perf_counter()
            times.append((t1 - t0))
        else:
            logits = model(bx, fit_norm=False)
        
        loss = model.ce(logits, by)
        total_loss += float(loss.item()) * bx.size(0)
        
        preds = torch.argmax(logits, dim=1).detach().cpu()
        targs_all.append(by.detach().cpu())
        preds_all.append(preds)
        n_seen += int(by.size(0))
    
    y_pred = torch.cat(preds_all).numpy() if preds_all else np.array([], dtype=int)
    y_true = torch.cat(targs_all).numpy() if targs_all else np.array([], dtype=int)
    
    loss_avg = total_loss / max(1, n_seen)
    
    if y_true.size:
        acc = float((y_pred == y_true).mean())
        prf = prf_all_averages(y_true, y_pred, zero_division=0)
        per_cls = per_class_f1(y_true, y_pred, n_classes_local=int(n_classes), zero_division=0)
    else:
        acc = float("nan")
        prf = {
            "macro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "micro": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
            "weighted": {"precision": float("nan"), "recall": float("nan"), "f1": float("nan")},
        }
        per_cls = {"precision": [], "recall": [], "f1": [], "support": []}
    
    # Timing stats
    if len(times) > 0 and n_seen > 0:
        total_t = float(np.sum(times))
        ms_per_batch = float(1000.0 * np.mean(times))
        ms_per_sample = float(1000.0 * (total_t / float(n_seen)))
        n_timed_batches = int(len(times))
    else:
        total_t = 0.0
        ms_per_batch = float("nan")
        ms_per_sample = float("nan")
        n_timed_batches = 0
    
    timing = {
        "warmup_batches": int(warmup_batches),
        "timed_batches": int(n_timed_batches),
        "timed_total_seconds": float(total_t),
        "ms_per_batch": float(ms_per_batch),
        "ms_per_sample": float(ms_per_sample),
    }
    
    metrics = {
        "loss": float(loss_avg),
        "acc": float(acc),
        "prf": prf,
        "per_class": per_cls,
        "timing": timing,
    }
    
    return metrics, y_true, y_pred

def early_stop_classwise(seq, k_map=None, default_k=3):
    """Early-stop aggregation with class-wise k values"""
    if k_map is None:
        k_map = {}
    run_val = None
    run_len = 0
    for v in seq:
        v = int(v)
        if v == run_val:
            run_len += 1
        else:
            run_val = v
            run_len = 1
        k_c = int(k_map.get(v, default_k))
        if run_len >= k_c:
            return v
    return None

def get_clip_ids_for_windows(ds):
    if hasattr(ds, "_index"):
        return [pair[0] for pair in ds._index]
    return list(range(len(ds)))

def get_clip_targets(base_ds, clip_to_targs, n_clips_local):
    targets = []
    for cid in range(n_clips_local):
        if hasattr(base_ds, "samples") and hasattr(base_ds, "label_map"):
            clean = re.sub(r"\(\d+\)$", "", base_ds.samples[cid]["label"])
            targets.append(int(base_ds.label_map[clean]))
        else:
            tgts = clip_to_targs.get(cid, [])
            targets.append(int(max(set(tgts), key=tgts.count)) if tgts else 0)
    return np.array(targets, dtype=int)

# =============================================================================
# TRAINING LOOP WITH COMPREHENSIVE METRICS
# =============================================================================
history = {
    "epoch": [],
    "train_loss": [],
    "val_loss": [],
    "train_acc": [],
    "val_acc": [],
    "train_f1_macro": [],
    "train_f1_micro": [],
    "train_f1_weighted": [],
    "val_f1_macro": [],
    "val_f1_micro": [],
    "val_f1_weighted": [],
    "train_precision_macro": [],
    "train_recall_macro": [],
    "val_precision_macro": [],
    "val_recall_macro": [],
    "lr": [],
}

best_val_f1 = None
best_state = None
no_improve = 0
features_fit_done = False

print("\n" + "=" * 60)
print("STARTING TRAINING")
print("=" * 60 + "\n")

for epoch in range(1, int(MAX_EPOCHS) + 1):
    model.train()
    total_loss = 0.0
    all_preds, all_targs = [], []

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{int(MAX_EPOCHS)}", leave=False)
    for bx, _, by in pbar:
        bx, by = bx.to(device), by.to(device)
        optimizer.zero_grad(set_to_none=True)

        # Fit Hydra feature normalization once (first batch of first epoch)
        logits = model(bx, fit_norm=(not features_fit_done))
        features_fit_done = True

        loss = model.ce(logits, by)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += float(loss.item()) * bx.size(0)
        pred = logits.argmax(1)
        all_preds.append(pred.detach().cpu().numpy())
        all_targs.append(by.detach().cpu().numpy())
        pbar.set_postfix(loss=float(loss.item()))

    scheduler.step()
    lr_now = float(optimizer.param_groups[0]["lr"])

    # Train metrics
    y_train_pred = np.concatenate(all_preds) if all_preds else np.array([], dtype=int)
    y_train_true = np.concatenate(all_targs) if all_targs else np.array([], dtype=int)

    train_loss = total_loss / max(1, len(train_loader.dataset))
    train_acc = float((y_train_pred == y_train_true).mean())
    train_prf = prf_all_averages(y_train_true, y_train_pred, zero_division=0)

    # Val metrics (comprehensive)
    val_metrics, _, _ = evaluate_comprehensive(val_loader, desc="Val", warmup_batches=5)

    # Record history
    history["epoch"].append(int(epoch))
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_metrics["loss"]))
    history["train_acc"].append(float(train_acc))
    history["val_acc"].append(float(val_metrics["acc"]))
    
    history["train_f1_macro"].append(float(train_prf["macro"]["f1"]))
    history["train_f1_micro"].append(float(train_prf["micro"]["f1"]))
    history["train_f1_weighted"].append(float(train_prf["weighted"]["f1"]))
    history["val_f1_macro"].append(float(val_metrics["prf"]["macro"]["f1"]))
    history["val_f1_micro"].append(float(val_metrics["prf"]["micro"]["f1"]))
    history["val_f1_weighted"].append(float(val_metrics["prf"]["weighted"]["f1"]))
    
    history["train_precision_macro"].append(float(train_prf["macro"]["precision"]))
    history["train_recall_macro"].append(float(train_prf["macro"]["recall"]))
    history["val_precision_macro"].append(float(val_metrics["prf"]["macro"]["precision"]))
    history["val_recall_macro"].append(float(val_metrics["prf"]["macro"]["recall"]))
    
    history["lr"].append(float(lr_now))

    print(
        f"Epoch {epoch:03d} | lr={lr_now:.3e} | "
        f"train_loss={train_loss:.4f} val_loss={val_metrics['loss']:.4f} | "
        f"train_acc={train_acc:.4f} val_acc={val_metrics['acc']:.4f} | "
        f"train_f1={train_prf['macro']['f1']:.4f} val_f1={val_metrics['prf']['macro']['f1']:.4f}"
    )

    # persist each epoch
    save_csv("hydra_metrics.csv", pd.DataFrame(history), index=False)
    save_json("history.json", history)

    # early stopping on val macro-f1
    val_f1 = val_metrics["prf"]["macro"]["f1"]
    improved = (best_val_f1 is None) or (val_f1 > best_val_f1 + 1e-4)
    if improved:
        best_val_f1 = float(val_f1)
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, BEST_CKPT_PATH)
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= int(PATIENCE):
            print(f"Early stopping at epoch {epoch} (no val_f1 improvement for {int(PATIENCE)} epochs).")
            break

# Load best checkpoint
if os.path.isfile(BEST_CKPT_PATH):
    model.load_state_dict(torch.load(BEST_CKPT_PATH, map_location=device))
    model.to(device).eval()

save_text("best_checkpoint_path.txt", BEST_CKPT_PATH)

# =============================================================================
# TRAINING CURVES (Loss, Accuracy, F1, LR)
# =============================================================================
print("\n" + "=" * 60)
print("GENERATING TRAINING CURVES")
print("=" * 60)

epochs = history["epoch"]

# Loss curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_loss"], label="Train loss")
plt.plot(epochs, history["val_loss"], label="Val loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Train vs Validation Loss")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_loss_train_val.png", dpi=300)

# Accuracy curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_acc"], label="Train accuracy")
plt.plot(epochs, history["val_acc"], label="Val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Train vs Validation Accuracy")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_acc_train_val.png", dpi=300)

# F1 curve (macro)
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["train_f1_macro"], label="Train F1 (macro)")
plt.plot(epochs, history["val_f1_macro"], label="Val F1 (macro)")
plt.xlabel("Epoch")
plt.ylabel("F1")
plt.title("Train vs Validation F1 (macro)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_f1_train_val.png", dpi=300)

# LR curve
plt.figure(figsize=(9, 5))
plt.plot(epochs, history["lr"], label="Learning rate")
plt.xlabel("Epoch")
plt.ylabel("LR")
plt.title("Learning Rate Schedule (ExponentialLR)")
plt.grid(True, alpha=0.3)
plt.legend()
save_fig("curve_lr.png", dpi=300)

# =============================================================================
# WINDOW-LEVEL EVALUATION (VAL + TEST) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("WINDOW-LEVEL EVALUATION")
print("=" * 60)

val_metrics, y_val_true, y_val_pred = evaluate_comprehensive(val_loader, desc="Val-final", warmup_batches=5)
test_metrics, y_test_true, y_test_pred = evaluate_comprehensive(test_loader, desc="Test", warmup_batches=5)

save_json(
    "metrics_window.json",
    {"val": val_metrics, "test": test_metrics, "best_ckpt": str(BEST_CKPT_PATH)},
)

# Detailed summary with all metrics
save_text(
    "metrics_summary_window.txt",
    "WINDOW-LEVEL METRICS\n"
    f"VAL  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} ms/sample={val_metrics['timing']['ms_per_sample']:.4f}\n"
    f"VAL  macro    P/R/F1={val_metrics['prf']['macro']['precision']:.4f}/{val_metrics['prf']['macro']['recall']:.4f}/{val_metrics['prf']['macro']['f1']:.4f}\n"
    f"VAL  micro    P/R/F1={val_metrics['prf']['micro']['precision']:.4f}/{val_metrics['prf']['micro']['recall']:.4f}/{val_metrics['prf']['micro']['f1']:.4f}\n"
    f"VAL  weighted P/R/F1={val_metrics['prf']['weighted']['precision']:.4f}/{val_metrics['prf']['weighted']['recall']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} ms/sample={test_metrics['timing']['ms_per_sample']:.4f}\n"
    f"TEST macro    P/R/F1={test_metrics['prf']['macro']['precision']:.4f}/{test_metrics['prf']['macro']['recall']:.4f}/{test_metrics['prf']['macro']['f1']:.4f}\n"
    f"TEST micro    P/R/F1={test_metrics['prf']['micro']['precision']:.4f}/{test_metrics['prf']['micro']['recall']:.4f}/{test_metrics['prf']['micro']['f1']:.4f}\n"
    f"TEST weighted P/R/F1={test_metrics['prf']['weighted']['precision']:.4f}/{test_metrics['prf']['weighted']['recall']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
)

save_json(
    "val_test_support.json",
    {
        "val_support": np.bincount(y_val_true, minlength=int(n_classes)).tolist(),
        "test_support": np.bincount(y_test_true, minlength=int(n_classes)).tolist(),
    },
)

# Confusion matrices - WINDOW LEVEL (Blues for window, as per Transformer)
_ = plot_conf_mat_norm01(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_val_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_norm01(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, normalized 0-1)",
    class_names=class_labels,
    out_png="cm_test_window.png",
    cmap="Blues",
    annot=True,
    annot_fmt=".2f",
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_val_true, y_val_pred,
    title="Confusion Matrix - VAL (window, counts)",
    class_names=class_labels,
    out_png="cm_val_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

_ = plot_conf_mat_counts(
    y_test_true, y_test_pred,
    title="Confusion Matrix - TEST (window, counts)",
    class_names=class_labels,
    out_png="cm_test_window_counts.png",
    cmap="Blues",
    annot=True,
    dpi=200,
)

# Classification reports
labels = np.arange(n_classes)
val_report = classification_report(y_val_true, y_val_pred, labels=labels, target_names=class_labels, digits=3, zero_division=0)
test_report = classification_report(y_test_true, y_test_pred, labels=labels, target_names=class_labels, digits=3, zero_division=0)
save_text("report_val_window.txt", val_report)
save_text("report_test_window.txt", test_report)

# =============================================================================
# CLIP-LEVEL EVALUATION (EARLY-STOP) + COMPREHENSIVE METRICS + CONFUSION MATRICES
# =============================================================================
print("\n" + "=" * 60)
print("CLIP-LEVEL EVALUATION")
print("=" * 60)

test_ds = dm.test_dataset
base_ds = getattr(test_ds, "base", test_ds)
n_clips = len(base_ds)

clip_ids_for_windows = get_clip_ids_for_windows(test_ds)

clip_metrics = None
clip_targets = None
clip_preds = None

if len(clip_ids_for_windows) != len(y_test_true):
    save_text(
        "clip_eval_note.txt",
        f"Skipping clip-level: clip_ids_for_windows={len(clip_ids_for_windows)} != n_windows={len(y_test_true)}"
    )
else:
    clip_to_preds = defaultdict(list)
    clip_to_targs = defaultdict(list)
    for cid, p, t in zip(clip_ids_for_windows, y_test_pred, y_test_true):
        clip_to_preds[int(cid)].append(int(p))
        clip_to_targs[int(cid)].append(int(t))

    clip_targets = get_clip_targets(base_ds, clip_to_targs, n_clips)

    # Get clip predictions with early-stop
    k_map = CLIP_K_MAP if isinstance(CLIP_K_MAP, dict) else {}
    clip_preds_list = []
    missed_early = 0
    for cid in range(n_clips):
        seq = clip_to_preds.get(cid, [])
        chosen = early_stop_classwise(seq, k_map=k_map, default_k=int(CLIP_DEFAULT_K))
        if chosen is None:
            missed_early += 1
            chosen = int(max(set(seq), key=seq.count) if seq else clip_targets[cid])
        clip_preds_list.append(int(chosen))
    clip_preds = np.array(clip_preds_list, dtype=int)

    # Compute comprehensive clip metrics
    clip_acc = float((clip_preds == clip_targets).mean()) if clip_targets.size else float("nan")
    clip_prf = prf_all_averages(clip_targets, clip_preds, zero_division=0) if clip_targets.size else None
    clip_per_cls = per_class_f1(clip_targets, clip_preds, n_classes_local=int(n_classes), zero_division=0) if clip_targets.size else None

    # Timing for aggregation (minimal)
    t0 = time.perf_counter()
    _ = clip_preds
    t1 = time.perf_counter()
    clip_agg_ms_total = float(1000.0 * (t1 - t0))
    clip_agg_ms_per_clip = float(clip_agg_ms_total / max(1, int(n_clips)))

    clip_metrics = {
        "acc": float(clip_acc),
        "prf": clip_prf,
        "per_class": clip_per_cls,
        "timing": {
            "window_forward_ms_per_sample": float(test_metrics["timing"]["ms_per_sample"]),
            "aggregation_ms_total": float(clip_agg_ms_total),
            "aggregation_ms_per_clip": float(clip_agg_ms_per_clip),
        },
        "early_stop": {
            "default_k": int(CLIP_DEFAULT_K),
            "k_map_size": int(len(k_map)),
            "missed_early": int(missed_early),
            "n_clips": int(n_clips),
        },
    }
    save_json("metrics_clip.json", clip_metrics)

    # Detailed summary
    save_text(
        "metrics_summary_clip.txt",
        "CLIP-LEVEL METRICS\n"
        f"TEST acc={clip_acc:.4f}\n"
        f"TEST macro    P/R/F1={clip_prf['macro']['precision']:.4f}/{clip_prf['macro']['recall']:.4f}/{clip_prf['macro']['f1']:.4f}\n"
        f"TEST micro    P/R/F1={clip_prf['micro']['precision']:.4f}/{clip_prf['micro']['recall']:.4f}/{clip_prf['micro']['f1']:.4f}\n"
        f"TEST weighted P/R/F1={clip_prf['weighted']['precision']:.4f}/{clip_prf['weighted']['recall']:.4f}/{clip_prf['weighted']['f1']:.4f}\n"
        f"Forward ms/sample (from window eval)={test_metrics['timing']['ms_per_sample']:.4f}\n"
        f"Aggregation ms/clip={clip_agg_ms_per_clip:.4f}\n"
        f"Early-stop default_k={int(CLIP_DEFAULT_K)} k_map_size={len(k_map)} majority_fallback={missed_early}/{n_clips}\n"
    )

    # Confusion matrices - CLIP LEVEL (Greens for clip/aggregation, as per Transformer)
    _ = plot_conf_mat_norm01(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, normalized 0-1)",
        class_names=class_labels,
        out_png="cm_test_clip_norm01.png",
        cmap="Greens",
        annot=True,
        annot_fmt=".2f",
        dpi=200,
    )

    _ = plot_conf_mat_counts(
        clip_targets, clip_preds,
        title="Confusion Matrix - TEST (clip, counts)",
        class_names=class_labels,
        out_png="cm_test_clip_counts.png",
        cmap="Greens",
        annot=True,
        dpi=200,
    )

    # Classification report
    clip_report = classification_report(clip_targets, clip_preds, labels=labels, target_names=class_labels, digits=3, zero_division=0)
    save_text("report_test_clip.txt", clip_report)

# =============================================================================
# FINAL SUMMARY
# =============================================================================
final_summary = (
    "FINAL SUMMARY\n"
    f"Params total: {n_total:,}\n"
    f"Best checkpoint: {BEST_CKPT_PATH}\n"
    f"VAL window:  loss={val_metrics['loss']:.4f} acc={val_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={val_metrics['prf']['macro']['f1']:.4f}/"
    f"{val_metrics['prf']['micro']['f1']:.4f}/{val_metrics['prf']['weighted']['f1']:.4f}\n"
    f"TEST window: loss={test_metrics['loss']:.4f} acc={test_metrics['acc']:.4f} "
    f"F1(macro/micro/weighted)={test_metrics['prf']['macro']['f1']:.4f}/"
    f"{test_metrics['prf']['micro']['f1']:.4f}/{test_metrics['prf']['weighted']['f1']:.4f}\n"
    + (
        f"TEST clip:   acc={clip_metrics['acc']:.4f} "
        f"F1(macro/micro/weighted)={clip_metrics['prf']['macro']['f1']:.4f}/"
        f"{clip_metrics['prf']['micro']['f1']:.4f}/{clip_metrics['prf']['weighted']['f1']:.4f}\n"
        if clip_metrics is not None and clip_metrics.get("prf") is not None else
        "TEST clip:   SKIPPED\n"
    ) +
    f"Artifacts:   {RUN_DIR}\n"
)
print(final_summary)
save_text("final_summary.txt", final_summary)

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("All artifacts saved under:", RUN_DIR)

## Benchmark

In [ ]:
# ==== COMPREHENSIVE MODEL BENCHMARK COMPARISON (FIXED) ====

import os
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional
import re

# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS TO YOUR OUTPUT DIRECTORY
# =============================================================================
OUTPUTS_ROOT = OUTPUTS  # Your main outputs directory

# Model type to subdirectory mapping
MODEL_DIRS = {
    "Transformer": "transformer",
    "Apple": "apple", 
    "InceptionTime": "inceptiontime",
    "Hydra": "hydra",
    "NormWear-Base": "normwear_base",
    "NormWear-LoRA": "normwear_lora",
}

# Output directory for benchmark results
BENCHMARK_DIR = os.path.join(OUTPUTS_ROOT, "benchmark_comparison")
Path(BENCHMARK_DIR).mkdir(parents=True, exist_ok=True)

print(f"Benchmark output directory: {BENCHMARK_DIR}\n")


# =============================================================================
# HELPER FUNCTIONS TO FIND LATEST RUN AND LOAD METRICS
# =============================================================================

def find_latest_run_dir(model_subdir: str, outputs_root: str) -> Optional[str]:
    """Find the most recent run_* directory for a given model type."""
    model_path = os.path.join(outputs_root, model_subdir)
    if not os.path.exists(model_path):
        print(f"⚠️  Warning: {model_path} does not exist")
        return None
    
    # Look for run_* directories
    run_dirs = glob.glob(os.path.join(model_path, "run_*"))
    if not run_dirs:
        print(f"⚠️  Warning: No run_* directories found in {model_path}")
        return None
    
    # Sort by timestamp (assuming format run_YYYYMMDD-HHMMSS or run_YYYYMMDD_HHMMSS)
    run_dirs.sort(reverse=True)
    latest = run_dirs[0]
    print(f"✓ Found latest run for {model_subdir}: {os.path.basename(latest)}")
    return latest


def safe_load_json(filepath: str, default: Any = None) -> Any:
    """Safely load JSON file, return default if not found."""
    if not os.path.exists(filepath):
        return default
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  Error loading {filepath}: {e}")
        return default


def safe_get_nested(data: Dict, *keys, default=None) -> Any:
    """Safely get nested dictionary value."""
    if data is None:
        return default
    current = data
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        else:
            return default
        if current is None:
            return default
    return current


def extract_model_metrics(run_dir: str, model_name: str) -> Dict[str, Any]:
    """Extract all relevant metrics from a model's run directory."""
    
    metrics = {
        "model": model_name,
        "run_dir": os.path.basename(run_dir),
    }
    
    # 1. Parameter statistics
    param_summary = safe_load_json(os.path.join(run_dir, "parameter_summary.json"))
    if param_summary:
        metrics["total_params"] = param_summary.get("model_total_params", np.nan)
        metrics["trainable_params"] = param_summary.get("model_trainable_params", np.nan)
        metrics["trainable_pct"] = param_summary.get("trainable_percent", np.nan)
    else:
        # Try parsing parameter_stats.txt
        param_file = os.path.join(run_dir, "parameter_stats.txt")
        if os.path.exists(param_file):
            with open(param_file, 'r', encoding='utf-8') as f:
                content = f.read()
                # Extract numbers using regex
                total_match = re.search(r"[Tt]otal params?:\s*([\d,]+)", content)
                trainable_match = re.search(r"[Tt]rainable params?:\s*([\d,]+)", content)
                if total_match:
                    metrics["total_params"] = int(total_match.group(1).replace(",", ""))
                if trainable_match:
                    metrics["trainable_params"] = int(trainable_match.group(1).replace(",", ""))
    
    # 2. Window-level metrics (VAL + TEST)
    window_metrics = safe_load_json(os.path.join(run_dir, "metrics_window.json"))
    
    if window_metrics:
        # VAL metrics
        val_data = window_metrics.get("val", {})
        metrics["val_loss"] = safe_get_nested(val_data, "loss", default=np.nan)
        metrics["val_acc"] = safe_get_nested(val_data, "acc", default=np.nan)
        
        val_prf = safe_get_nested(val_data, "prf", default={})
        metrics["val_p_macro"] = safe_get_nested(val_prf, "macro", "precision", default=np.nan)
        metrics["val_r_macro"] = safe_get_nested(val_prf, "macro", "recall", default=np.nan)
        metrics["val_f1_macro"] = safe_get_nested(val_prf, "macro", "f1", default=np.nan)
        metrics["val_p_micro"] = safe_get_nested(val_prf, "micro", "precision", default=np.nan)
        metrics["val_r_micro"] = safe_get_nested(val_prf, "micro", "recall", default=np.nan)
        metrics["val_f1_micro"] = safe_get_nested(val_prf, "micro", "f1", default=np.nan)
        metrics["val_p_weighted"] = safe_get_nested(val_prf, "weighted", "precision", default=np.nan)
        metrics["val_r_weighted"] = safe_get_nested(val_prf, "weighted", "recall", default=np.nan)
        metrics["val_f1_weighted"] = safe_get_nested(val_prf, "weighted", "f1", default=np.nan)
        
        # TEST metrics
        test_data = window_metrics.get("test", {})
        metrics["test_loss"] = safe_get_nested(test_data, "loss", default=np.nan)
        metrics["test_acc"] = safe_get_nested(test_data, "acc", default=np.nan)
        
        test_prf = safe_get_nested(test_data, "prf", default={})
        metrics["test_p_macro"] = safe_get_nested(test_prf, "macro", "precision", default=np.nan)
        metrics["test_r_macro"] = safe_get_nested(test_prf, "macro", "recall", default=np.nan)
        metrics["test_f1_macro"] = safe_get_nested(test_prf, "macro", "f1", default=np.nan)
        metrics["test_p_micro"] = safe_get_nested(test_prf, "micro", "precision", default=np.nan)
        metrics["test_r_micro"] = safe_get_nested(test_prf, "micro", "recall", default=np.nan)
        metrics["test_f1_micro"] = safe_get_nested(test_prf, "micro", "f1", default=np.nan)
        metrics["test_p_weighted"] = safe_get_nested(test_prf, "weighted", "precision", default=np.nan)
        metrics["test_r_weighted"] = safe_get_nested(test_prf, "weighted", "recall", default=np.nan)
        metrics["test_f1_weighted"] = safe_get_nested(test_prf, "weighted", "f1", default=np.nan)
        
        # Timing (window-level)
        test_timing = safe_get_nested(test_data, "timing", default={})
        metrics["test_ms_per_sample"] = safe_get_nested(test_timing, "ms_per_sample", default=np.nan)
        metrics["test_ms_per_batch"] = safe_get_nested(test_timing, "ms_per_batch", default=np.nan)
    
    # 3. Clip-level metrics (TEST only)
    clip_metrics = safe_load_json(os.path.join(run_dir, "metrics_clip.json"))
    
    if clip_metrics:
        clip_data = clip_metrics.get("test", clip_metrics)  # Handle both formats
        metrics["clip_acc"] = safe_get_nested(clip_data, "acc", default=np.nan)
        
        clip_prf = safe_get_nested(clip_data, "prf", default={})
        metrics["clip_p_macro"] = safe_get_nested(clip_prf, "macro", "precision", default=np.nan)
        metrics["clip_r_macro"] = safe_get_nested(clip_prf, "macro", "recall", default=np.nan)
        metrics["clip_f1_macro"] = safe_get_nested(clip_prf, "macro", "f1", default=np.nan)
        metrics["clip_p_micro"] = safe_get_nested(clip_prf, "micro", "precision", default=np.nan)
        metrics["clip_r_micro"] = safe_get_nested(clip_prf, "micro", "recall", default=np.nan)
        metrics["clip_f1_micro"] = safe_get_nested(clip_prf, "micro", "f1", default=np.nan)
        metrics["clip_p_weighted"] = safe_get_nested(clip_prf, "weighted", "precision", default=np.nan)
        metrics["clip_r_weighted"] = safe_get_nested(clip_prf, "weighted", "recall", default=np.nan)
        metrics["clip_f1_weighted"] = safe_get_nested(clip_prf, "weighted", "f1", default=np.nan)
        
        # Clip timing
        clip_timing = safe_get_nested(clip_metrics, "timing", default={})
        metrics["clip_ms_per_clip"] = safe_get_nested(clip_timing, "aggregation_ms_per_clip", default=np.nan)
        
        # Early-stop info
        early_stop = safe_get_nested(clip_metrics, "early_stop", default={})
        metrics["clip_default_k"] = safe_get_nested(early_stop, "default_k", default=np.nan)
        metrics["clip_missed_early"] = safe_get_nested(early_stop, "missed_early", default=np.nan)
        metrics["clip_n_clips"] = safe_get_nested(early_stop, "n_clips", default=np.nan)
    
    # 4. Inference benchmark (for NormWear models)
    benchmark = safe_load_json(os.path.join(run_dir, "inference_benchmark_window.json"))
    if benchmark:
        metrics["benchmark_median_window_ms"] = safe_get_nested(benchmark, "median_per_window_ms", default=np.nan)
        metrics["benchmark_throughput"] = safe_get_nested(benchmark, "throughput_windows_per_s", default=np.nan)
    
    return metrics


# =============================================================================
# COLLECT ALL MODEL METRICS
# =============================================================================

print("=" * 80)
print("COLLECTING METRICS FROM ALL MODELS")
print("=" * 80 + "\n")

all_metrics = []

for model_name, model_subdir in MODEL_DIRS.items():
    print(f"\n📊 Processing: {model_name}")
    print("-" * 60)
    
    run_dir = find_latest_run_dir(model_subdir, OUTPUTS_ROOT)
    
    if run_dir is None:
        print(f"❌ Skipping {model_name} (no valid run directory found)\n")
        continue
    
    metrics = extract_model_metrics(run_dir, model_name)
    all_metrics.append(metrics)
    
    print(f"✓ Extracted {len(metrics)} metric fields")
    print()

if not all_metrics:
    print("❌ No metrics found! Please check your MODEL_DIRS configuration.")
    exit(1)

# Create DataFrame
df = pd.DataFrame(all_metrics)

# Reorder columns for better readability
column_order = [
    "model",
    "run_dir",
    # Parameters
    "total_params",
    "trainable_params",
    "trainable_pct",
    # VAL metrics
    "val_loss",
    "val_acc",
    "val_f1_macro",
    "val_f1_micro",
    "val_f1_weighted",
    "val_p_macro",
    "val_r_macro",
    "val_p_micro",
    "val_r_micro",
    "val_p_weighted",
    "val_r_weighted",
    # TEST Window metrics
    "test_loss",
    "test_acc",
    "test_f1_macro",
    "test_f1_micro",
    "test_f1_weighted",
    "test_p_macro",
    "test_r_macro",
    "test_p_micro",
    "test_r_micro",
    "test_p_weighted",
    "test_r_weighted",
    "test_ms_per_sample",
    "test_ms_per_batch",
    # TEST Clip metrics
    "clip_acc",
    "clip_f1_macro",
    "clip_f1_micro",
    "clip_f1_weighted",
    "clip_p_macro",
    "clip_r_macro",
    "clip_p_micro",
    "clip_r_micro",
    "clip_p_weighted",
    "clip_r_weighted",
    "clip_ms_per_clip",
    "clip_default_k",
    "clip_missed_early",
    "clip_n_clips",
    # Benchmark
    "benchmark_median_window_ms",
    "benchmark_throughput",
]

# Only include columns that exist
column_order = [col for col in column_order if col in df.columns]
df = df[column_order]

print("\n" + "=" * 80)
print("METRICS COLLECTION COMPLETE")
print("=" * 80)
print(f"\n✓ Collected metrics from {len(df)} models")
print(f"✓ Total metrics per model: {len(df.columns)}")


# =============================================================================
# SAVE FULL TABLE (CSV + EXCEL)
# =============================================================================

csv_path = os.path.join(BENCHMARK_DIR, "full_benchmark_table.csv")
df.to_csv(csv_path, index=False, float_format="%.6f")
print(f"\n✓ Saved full table: {csv_path}")

try:
    excel_path = os.path.join(BENCHMARK_DIR, "full_benchmark_table.xlsx")
    df.to_excel(excel_path, index=False, float_format="%.6f")
    print(f"✓ Saved Excel table: {excel_path}")
except Exception as e:
    print(f"⚠️  Could not save Excel (install openpyxl): {e}")


# =============================================================================
# CREATE PAPER-READY SUMMARY TABLES
# =============================================================================

print("\n" + "=" * 80)
print("GENERATING PAPER-READY TABLES")
print("=" * 80 + "\n")

# Table 1: Model Overview (Parameters + Test Window Performance)
# ================================================================
table1_cols = [
    "model",
    "total_params",
    "trainable_params",
    "test_acc",
    "test_f1_macro",
    "test_f1_micro",
    "test_f1_weighted",
    "test_ms_per_sample",
]
table1 = df[[col for col in table1_cols if col in df.columns]].copy()

# Format for readability
if "total_params" in table1.columns:
    table1["total_params"] = table1["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
if "trainable_params" in table1.columns:
    table1["trainable_params"] = table1["trainable_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")

# Rename columns for publication
table1.rename(columns={
    "model": "Model",
    "total_params": "Params (Total)",
    "trainable_params": "Params (Train)",
    "test_acc": "Acc",
    "test_f1_macro": "F1 (M)",
    "test_f1_micro": "F1 (mi)",
    "test_f1_weighted": "F1 (W)",
    "test_ms_per_sample": "ms/sample",
}, inplace=True)

print("TABLE 1: Model Overview (Test Window-Level Performance)")
print("=" * 120)
print(table1.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print()

table1_latex = table1.to_latex(
    index=False,
    float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    caption="Model comparison: Parameters and test window-level performance.",
    label="tab:model_overview",
)

latex_path1 = os.path.join(BENCHMARK_DIR, "table1_model_overview.tex")
with open(latex_path1, 'w', encoding='utf-8') as f:  # UTF-8 ENCODING ADDED
    f.write(table1_latex)
print(f"✓ Saved LaTeX: {latex_path1}\n")


# Table 2: Detailed Window-Level Metrics (TEST)
# ================================================================
table2_cols = [
    "model",
    "test_acc",
    "test_p_macro",
    "test_r_macro",
    "test_f1_macro",
    "test_p_micro",
    "test_r_micro",
    "test_f1_micro",
    "test_p_weighted",
    "test_r_weighted",
    "test_f1_weighted",
]
table2 = df[[col for col in table2_cols if col in df.columns]].copy()

table2.rename(columns={
    "model": "Model",
    "test_acc": "Acc",
    "test_p_macro": "P (M)",
    "test_r_macro": "R (M)",
    "test_f1_macro": "F1 (M)",
    "test_p_micro": "P (mi)",
    "test_r_micro": "R (mi)",
    "test_f1_micro": "F1 (mi)",
    "test_p_weighted": "P (W)",
    "test_r_weighted": "R (W)",
    "test_f1_weighted": "F1 (W)",
}, inplace=True)

print("TABLE 2: Detailed Window-Level Metrics (TEST)")
print("=" * 120)
print(table2.to_string(index=False, float_format="%.4f"))
print()

table2_latex = table2.to_latex(
    index=False,
    float_format="%.4f",
    caption="Detailed window-level test metrics: Precision (P), Recall (R), F1 for macro (M), micro (mi), and weighted (W) averages.",
    label="tab:window_metrics",
)

latex_path2 = os.path.join(BENCHMARK_DIR, "table2_window_metrics.tex")
with open(latex_path2, 'w', encoding='utf-8') as f:  # UTF-8 ENCODING ADDED
    f.write(table2_latex)
print(f"✓ Saved LaTeX: {latex_path2}\n")


# Table 3: Clip-Level Performance (TEST)
# ================================================================
table3_cols = [
    "model",
    "clip_acc",
    "clip_p_macro",
    "clip_r_macro",
    "clip_f1_macro",
    "clip_p_micro",
    "clip_r_micro",
    "clip_f1_micro",
    "clip_p_weighted",
    "clip_r_weighted",
    "clip_f1_weighted",
    "clip_default_k",
]
table3 = df[[col for col in table3_cols if col in df.columns]].copy()

table3.rename(columns={
    "model": "Model",
    "clip_acc": "Acc",
    "clip_p_macro": "P (M)",
    "clip_r_macro": "R (M)",
    "clip_f1_macro": "F1 (M)",
    "clip_p_micro": "P (mi)",
    "clip_r_micro": "R (mi)",
    "clip_f1_micro": "F1 (mi)",
    "clip_p_weighted": "P (W)",
    "clip_r_weighted": "R (W)",
    "clip_f1_weighted": "F1 (W)",
    "clip_default_k": "k",
}, inplace=True)

print("TABLE 3: Clip-Level Performance (TEST, Early-Stop Aggregation)")
print("=" * 120)
print(table3.to_string(index=False, float_format="%.4f"))
print()

table3_latex = table3.to_latex(
    index=False,
    float_format="%.4f",
    caption="Clip-level test metrics with early-stop aggregation: Precision (P), Recall (R), F1 for macro (M), micro (mi), and weighted (W) averages.",
    label="tab:clip_metrics",
)

latex_path3 = os.path.join(BENCHMARK_DIR, "table3_clip_metrics.tex")
with open(latex_path3, 'w', encoding='utf-8') as f:  # UTF-8 ENCODING ADDED
    f.write(table3_latex)
print(f"✓ Saved LaTeX: {latex_path3}\n")


# Table 4: Inference Timing
# ================================================================
table4_cols = [
    "model",
    "total_params",
    "test_ms_per_sample",
    "clip_ms_per_clip",
    "benchmark_median_window_ms",
    "benchmark_throughput",
]
table4 = df[[col for col in table4_cols if col in df.columns]].copy()

if "total_params" in table4.columns:
    table4["total_params"] = df["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")

table4.rename(columns={
    "model": "Model",
    "total_params": "Params",
    "test_ms_per_sample": "ms/window (eval)",
    "clip_ms_per_clip": "ms/clip (agg)",
    "benchmark_median_window_ms": "ms/window (bench)",
    "benchmark_throughput": "windows/s",
}, inplace=True)

print("TABLE 4: Inference Timing")
print("=" * 120)
print(table4.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
print()

table4_latex = table4.to_latex(
    index=False,
    float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
    caption="Inference timing benchmarks: per-window and per-clip processing times, plus throughput.",
    label="tab:timing",
)

latex_path4 = os.path.join(BENCHMARK_DIR, "table4_timing.tex")
with open(latex_path4, 'w', encoding='utf-8') as f:  # UTF-8 ENCODING ADDED
    f.write(table4_latex)
print(f"✓ Saved LaTeX: {latex_path4}\n")


# =============================================================================
# BEST MODEL RANKING
# =============================================================================

print("\n" + "=" * 80)
print("BEST MODEL RANKINGS")
print("=" * 80 + "\n")

rankings = {}

if "test_f1_macro" in df.columns:
    sorted_f1 = df.sort_values("test_f1_macro", ascending=False)
    print("Test F1 (Macro) - Window Level:")
    for i, row in sorted_f1.iterrows():
        print(f"  {row['model']:20s} → {row['test_f1_macro']:.4f}")
    rankings["test_f1_macro"] = sorted_f1[["model", "test_f1_macro"]].to_dict('records')

if "clip_f1_macro" in df.columns:
    print("\nTest F1 (Macro) - Clip Level:")
    sorted_clip_f1 = df.sort_values("clip_f1_macro", ascending=False)
    for i, row in sorted_clip_f1.iterrows():
        print(f"  {row['model']:20s} → {row['clip_f1_macro']:.4f}")
    rankings["clip_f1_macro"] = sorted_clip_f1[["model", "clip_f1_macro"]].to_dict('records')

if "test_acc" in df.columns:
    print("\nTest Accuracy - Window Level:")
    sorted_acc = df.sort_values("test_acc", ascending=False)
    for i, row in sorted_acc.iterrows():
        print(f"  {row['model']:20s} → {row['test_acc']:.4f}")
    rankings["test_acc"] = sorted_acc[["model", "test_acc"]].to_dict('records')

if "test_ms_per_sample" in df.columns:
    print("\n⚡ Fastest Inference - Window Level:")
    sorted_speed = df.sort_values("test_ms_per_sample", ascending=True)
    for i, row in sorted_speed.iterrows():
        if pd.notna(row["test_ms_per_sample"]):
            print(f"  {row['model']:20s} → {row['test_ms_per_sample']:.4f} ms/sample")
    rankings["inference_speed"] = sorted_speed[["model", "test_ms_per_sample"]].to_dict('records')

# Save rankings
rankings_path = os.path.join(BENCHMARK_DIR, "model_rankings.json")
with open(rankings_path, 'w') as f:
    json.dump(rankings, f, indent=2)
print(f"\n✓ Saved rankings: {rankings_path}")


# =============================================================================
# COMBINED LATEX DOCUMENT
# =============================================================================

combined_latex = r"""\documentclass{article}
\usepackage{booktabs}
\usepackage{caption}
\usepackage{geometry}
\geometry{margin=1in, landscape}

\begin{document}

\section*{Model Benchmark Comparison}

""" + table1_latex + r"""

\clearpage

""" + table2_latex + r"""

\clearpage

""" + table3_latex + r"""

\clearpage

""" + table4_latex + r"""

\end{document}
"""

combined_latex_path = os.path.join(BENCHMARK_DIR, "combined_benchmark_tables.tex")
with open(combined_latex_path, 'w', encoding='utf-8') as f:  # UTF-8 ENCODING ADDED
    f.write(combined_latex)

print(f"\n✓ Saved combined LaTeX document: {combined_latex_path}")
print("  (Compile with: pdflatex combined_benchmark_tables.tex)")


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 80)
print("BENCHMARK GENERATION COMPLETE")
print("=" * 80)
print(f"\n📁 All files saved to: {BENCHMARK_DIR}\n")
print("Generated files:")
print(f"  • full_benchmark_table.csv")
print(f"  • full_benchmark_table.xlsx")
print(f"  • table1_model_overview.tex")
print(f"  • table2_window_metrics.tex")
print(f"  • table3_clip_metrics.tex")
print(f"  • table4_timing.tex")
print(f"  • combined_benchmark_tables.tex")
print(f"  • model_rankings.json")
print("\n" + "=" * 80 + "\n")

In [ ]:
# ==== COMPREHENSIVE MODEL BENCHMARK COMPARISON (FLEXIBLE SELECTION) ====

import os
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional, Union
import re

# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS TO YOUR OUTPUT DIRECTORY
# =============================================================================
def default_outputs_root() -> str:
    # You can hard-set this if you want:
    win_default = r"C:\Users\youss\Projects\Code\outputs\Benchmark"
    if os.path.exists(win_default):
        return win_default

    # workspace / cwd / tmp fallbacks
    if os.path.exists("/workspace") and os.access("/workspace", os.W_OK):
        return "/workspace/base/src/outputs/Benchmark"
    if os.access(os.getcwd(), os.W_OK):
        return os.path.join(os.getcwd(), "outputs/Benchmark")
    return "/tmp/outputs/Benchmark"


OUTPUTS_ROOT = os.environ.get("OUTPUTS_ROOT", default_outputs_root())

# =============================================================================
# MODEL SELECTION - CONFIGURE WHICH MODELS TO EVALUATE
# =============================================================================
# 
# Option 1: Use "latest" to auto-select the most recent run
# Option 2: Specify exact run folder name (e.g., "run_20250204-143022")
# Option 3: Set to None to skip that model
# Option 4: Provide a list of runs to include multiple experiments per model type
#
# Examples:
#   "Transformer": "latest"                           # Auto-select latest
#   "Transformer": "run_20250204-143022"              # Specific run
#   "Transformer": None                               # Skip this model
#   "Transformer": ["run_20250201", "run_20250204"]   # Multiple runs (will add suffix)
#

MODEL_SELECTION = {
    "Transformer": "transformer_aug",       # Change to specific run name or None to skip
    "Apple": "apple_aug",
    "InceptionTime": "inception_aug",
    "Hydra": "hydra_aug",
    "NormWear-Base": "base_aug",
    "NormWear-LoRA": "lora_aug",
}

# Model type to subdirectory mapping (don't modify unless folder structure changes)
MODEL_SUBDIRS = {
    "Transformer": "transformer",
    "Apple": "apple",
    "InceptionTime": "inceptiontime",
    "Hydra": "hydra",
    "NormWear-Base": "normwear_base",
    "NormWear-LoRA": "normwear_lora",
}

# Output directory for benchmark results
BENCHMARK_DIR = os.path.join(OUTPUTS_ROOT, "benchmark_comparison")


# =============================================================================
# HELPER: LIST AVAILABLE RUNS
# =============================================================================

def list_available_runs(outputs_root: str = OUTPUTS_ROOT) -> Dict[str, List[str]]:
    """List all available run directories for each model type."""
    available = {}
    
    for model_name, subdir in MODEL_SUBDIRS.items():
        model_path = os.path.join(outputs_root, subdir)
        if os.path.exists(model_path):
            runs = sorted(glob.glob(os.path.join(model_path, "run_*")), reverse=True)
            available[model_name] = [os.path.basename(r) for r in runs]
        else:
            available[model_name] = []
    
    return available


def print_available_runs(outputs_root: str = OUTPUTS_ROOT):
    """Pretty-print all available runs for easy selection."""
    available = list_available_runs(outputs_root)
    
    print("\n" + "=" * 80)
    print("AVAILABLE MODEL RUNS")
    print("=" * 80)
    
    for model_name, runs in available.items():
        print(f"\n📁 {model_name} ({MODEL_SUBDIRS[model_name]}/)")
        if runs:
            for i, run in enumerate(runs):
                marker = "  → " if i == 0 else "    "
                latest_tag = " [latest]" if i == 0 else ""
                print(f"{marker}{run}{latest_tag}")
        else:
            print("    (no runs found)")
    
    print("\n" + "=" * 80)
    print("Copy run names above into MODEL_SELECTION to use specific runs.")
    print("=" * 80 + "\n")


# =============================================================================
# HELPER FUNCTIONS TO RESOLVE AND LOAD METRICS
# =============================================================================

def resolve_run_dir(
    model_name: str,
    selection: Union[str, List[str], None],
    outputs_root: str
) -> List[Optional[str]]:
    """
    Resolve the run directory based on selection.
    
    Returns a list of (model_label, run_dir) tuples to support multiple runs.
    """
    if selection is None:
        return []
    
    subdir = MODEL_SUBDIRS.get(model_name)
    if subdir is None:
        print(f"⚠️  Unknown model type: {model_name}")
        return []
    
    model_path = os.path.join(outputs_root, subdir)
    
    if not os.path.exists(model_path):
        print(f"⚠️  Model directory does not exist: {model_path}")
        return []
    
    # Handle list of runs (multiple experiments)
    if isinstance(selection, list):
        results = []
        for i, sel in enumerate(selection):
            run_dir = _resolve_single_run(model_path, sel, model_name)
            if run_dir:
                # Add suffix to distinguish multiple runs of same model
                label = f"{model_name} ({os.path.basename(run_dir)})"
                results.append((label, run_dir))
        return results
    
    # Handle single selection
    run_dir = _resolve_single_run(model_path, selection, model_name)
    if run_dir:
        return [(model_name, run_dir)]
    return []


def _resolve_single_run(model_path: str, selection: str, model_name: str) -> Optional[str]:
    """Resolve a single run selection to a directory path."""
    
    if selection == "latest":
        # Find latest run
        run_dirs = sorted(glob.glob(os.path.join(model_path, "run_*")), reverse=True)
        if run_dirs:
            print(f"✓ {model_name}: Using latest → {os.path.basename(run_dirs[0])}")
            return run_dirs[0]
        else:
            print(f"⚠️  {model_name}: No run_* directories found")
            return None
    
    # Specific run name
    run_dir = os.path.join(model_path, selection)
    if os.path.exists(run_dir):
        print(f"✓ {model_name}: Using specified → {selection}")
        return run_dir
    
    # Try partial match
    matches = glob.glob(os.path.join(model_path, f"*{selection}*"))
    if len(matches) == 1:
        print(f"✓ {model_name}: Matched → {os.path.basename(matches[0])}")
        return matches[0]
    elif len(matches) > 1:
        print(f"⚠️  {model_name}: Multiple matches for '{selection}':")
        for m in matches:
            print(f"      {os.path.basename(m)}")
        print(f"    Using first match: {os.path.basename(matches[0])}")
        return matches[0]
    
    print(f"⚠️  {model_name}: Run not found: {selection}")
    return None


def safe_load_json(filepath: str, default: Any = None) -> Any:
    """Safely load JSON file, return default if not found."""
    if not os.path.exists(filepath):
        return default
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  Error loading {filepath}: {e}")
        return default


def safe_get_nested(data: Dict, *keys, default=None) -> Any:
    """Safely get nested dictionary value."""
    if data is None:
        return default
    current = data
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        else:
            return default
        if current is None:
            return default
    return current


def extract_model_metrics(run_dir: str, model_name: str) -> Dict[str, Any]:
    """Extract all relevant metrics from a model's run directory."""
    
    metrics = {
        "model": model_name,
        "run_dir": os.path.basename(run_dir),
    }
    
    # 1. Parameter statistics
    param_summary = safe_load_json(os.path.join(run_dir, "parameter_summary.json"))
    if param_summary:
        metrics["total_params"] = param_summary.get("model_total_params", np.nan)
        metrics["trainable_params"] = param_summary.get("model_trainable_params", np.nan)
        metrics["trainable_pct"] = param_summary.get("trainable_percent", np.nan)
    else:
        # Try parsing parameter_stats.txt
        param_file = os.path.join(run_dir, "parameter_stats.txt")
        if os.path.exists(param_file):
            with open(param_file, 'r', encoding='utf-8') as f:
                content = f.read()
                total_match = re.search(r"[Tt]otal params?:\s*([\d,]+)", content)
                trainable_match = re.search(r"[Tt]rainable params?:\s*([\d,]+)", content)
                if total_match:
                    metrics["total_params"] = int(total_match.group(1).replace(",", ""))
                if trainable_match:
                    metrics["trainable_params"] = int(trainable_match.group(1).replace(",", ""))
    
    # 2. Window-level metrics (VAL + TEST)
    window_metrics = safe_load_json(os.path.join(run_dir, "metrics_window.json"))
    
    if window_metrics:
        # VAL metrics
        val_data = window_metrics.get("val", {})
        metrics["val_loss"] = safe_get_nested(val_data, "loss", default=np.nan)
        metrics["val_acc"] = safe_get_nested(val_data, "acc", default=np.nan)
        
        val_prf = safe_get_nested(val_data, "prf", default={})
        metrics["val_p_macro"] = safe_get_nested(val_prf, "macro", "precision", default=np.nan)
        metrics["val_r_macro"] = safe_get_nested(val_prf, "macro", "recall", default=np.nan)
        metrics["val_f1_macro"] = safe_get_nested(val_prf, "macro", "f1", default=np.nan)
        metrics["val_p_micro"] = safe_get_nested(val_prf, "micro", "precision", default=np.nan)
        metrics["val_r_micro"] = safe_get_nested(val_prf, "micro", "recall", default=np.nan)
        metrics["val_f1_micro"] = safe_get_nested(val_prf, "micro", "f1", default=np.nan)
        metrics["val_p_weighted"] = safe_get_nested(val_prf, "weighted", "precision", default=np.nan)
        metrics["val_r_weighted"] = safe_get_nested(val_prf, "weighted", "recall", default=np.nan)
        metrics["val_f1_weighted"] = safe_get_nested(val_prf, "weighted", "f1", default=np.nan)
        
        # TEST metrics
        test_data = window_metrics.get("test", {})
        metrics["test_loss"] = safe_get_nested(test_data, "loss", default=np.nan)
        metrics["test_acc"] = safe_get_nested(test_data, "acc", default=np.nan)
        
        test_prf = safe_get_nested(test_data, "prf", default={})
        metrics["test_p_macro"] = safe_get_nested(test_prf, "macro", "precision", default=np.nan)
        metrics["test_r_macro"] = safe_get_nested(test_prf, "macro", "recall", default=np.nan)
        metrics["test_f1_macro"] = safe_get_nested(test_prf, "macro", "f1", default=np.nan)
        metrics["test_p_micro"] = safe_get_nested(test_prf, "micro", "precision", default=np.nan)
        metrics["test_r_micro"] = safe_get_nested(test_prf, "micro", "recall", default=np.nan)
        metrics["test_f1_micro"] = safe_get_nested(test_prf, "micro", "f1", default=np.nan)
        metrics["test_p_weighted"] = safe_get_nested(test_prf, "weighted", "precision", default=np.nan)
        metrics["test_r_weighted"] = safe_get_nested(test_prf, "weighted", "recall", default=np.nan)
        metrics["test_f1_weighted"] = safe_get_nested(test_prf, "weighted", "f1", default=np.nan)
        
        # Timing (window-level) from metrics_window.json
        test_timing = safe_get_nested(test_data, "timing", default={})
        metrics["test_ms_per_sample"] = safe_get_nested(test_timing, "ms_per_sample", default=np.nan)
        metrics["test_ms_per_batch"] = safe_get_nested(test_timing, "ms_per_batch", default=np.nan)
    
    # 3. Clip-level metrics (TEST only)
    clip_metrics = safe_load_json(os.path.join(run_dir, "metrics_clip.json"))
    
    if clip_metrics:
        clip_data = clip_metrics.get("test", clip_metrics)
        metrics["clip_acc"] = safe_get_nested(clip_data, "acc", default=np.nan)
        
        clip_prf = safe_get_nested(clip_data, "prf", default={})
        metrics["clip_p_macro"] = safe_get_nested(clip_prf, "macro", "precision", default=np.nan)
        metrics["clip_r_macro"] = safe_get_nested(clip_prf, "macro", "recall", default=np.nan)
        metrics["clip_f1_macro"] = safe_get_nested(clip_prf, "macro", "f1", default=np.nan)
        metrics["clip_p_micro"] = safe_get_nested(clip_prf, "micro", "precision", default=np.nan)
        metrics["clip_r_micro"] = safe_get_nested(clip_prf, "micro", "recall", default=np.nan)
        metrics["clip_f1_micro"] = safe_get_nested(clip_prf, "micro", "f1", default=np.nan)
        metrics["clip_p_weighted"] = safe_get_nested(clip_prf, "weighted", "precision", default=np.nan)
        metrics["clip_r_weighted"] = safe_get_nested(clip_prf, "weighted", "recall", default=np.nan)
        metrics["clip_f1_weighted"] = safe_get_nested(clip_prf, "weighted", "f1", default=np.nan)
        
        clip_timing = safe_get_nested(clip_metrics, "timing", default={})
        metrics["clip_ms_per_clip"] = safe_get_nested(clip_timing, "aggregation_ms_per_clip", default=np.nan)
        
        early_stop = safe_get_nested(clip_metrics, "early_stop", default={})
        metrics["clip_default_k"] = safe_get_nested(early_stop, "default_k", default=np.nan)
        metrics["clip_missed_early"] = safe_get_nested(early_stop, "missed_early", default=np.nan)
        metrics["clip_n_clips"] = safe_get_nested(early_stop, "n_clips", default=np.nan)
    
    # 4. Inference benchmark (support multiple file formats)
    # Initialize timing fields
    metrics["benchmark_mean_window_ms"] = np.nan
    metrics["benchmark_median_window_ms"] = np.nan
    metrics["benchmark_std_window_ms"] = np.nan
    metrics["benchmark_min_window_ms"] = np.nan
    metrics["benchmark_max_window_ms"] = np.nan
    metrics["benchmark_p95_window_ms"] = np.nan
    metrics["benchmark_p99_window_ms"] = np.nan
    metrics["benchmark_throughput"] = np.nan
    
    # Try the standard benchmark file first (NormWear format)
    benchmark = safe_load_json(os.path.join(run_dir, "inference_benchmark_window.json"))
    if benchmark:
        metrics["benchmark_median_window_ms"] = safe_get_nested(benchmark, "median_per_window_ms", default=np.nan)
        metrics["benchmark_throughput"] = safe_get_nested(benchmark, "throughput_windows_per_s", default=np.nan)
        metrics["benchmark_mean_window_ms"] = safe_get_nested(benchmark, "mean_per_window_ms", default=np.nan)
        metrics["benchmark_std_window_ms"] = safe_get_nested(benchmark, "std_per_window_ms", default=np.nan)
        metrics["benchmark_min_window_ms"] = safe_get_nested(benchmark, "min_per_window_ms", default=np.nan)
        metrics["benchmark_max_window_ms"] = safe_get_nested(benchmark, "max_per_window_ms", default=np.nan)
        metrics["benchmark_p95_window_ms"] = safe_get_nested(benchmark, "p95_per_window_ms", default=np.nan)
        metrics["benchmark_p99_window_ms"] = safe_get_nested(benchmark, "p99_per_window_ms", default=np.nan)
    
    # Also try inference_timing.json (Transformer format from standalone timing cell)
    inference_timing = safe_load_json(os.path.join(run_dir, "inference_timing.json"))
    if inference_timing:
        time_stats = safe_get_nested(inference_timing, "time_per_window_ms", default={})
        
        # Only overwrite if not already set from benchmark file
        if pd.isna(metrics.get("benchmark_median_window_ms", np.nan)):
            metrics["benchmark_median_window_ms"] = safe_get_nested(time_stats, "median", default=np.nan)
        if pd.isna(metrics.get("benchmark_throughput", np.nan)):
            metrics["benchmark_throughput"] = safe_get_nested(inference_timing, "throughput_windows_per_sec", default=np.nan)
        if pd.isna(metrics.get("benchmark_mean_window_ms", np.nan)):
            metrics["benchmark_mean_window_ms"] = safe_get_nested(time_stats, "mean", default=np.nan)
        if pd.isna(metrics.get("benchmark_std_window_ms", np.nan)):
            metrics["benchmark_std_window_ms"] = safe_get_nested(time_stats, "std", default=np.nan)
        if pd.isna(metrics.get("benchmark_min_window_ms", np.nan)):
            metrics["benchmark_min_window_ms"] = safe_get_nested(time_stats, "min", default=np.nan)
        if pd.isna(metrics.get("benchmark_max_window_ms", np.nan)):
            metrics["benchmark_max_window_ms"] = safe_get_nested(time_stats, "max", default=np.nan)
        if pd.isna(metrics.get("benchmark_p95_window_ms", np.nan)):
            metrics["benchmark_p95_window_ms"] = safe_get_nested(time_stats, "p95", default=np.nan)
        if pd.isna(metrics.get("benchmark_p99_window_ms", np.nan)):
            metrics["benchmark_p99_window_ms"] = safe_get_nested(time_stats, "p99", default=np.nan)
    
    return metrics


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_benchmark(
    model_selection: Dict[str, Union[str, List[str], None]] = None,
    outputs_root: str = OUTPUTS_ROOT,
    benchmark_dir: str = None,
    list_only: bool = False,
):
    """
    Run the benchmark comparison.
    
    Args:
        model_selection: Dict mapping model names to run selections.
                        Use "latest", specific run name, None to skip, or list for multiple.
        outputs_root: Root directory containing model outputs.
        benchmark_dir: Output directory for benchmark results.
        list_only: If True, just list available runs and exit.
    """
    
    if list_only:
        print_available_runs(outputs_root)
        return None
    
    if model_selection is None:
        model_selection = MODEL_SELECTION
    
    if benchmark_dir is None:
        benchmark_dir = os.path.join(outputs_root, "benchmark_comparison")
    
    Path(benchmark_dir).mkdir(parents=True, exist_ok=True)
    print(f"Benchmark output directory: {benchmark_dir}\n")
    
    # ==========================================================================
    # COLLECT ALL MODEL METRICS
    # ==========================================================================
    
    print("=" * 80)
    print("RESOLVING MODEL RUNS")
    print("=" * 80 + "\n")
    
    runs_to_evaluate = []
    
    for model_name, selection in model_selection.items():
        if selection is None:
            print(f"⏭️  {model_name}: Skipped (set to None)")
            continue
        
        resolved = resolve_run_dir(model_name, selection, outputs_root)
        runs_to_evaluate.extend(resolved)
    
    if not runs_to_evaluate:
        print("\n❌ No valid runs to evaluate!")
        print("   Run with list_only=True to see available runs.")
        return None
    
    print(f"\n✓ Will evaluate {len(runs_to_evaluate)} model run(s)\n")
    
    print("=" * 80)
    print("COLLECTING METRICS")
    print("=" * 80 + "\n")
    
    all_metrics = []
    
    for model_label, run_dir in runs_to_evaluate:
        print(f"📊 Extracting metrics: {model_label}")
        metrics = extract_model_metrics(run_dir, model_label)
        all_metrics.append(metrics)
        print(f"   ✓ {len(metrics)} fields extracted\n")
    
    # Create DataFrame
    df = pd.DataFrame(all_metrics)
    
    # Reorder columns
    column_order = [
        "model", "run_dir",
        "total_params", "trainable_params", "trainable_pct",
        "val_loss", "val_acc", "val_f1_macro", "val_f1_micro", "val_f1_weighted",
        "val_p_macro", "val_r_macro", "val_p_micro", "val_r_micro", "val_p_weighted", "val_r_weighted",
        "test_loss", "test_acc", "test_f1_macro", "test_f1_micro", "test_f1_weighted",
        "test_p_macro", "test_r_macro", "test_p_micro", "test_r_micro", "test_p_weighted", "test_r_weighted",
        "test_ms_per_sample", "test_ms_per_batch",
        "clip_acc", "clip_f1_macro", "clip_f1_micro", "clip_f1_weighted",
        "clip_p_macro", "clip_r_macro", "clip_p_micro", "clip_r_micro", "clip_p_weighted", "clip_r_weighted",
        "clip_ms_per_clip", "clip_default_k", "clip_missed_early", "clip_n_clips",
        "benchmark_mean_window_ms", "benchmark_median_window_ms", "benchmark_std_window_ms",
        "benchmark_min_window_ms", "benchmark_max_window_ms",
        "benchmark_p95_window_ms", "benchmark_p99_window_ms", "benchmark_throughput",
    ]
    column_order = [col for col in column_order if col in df.columns]
    df = df[column_order]
    
    print("=" * 80)
    print("METRICS COLLECTION COMPLETE")
    print("=" * 80)
    print(f"\n✓ Collected metrics from {len(df)} model run(s)")
    print(f"✓ Total metrics per model: {len(df.columns)}")
    
    # ==========================================================================
    # SAVE OUTPUTS
    # ==========================================================================
    
    csv_path = os.path.join(benchmark_dir, "full_benchmark_table.csv")
    df.to_csv(csv_path, index=False, float_format="%.6f")
    print(f"\n✓ Saved: {csv_path}")
    
    try:
        excel_path = os.path.join(benchmark_dir, "full_benchmark_table.xlsx")
        df.to_excel(excel_path, index=False, float_format="%.6f")
        print(f"✓ Saved: {excel_path}")
    except Exception as e:
        print(f"⚠️  Excel export failed: {e}")
    
    # ==========================================================================
    # PAPER-READY TABLES
    # ==========================================================================
    
    print("\n" + "=" * 80)
    print("GENERATING PAPER-READY TABLES")
    print("=" * 80 + "\n")
    
    # Table 1: Model Overview
    table1_cols = ["model", "total_params", "trainable_params", "test_acc", 
                   "test_f1_macro", "test_f1_micro", "test_f1_weighted", 
                   "benchmark_median_window_ms"]
    table1 = df[[col for col in table1_cols if col in df.columns]].copy()
    
    if "total_params" in table1.columns:
        table1["total_params"] = table1["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    if "trainable_params" in table1.columns:
        table1["trainable_params"] = table1["trainable_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    
    table1.rename(columns={
        "model": "Model", "total_params": "Params (Total)", "trainable_params": "Params (Train)",
        "test_acc": "Acc", "test_f1_macro": "F1 (M)", "test_f1_micro": "F1 (mi)",
        "test_f1_weighted": "F1 (W)", "benchmark_median_window_ms": "ms/window",
    }, inplace=True)
    
    print("TABLE 1: Model Overview (Test Window-Level)")
    print("-" * 100)
    print(table1.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print()
    
    table1_latex = table1.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
        caption="Model comparison: Parameters and test window-level performance.",
        label="tab:model_overview",
    )
    with open(os.path.join(benchmark_dir, "table1_model_overview.tex"), 'w', encoding='utf-8') as f:
        f.write(table1_latex)
    
    # Table 2: Detailed Window Metrics
    table2_cols = ["model", "test_acc", "test_p_macro", "test_r_macro", "test_f1_macro",
                   "test_p_micro", "test_r_micro", "test_f1_micro",
                   "test_p_weighted", "test_r_weighted", "test_f1_weighted"]
    table2 = df[[col for col in table2_cols if col in df.columns]].copy()
    table2.rename(columns={
        "model": "Model", "test_acc": "Acc",
        "test_p_macro": "P (M)", "test_r_macro": "R (M)", "test_f1_macro": "F1 (M)",
        "test_p_micro": "P (mi)", "test_r_micro": "R (mi)", "test_f1_micro": "F1 (mi)",
        "test_p_weighted": "P (W)", "test_r_weighted": "R (W)", "test_f1_weighted": "F1 (W)",
    }, inplace=True)
    
    print("TABLE 2: Detailed Window-Level Metrics (TEST)")
    print("-" * 100)
    print(table2.to_string(index=False, float_format="%.4f"))
    print()
    
    table2_latex = table2.to_latex(index=False, float_format="%.4f",
        caption="Detailed window-level test metrics.", label="tab:window_metrics")
    with open(os.path.join(benchmark_dir, "table2_window_metrics.tex"), 'w', encoding='utf-8') as f:
        f.write(table2_latex)
    
    # Table 3: Clip-Level
    table3_cols = ["model", "clip_acc", "clip_p_macro", "clip_r_macro", "clip_f1_macro",
                   "clip_p_micro", "clip_r_micro", "clip_f1_micro",
                   "clip_p_weighted", "clip_r_weighted", "clip_f1_weighted", "clip_default_k"]
    table3 = df[[col for col in table3_cols if col in df.columns]].copy()
    table3.rename(columns={
        "model": "Model", "clip_acc": "Acc",
        "clip_p_macro": "P (M)", "clip_r_macro": "R (M)", "clip_f1_macro": "F1 (M)",
        "clip_p_micro": "P (mi)", "clip_r_micro": "R (mi)", "clip_f1_micro": "F1 (mi)",
        "clip_p_weighted": "P (W)", "clip_r_weighted": "R (W)", "clip_f1_weighted": "F1 (W)",
        "clip_default_k": "k",
    }, inplace=True)
    
    print("TABLE 3: Clip-Level Performance (TEST)")
    print("-" * 100)
    print(table3.to_string(index=False, float_format="%.4f"))
    print()
    
    table3_latex = table3.to_latex(index=False, float_format="%.4f",
        caption="Clip-level test metrics.", label="tab:clip_metrics")
    with open(os.path.join(benchmark_dir, "table3_clip_metrics.tex"), 'w', encoding='utf-8') as f:
        f.write(table3_latex)
    
    # Table 4: Timing (updated with comprehensive timing stats)
    table4_cols = ["model", "total_params", "test_ms_per_sample",
                   "benchmark_mean_window_ms", "benchmark_median_window_ms",
                   "benchmark_std_window_ms", "benchmark_p95_window_ms", "benchmark_throughput"]
    table4 = df[[col for col in table4_cols if col in df.columns]].copy()
    if "total_params" in table4.columns:
        table4["total_params"] = df["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    table4.rename(columns={
        "model": "Model", "total_params": "Params",
        "test_ms_per_sample": "ms/win (eval)",
        "benchmark_mean_window_ms": "Mean (ms)",
        "benchmark_median_window_ms": "Median (ms)",
        "benchmark_std_window_ms": "Std (ms)",
        "benchmark_p95_window_ms": "P95 (ms)",
        "benchmark_throughput": "windows/s",
    }, inplace=True)
    
    print("TABLE 4: Inference Timing")
    print("-" * 100)
    print(table4.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print()
    
    table4_latex = table4.to_latex(index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
        caption="Inference timing benchmarks.", label="tab:timing")
    with open(os.path.join(benchmark_dir, "table4_timing.tex"), 'w', encoding='utf-8') as f:
        f.write(table4_latex)
    
    # ==========================================================================
    # RANKINGS
    # ==========================================================================
    
    print("=" * 80)
    print("BEST MODEL RANKINGS")
    print("=" * 80 + "\n")
    
    rankings = {}
    
    for metric, ascending, label in [
        ("test_f1_macro", False, "Test F1 (Macro) - Window"),
        ("clip_f1_macro", False, "Test F1 (Macro) - Clip"),
        ("test_acc", False, "Test Accuracy - Window"),
        ("benchmark_median_window_ms", True, "⚡ Fastest Inference (Median)"),
        ("benchmark_throughput", False, "⚡ Highest Throughput"),
    ]:
        if metric in df.columns and df[metric].notna().any():
            sorted_df = df.sort_values(metric, ascending=ascending)
            print(f"{label}:")
            for _, row in sorted_df.iterrows():
                val = row[metric]
                if pd.notna(val):
                    if "ms" in metric:
                        suffix = " ms"
                    elif "throughput" in metric:
                        suffix = " windows/s"
                    else:
                        suffix = ""
                    print(f"  {row['model']:30s} → {val:.4f}{suffix}")
            print()
            rankings[metric] = sorted_df[["model", metric]].dropna().to_dict('records')
    
    with open(os.path.join(benchmark_dir, "model_rankings.json"), 'w') as f:
        json.dump(rankings, f, indent=2)
    
    # ==========================================================================
    # SUMMARY
    # ==========================================================================
    
    print("=" * 80)
    print("BENCHMARK COMPLETE")
    print("=" * 80)
    print(f"\n📁 Output: {benchmark_dir}\n")
    
    return df


# =============================================================================
# COMMAND-LINE INTERFACE
# =============================================================================

if __name__ == "__main__":
    # Step 1: List available runs to see what's there
    print_available_runs(OUTPUTS_ROOT)
    
    # Step 2: Run the benchmark with MODEL_SELECTION defined above
    df = run_benchmark()
    
    # Or run with custom selection inline:
    # df = run_benchmark(
    #     model_selection={
    #         "Transformer": "transformer_aug",   # specific run
    #         "Apple": "latest",                   # most recent
    #         "InceptionTime": None,               # skip this one
    #     },
    #     outputs_root=OUTPUTS_ROOT,
    # )

In [ ]:

# ==== COMPREHENSIVE MODEL BENCHMARK COMPARISON (FLEXIBLE SELECTION) ====

import os
import json
import glob
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Dict, Any, List, Optional, Union
import re

# =============================================================================
# CONFIGURATION - UPDATE THESE PATHS TO YOUR OUTPUT DIRECTORY
# =============================================================================
def default_outputs_root() -> str:
    # You can hard-set this if you want:
    win_default = r"C:\Users\youss\Projects\Code\outputs\Benchmark"
    if os.path.exists(win_default):
        return win_default

    # workspace / cwd / tmp fallbacks
    if os.path.exists("/workspace") and os.access("/workspace", os.W_OK):
        return "/workspace/base/src/outputs/Benchmark"
    if os.access(os.getcwd(), os.W_OK):
        return os.path.join(os.getcwd(), "outputs/Benchmark")
    return "/tmp/outputs/Benchmark"


OUTPUTS_ROOT = os.environ.get("OUTPUTS_ROOT", default_outputs_root())

# =============================================================================
# MODEL SELECTION - CONFIGURE WHICH MODELS TO EVALUATE
# =============================================================================
# 
# Option 1: Use "latest" to auto-select the most recent run
# Option 2: Specify exact run folder name (e.g., "run_20250204-143022")
# Option 3: Set to None to skip that model
# Option 4: Provide a list of runs to include multiple experiments per model type
#
# Examples:
#   "Transformer": "latest"                           # Auto-select latest
#   "Transformer": "run_20250204-143022"              # Specific run
#   "Transformer": None                               # Skip this model
#   "Transformer": ["run_20250201", "run_20250204"]   # Multiple runs (will add suffix)
#



MODEL_SELECTION = {
    "Transformer": "transformer_no_aug",       # Change to specific run name or None to skip
    "Apple": "apple_no_aug",
    "InceptionTime": "inception_no_aug",
    "Hydra": "hydra_no_aug",
    "NormWear-Base": "base_no_aug",
    "NormWear-LoRA": "lora_no_aug",
}

# Model type to subdirectory mapping (don't modify unless folder structure changes)
MODEL_SUBDIRS = {
    "Transformer": "transformer",
    "Apple": "apple",
    "InceptionTime": "inceptiontime",
    "Hydra": "hydra",
    "NormWear-Base": "normwear_base",
    "NormWear-LoRA": "normwear_lora",
}


# Output directory for benchmark results
BENCHMARK_DIR = os.path.join(OUTPUTS_ROOT, "benchmark_comparison")


# =============================================================================
# HELPER: LIST AVAILABLE RUNS
# =============================================================================

def list_available_runs(outputs_root: str = OUTPUTS_ROOT) -> Dict[str, List[str]]:
    """List all available run directories for each model type."""
    available = {}
    
    for model_name, subdir in MODEL_SUBDIRS.items():
        model_path = os.path.join(outputs_root, subdir)
        if os.path.exists(model_path):
            runs = sorted(glob.glob(os.path.join(model_path, "run_*")), reverse=True)
            available[model_name] = [os.path.basename(r) for r in runs]
        else:
            available[model_name] = []
    
    return available


def print_available_runs(outputs_root: str = OUTPUTS_ROOT):
    """Pretty-print all available runs for easy selection."""
    available = list_available_runs(outputs_root)
    
    print("\n" + "=" * 80)
    print("AVAILABLE MODEL RUNS")
    print("=" * 80)
    
    for model_name, runs in available.items():
        print(f"\n📁 {model_name} ({MODEL_SUBDIRS[model_name]}/)")
        if runs:
            for i, run in enumerate(runs):
                marker = "  → " if i == 0 else "    "
                latest_tag = " [latest]" if i == 0 else ""
                print(f"{marker}{run}{latest_tag}")
        else:
            print("    (no runs found)")
    
    print("\n" + "=" * 80)
    print("Copy run names above into MODEL_SELECTION to use specific runs.")
    print("=" * 80 + "\n")


# =============================================================================
# HELPER FUNCTIONS TO RESOLVE AND LOAD METRICS
# =============================================================================

def resolve_run_dir(
    model_name: str,
    selection: Union[str, List[str], None],
    outputs_root: str
) -> List[Optional[str]]:
    """
    Resolve the run directory based on selection.
    
    Returns a list of (model_label, run_dir) tuples to support multiple runs.
    """
    if selection is None:
        return []
    
    subdir = MODEL_SUBDIRS.get(model_name)
    if subdir is None:
        print(f"⚠️  Unknown model type: {model_name}")
        return []
    
    model_path = os.path.join(outputs_root, subdir)
    
    if not os.path.exists(model_path):
        print(f"⚠️  Model directory does not exist: {model_path}")
        return []
    
    # Handle list of runs (multiple experiments)
    if isinstance(selection, list):
        results = []
        for i, sel in enumerate(selection):
            run_dir = _resolve_single_run(model_path, sel, model_name)
            if run_dir:
                # Add suffix to distinguish multiple runs of same model
                label = f"{model_name} ({os.path.basename(run_dir)})"
                results.append((label, run_dir))
        return results
    
    # Handle single selection
    run_dir = _resolve_single_run(model_path, selection, model_name)
    if run_dir:
        return [(model_name, run_dir)]
    return []


def _resolve_single_run(model_path: str, selection: str, model_name: str) -> Optional[str]:
    """Resolve a single run selection to a directory path."""
    
    if selection == "latest":
        # Find latest run
        run_dirs = sorted(glob.glob(os.path.join(model_path, "run_*")), reverse=True)
        if run_dirs:
            print(f"✓ {model_name}: Using latest → {os.path.basename(run_dirs[0])}")
            return run_dirs[0]
        else:
            print(f"⚠️  {model_name}: No run_* directories found")
            return None
    
    # Specific run name
    run_dir = os.path.join(model_path, selection)
    if os.path.exists(run_dir):
        print(f"✓ {model_name}: Using specified → {selection}")
        return run_dir
    
    # Try partial match
    matches = glob.glob(os.path.join(model_path, f"*{selection}*"))
    if len(matches) == 1:
        print(f"✓ {model_name}: Matched → {os.path.basename(matches[0])}")
        return matches[0]
    elif len(matches) > 1:
        print(f"⚠️  {model_name}: Multiple matches for '{selection}':")
        for m in matches:
            print(f"      {os.path.basename(m)}")
        print(f"    Using first match: {os.path.basename(matches[0])}")
        return matches[0]
    
    print(f"⚠️  {model_name}: Run not found: {selection}")
    return None


def safe_load_json(filepath: str, default: Any = None) -> Any:
    """Safely load JSON file, return default if not found."""
    if not os.path.exists(filepath):
        return default
    try:
        with open(filepath, 'r') as f:
            return json.load(f)
    except Exception as e:
        print(f"⚠️  Error loading {filepath}: {e}")
        return default


def safe_get_nested(data: Dict, *keys, default=None) -> Any:
    """Safely get nested dictionary value."""
    if data is None:
        return default
    current = data
    for key in keys:
        if isinstance(current, dict):
            current = current.get(key)
        else:
            return default
        if current is None:
            return default
    return current


def extract_model_metrics(run_dir: str, model_name: str) -> Dict[str, Any]:
    """Extract all relevant metrics from a model's run directory."""
    
    metrics = {
        "model": model_name,
        "run_dir": os.path.basename(run_dir),
    }
    
    # 1. Parameter statistics
    param_summary = safe_load_json(os.path.join(run_dir, "parameter_summary.json"))
    if param_summary:
        metrics["total_params"] = param_summary.get("model_total_params", np.nan)
        metrics["trainable_params"] = param_summary.get("model_trainable_params", np.nan)
        metrics["trainable_pct"] = param_summary.get("trainable_percent", np.nan)
    else:
        # Try parsing parameter_stats.txt
        param_file = os.path.join(run_dir, "parameter_stats.txt")
        if os.path.exists(param_file):
            with open(param_file, 'r', encoding='utf-8') as f:
                content = f.read()
                total_match = re.search(r"[Tt]otal params?:\s*([\d,]+)", content)
                trainable_match = re.search(r"[Tt]rainable params?:\s*([\d,]+)", content)
                if total_match:
                    metrics["total_params"] = int(total_match.group(1).replace(",", ""))
                if trainable_match:
                    metrics["trainable_params"] = int(trainable_match.group(1).replace(",", ""))
    
    # 2. Window-level metrics (VAL + TEST)
    window_metrics = safe_load_json(os.path.join(run_dir, "metrics_window.json"))
    
    if window_metrics:
        # VAL metrics
        val_data = window_metrics.get("val", {})
        metrics["val_loss"] = safe_get_nested(val_data, "loss", default=np.nan)
        metrics["val_acc"] = safe_get_nested(val_data, "acc", default=np.nan)
        
        val_prf = safe_get_nested(val_data, "prf", default={})
        metrics["val_p_macro"] = safe_get_nested(val_prf, "macro", "precision", default=np.nan)
        metrics["val_r_macro"] = safe_get_nested(val_prf, "macro", "recall", default=np.nan)
        metrics["val_f1_macro"] = safe_get_nested(val_prf, "macro", "f1", default=np.nan)
        metrics["val_p_micro"] = safe_get_nested(val_prf, "micro", "precision", default=np.nan)
        metrics["val_r_micro"] = safe_get_nested(val_prf, "micro", "recall", default=np.nan)
        metrics["val_f1_micro"] = safe_get_nested(val_prf, "micro", "f1", default=np.nan)
        metrics["val_p_weighted"] = safe_get_nested(val_prf, "weighted", "precision", default=np.nan)
        metrics["val_r_weighted"] = safe_get_nested(val_prf, "weighted", "recall", default=np.nan)
        metrics["val_f1_weighted"] = safe_get_nested(val_prf, "weighted", "f1", default=np.nan)
        
        # TEST metrics
        test_data = window_metrics.get("test", {})
        metrics["test_loss"] = safe_get_nested(test_data, "loss", default=np.nan)
        metrics["test_acc"] = safe_get_nested(test_data, "acc", default=np.nan)
        
        test_prf = safe_get_nested(test_data, "prf", default={})
        metrics["test_p_macro"] = safe_get_nested(test_prf, "macro", "precision", default=np.nan)
        metrics["test_r_macro"] = safe_get_nested(test_prf, "macro", "recall", default=np.nan)
        metrics["test_f1_macro"] = safe_get_nested(test_prf, "macro", "f1", default=np.nan)
        metrics["test_p_micro"] = safe_get_nested(test_prf, "micro", "precision", default=np.nan)
        metrics["test_r_micro"] = safe_get_nested(test_prf, "micro", "recall", default=np.nan)
        metrics["test_f1_micro"] = safe_get_nested(test_prf, "micro", "f1", default=np.nan)
        metrics["test_p_weighted"] = safe_get_nested(test_prf, "weighted", "precision", default=np.nan)
        metrics["test_r_weighted"] = safe_get_nested(test_prf, "weighted", "recall", default=np.nan)
        metrics["test_f1_weighted"] = safe_get_nested(test_prf, "weighted", "f1", default=np.nan)
        
        # Timing (window-level) from metrics_window.json
        test_timing = safe_get_nested(test_data, "timing", default={})
        metrics["test_ms_per_sample"] = safe_get_nested(test_timing, "ms_per_sample", default=np.nan)
        metrics["test_ms_per_batch"] = safe_get_nested(test_timing, "ms_per_batch", default=np.nan)
    
    # 3. Clip-level metrics (TEST only)
    clip_metrics = safe_load_json(os.path.join(run_dir, "metrics_clip.json"))
    
    if clip_metrics:
        clip_data = clip_metrics.get("test", clip_metrics)
        metrics["clip_acc"] = safe_get_nested(clip_data, "acc", default=np.nan)
        
        clip_prf = safe_get_nested(clip_data, "prf", default={})
        metrics["clip_p_macro"] = safe_get_nested(clip_prf, "macro", "precision", default=np.nan)
        metrics["clip_r_macro"] = safe_get_nested(clip_prf, "macro", "recall", default=np.nan)
        metrics["clip_f1_macro"] = safe_get_nested(clip_prf, "macro", "f1", default=np.nan)
        metrics["clip_p_micro"] = safe_get_nested(clip_prf, "micro", "precision", default=np.nan)
        metrics["clip_r_micro"] = safe_get_nested(clip_prf, "micro", "recall", default=np.nan)
        metrics["clip_f1_micro"] = safe_get_nested(clip_prf, "micro", "f1", default=np.nan)
        metrics["clip_p_weighted"] = safe_get_nested(clip_prf, "weighted", "precision", default=np.nan)
        metrics["clip_r_weighted"] = safe_get_nested(clip_prf, "weighted", "recall", default=np.nan)
        metrics["clip_f1_weighted"] = safe_get_nested(clip_prf, "weighted", "f1", default=np.nan)
        
        clip_timing = safe_get_nested(clip_metrics, "timing", default={})
        metrics["clip_ms_per_clip"] = safe_get_nested(clip_timing, "aggregation_ms_per_clip", default=np.nan)
        
        early_stop = safe_get_nested(clip_metrics, "early_stop", default={})
        metrics["clip_default_k"] = safe_get_nested(early_stop, "default_k", default=np.nan)
        metrics["clip_missed_early"] = safe_get_nested(early_stop, "missed_early", default=np.nan)
        metrics["clip_n_clips"] = safe_get_nested(early_stop, "n_clips", default=np.nan)
    
    # 4. Inference benchmark (support multiple file formats)
    # Initialize timing fields
    metrics["benchmark_mean_window_ms"] = np.nan
    metrics["benchmark_median_window_ms"] = np.nan
    metrics["benchmark_std_window_ms"] = np.nan
    metrics["benchmark_min_window_ms"] = np.nan
    metrics["benchmark_max_window_ms"] = np.nan
    metrics["benchmark_p95_window_ms"] = np.nan
    metrics["benchmark_p99_window_ms"] = np.nan
    metrics["benchmark_throughput"] = np.nan
    
    # Try the standard benchmark file first (NormWear format)
    benchmark = safe_load_json(os.path.join(run_dir, "inference_benchmark_window.json"))
    if benchmark:
        metrics["benchmark_median_window_ms"] = safe_get_nested(benchmark, "median_per_window_ms", default=np.nan)
        metrics["benchmark_throughput"] = safe_get_nested(benchmark, "throughput_windows_per_s", default=np.nan)
        metrics["benchmark_mean_window_ms"] = safe_get_nested(benchmark, "mean_per_window_ms", default=np.nan)
        metrics["benchmark_std_window_ms"] = safe_get_nested(benchmark, "std_per_window_ms", default=np.nan)
        metrics["benchmark_min_window_ms"] = safe_get_nested(benchmark, "min_per_window_ms", default=np.nan)
        metrics["benchmark_max_window_ms"] = safe_get_nested(benchmark, "max_per_window_ms", default=np.nan)
        metrics["benchmark_p95_window_ms"] = safe_get_nested(benchmark, "p95_per_window_ms", default=np.nan)
        metrics["benchmark_p99_window_ms"] = safe_get_nested(benchmark, "p99_per_window_ms", default=np.nan)
    
    # Also try inference_timing.json (Transformer format from standalone timing cell)
    inference_timing = safe_load_json(os.path.join(run_dir, "inference_timing.json"))
    if inference_timing:
        time_stats = safe_get_nested(inference_timing, "time_per_window_ms", default={})
        
        # Only overwrite if not already set from benchmark file
        if pd.isna(metrics.get("benchmark_median_window_ms", np.nan)):
            metrics["benchmark_median_window_ms"] = safe_get_nested(time_stats, "median", default=np.nan)
        if pd.isna(metrics.get("benchmark_throughput", np.nan)):
            metrics["benchmark_throughput"] = safe_get_nested(inference_timing, "throughput_windows_per_sec", default=np.nan)
        if pd.isna(metrics.get("benchmark_mean_window_ms", np.nan)):
            metrics["benchmark_mean_window_ms"] = safe_get_nested(time_stats, "mean", default=np.nan)
        if pd.isna(metrics.get("benchmark_std_window_ms", np.nan)):
            metrics["benchmark_std_window_ms"] = safe_get_nested(time_stats, "std", default=np.nan)
        if pd.isna(metrics.get("benchmark_min_window_ms", np.nan)):
            metrics["benchmark_min_window_ms"] = safe_get_nested(time_stats, "min", default=np.nan)
        if pd.isna(metrics.get("benchmark_max_window_ms", np.nan)):
            metrics["benchmark_max_window_ms"] = safe_get_nested(time_stats, "max", default=np.nan)
        if pd.isna(metrics.get("benchmark_p95_window_ms", np.nan)):
            metrics["benchmark_p95_window_ms"] = safe_get_nested(time_stats, "p95", default=np.nan)
        if pd.isna(metrics.get("benchmark_p99_window_ms", np.nan)):
            metrics["benchmark_p99_window_ms"] = safe_get_nested(time_stats, "p99", default=np.nan)
    
    return metrics


# =============================================================================
# MAIN EXECUTION
# =============================================================================

def run_benchmark(
    model_selection: Dict[str, Union[str, List[str], None]] = None,
    outputs_root: str = OUTPUTS_ROOT,
    benchmark_dir: str = None,
    list_only: bool = False,
):
    """
    Run the benchmark comparison.
    
    Args:
        model_selection: Dict mapping model names to run selections.
                        Use "latest", specific run name, None to skip, or list for multiple.
        outputs_root: Root directory containing model outputs.
        benchmark_dir: Output directory for benchmark results.
        list_only: If True, just list available runs and exit.
    """
    
    if list_only:
        print_available_runs(outputs_root)
        return None
    
    if model_selection is None:
        model_selection = MODEL_SELECTION
    
    if benchmark_dir is None:
        benchmark_dir = os.path.join(outputs_root, "benchmark_comparison")
    
    Path(benchmark_dir).mkdir(parents=True, exist_ok=True)
    print(f"Benchmark output directory: {benchmark_dir}\n")
    
    # ==========================================================================
    # COLLECT ALL MODEL METRICS
    # ==========================================================================
    
    print("=" * 80)
    print("RESOLVING MODEL RUNS")
    print("=" * 80 + "\n")
    
    runs_to_evaluate = []
    
    for model_name, selection in model_selection.items():
        if selection is None:
            print(f"⏭️  {model_name}: Skipped (set to None)")
            continue
        
        resolved = resolve_run_dir(model_name, selection, outputs_root)
        runs_to_evaluate.extend(resolved)
    
    if not runs_to_evaluate:
        print("\n❌ No valid runs to evaluate!")
        print("   Run with list_only=True to see available runs.")
        return None
    
    print(f"\n✓ Will evaluate {len(runs_to_evaluate)} model run(s)\n")
    
    print("=" * 80)
    print("COLLECTING METRICS")
    print("=" * 80 + "\n")
    
    all_metrics = []
    
    for model_label, run_dir in runs_to_evaluate:
        print(f"📊 Extracting metrics: {model_label}")
        metrics = extract_model_metrics(run_dir, model_label)
        all_metrics.append(metrics)
        print(f"   ✓ {len(metrics)} fields extracted\n")
    
    # Create DataFrame
    df = pd.DataFrame(all_metrics)
    
    # Reorder columns
    column_order = [
        "model", "run_dir",
        "total_params", "trainable_params", "trainable_pct",
        "val_loss", "val_acc", "val_f1_macro", "val_f1_micro", "val_f1_weighted",
        "val_p_macro", "val_r_macro", "val_p_micro", "val_r_micro", "val_p_weighted", "val_r_weighted",
        "test_loss", "test_acc", "test_f1_macro", "test_f1_micro", "test_f1_weighted",
        "test_p_macro", "test_r_macro", "test_p_micro", "test_r_micro", "test_p_weighted", "test_r_weighted",
        "test_ms_per_sample", "test_ms_per_batch",
        "clip_acc", "clip_f1_macro", "clip_f1_micro", "clip_f1_weighted",
        "clip_p_macro", "clip_r_macro", "clip_p_micro", "clip_r_micro", "clip_p_weighted", "clip_r_weighted",
        "clip_ms_per_clip", "clip_default_k", "clip_missed_early", "clip_n_clips",
        "benchmark_mean_window_ms", "benchmark_median_window_ms", "benchmark_std_window_ms",
        "benchmark_min_window_ms", "benchmark_max_window_ms",
        "benchmark_p95_window_ms", "benchmark_p99_window_ms", "benchmark_throughput",
    ]
    column_order = [col for col in column_order if col in df.columns]
    df = df[column_order]
    
    print("=" * 80)
    print("METRICS COLLECTION COMPLETE")
    print("=" * 80)
    print(f"\n✓ Collected metrics from {len(df)} model run(s)")
    print(f"✓ Total metrics per model: {len(df.columns)}")
    
    # ==========================================================================
    # SAVE OUTPUTS
    # ==========================================================================
    
    csv_path = os.path.join(benchmark_dir, "full_benchmark_table.csv")
    df.to_csv(csv_path, index=False, float_format="%.6f")
    print(f"\n✓ Saved: {csv_path}")
    
    try:
        excel_path = os.path.join(benchmark_dir, "full_benchmark_table.xlsx")
        df.to_excel(excel_path, index=False, float_format="%.6f")
        print(f"✓ Saved: {excel_path}")
    except Exception as e:
        print(f"⚠️  Excel export failed: {e}")
    
    # ==========================================================================
    # PAPER-READY TABLES
    # ==========================================================================
    
    print("\n" + "=" * 80)
    print("GENERATING PAPER-READY TABLES")
    print("=" * 80 + "\n")
    
    # Table 1: Model Overview
    table1_cols = ["model", "total_params", "trainable_params", "test_acc", 
                   "test_f1_macro", "test_f1_micro", "test_f1_weighted", 
                   "benchmark_median_window_ms"]
    table1 = df[[col for col in table1_cols if col in df.columns]].copy()
    
    if "total_params" in table1.columns:
        table1["total_params"] = table1["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    if "trainable_params" in table1.columns:
        table1["trainable_params"] = table1["trainable_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    
    table1.rename(columns={
        "model": "Model", "total_params": "Params (Total)", "trainable_params": "Params (Train)",
        "test_acc": "Acc", "test_f1_macro": "F1 (M)", "test_f1_micro": "F1 (mi)",
        "test_f1_weighted": "F1 (W)", "benchmark_median_window_ms": "ms/window",
    }, inplace=True)
    
    print("TABLE 1: Model Overview (Test Window-Level)")
    print("-" * 100)
    print(table1.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print()
    
    table1_latex = table1.to_latex(
        index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
        caption="Model comparison: Parameters and test window-level performance.",
        label="tab:model_overview",
    )
    with open(os.path.join(benchmark_dir, "table1_model_overview.tex"), 'w', encoding='utf-8') as f:
        f.write(table1_latex)
    
    # Table 2: Detailed Window Metrics
    table2_cols = ["model", "test_acc", "test_p_macro", "test_r_macro", "test_f1_macro",
                   "test_p_micro", "test_r_micro", "test_f1_micro",
                   "test_p_weighted", "test_r_weighted", "test_f1_weighted"]
    table2 = df[[col for col in table2_cols if col in df.columns]].copy()
    table2.rename(columns={
        "model": "Model", "test_acc": "Acc",
        "test_p_macro": "P (M)", "test_r_macro": "R (M)", "test_f1_macro": "F1 (M)",
        "test_p_micro": "P (mi)", "test_r_micro": "R (mi)", "test_f1_micro": "F1 (mi)",
        "test_p_weighted": "P (W)", "test_r_weighted": "R (W)", "test_f1_weighted": "F1 (W)",
    }, inplace=True)
    
    print("TABLE 2: Detailed Window-Level Metrics (TEST)")
    print("-" * 100)
    print(table2.to_string(index=False, float_format="%.4f"))
    print()
    
    table2_latex = table2.to_latex(index=False, float_format="%.4f",
        caption="Detailed window-level test metrics.", label="tab:window_metrics")
    with open(os.path.join(benchmark_dir, "table2_window_metrics.tex"), 'w', encoding='utf-8') as f:
        f.write(table2_latex)
    
    # Table 3: Clip-Level
    table3_cols = ["model", "clip_acc", "clip_p_macro", "clip_r_macro", "clip_f1_macro",
                   "clip_p_micro", "clip_r_micro", "clip_f1_micro",
                   "clip_p_weighted", "clip_r_weighted", "clip_f1_weighted", "clip_default_k"]
    table3 = df[[col for col in table3_cols if col in df.columns]].copy()
    table3.rename(columns={
        "model": "Model", "clip_acc": "Acc",
        "clip_p_macro": "P (M)", "clip_r_macro": "R (M)", "clip_f1_macro": "F1 (M)",
        "clip_p_micro": "P (mi)", "clip_r_micro": "R (mi)", "clip_f1_micro": "F1 (mi)",
        "clip_p_weighted": "P (W)", "clip_r_weighted": "R (W)", "clip_f1_weighted": "F1 (W)",
        "clip_default_k": "k",
    }, inplace=True)
    
    print("TABLE 3: Clip-Level Performance (TEST)")
    print("-" * 100)
    print(table3.to_string(index=False, float_format="%.4f"))
    print()
    
    table3_latex = table3.to_latex(index=False, float_format="%.4f",
        caption="Clip-level test metrics.", label="tab:clip_metrics")
    with open(os.path.join(benchmark_dir, "table3_clip_metrics.tex"), 'w', encoding='utf-8') as f:
        f.write(table3_latex)
    
    # Table 4: Timing (updated with comprehensive timing stats)
    table4_cols = ["model", "total_params", "test_ms_per_sample",
                   "benchmark_mean_window_ms", "benchmark_median_window_ms",
                   "benchmark_std_window_ms", "benchmark_p95_window_ms", "benchmark_throughput"]
    table4 = df[[col for col in table4_cols if col in df.columns]].copy()
    if "total_params" in table4.columns:
        table4["total_params"] = df["total_params"].apply(lambda x: f"{int(x):,}" if pd.notna(x) else "N/A")
    table4.rename(columns={
        "model": "Model", "total_params": "Params",
        "test_ms_per_sample": "ms/win (eval)",
        "benchmark_mean_window_ms": "Mean (ms)",
        "benchmark_median_window_ms": "Median (ms)",
        "benchmark_std_window_ms": "Std (ms)",
        "benchmark_p95_window_ms": "P95 (ms)",
        "benchmark_throughput": "windows/s",
    }, inplace=True)
    
    print("TABLE 4: Inference Timing")
    print("-" * 100)
    print(table4.to_string(index=False, float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x)))
    print()
    
    table4_latex = table4.to_latex(index=False,
        float_format=lambda x: f"{x:.4f}" if isinstance(x, float) else str(x),
        caption="Inference timing benchmarks.", label="tab:timing")
    with open(os.path.join(benchmark_dir, "table4_timing.tex"), 'w', encoding='utf-8') as f:
        f.write(table4_latex)
    
    # ==========================================================================
    # RANKINGS
    # ==========================================================================
    
    print("=" * 80)
    print("BEST MODEL RANKINGS")
    print("=" * 80 + "\n")
    
    rankings = {}
    
    for metric, ascending, label in [
        ("test_f1_macro", False, "Test F1 (Macro) - Window"),
        ("clip_f1_macro", False, "Test F1 (Macro) - Clip"),
        ("test_acc", False, "Test Accuracy - Window"),
        ("benchmark_median_window_ms", True, "⚡ Fastest Inference (Median)"),
        ("benchmark_throughput", False, "⚡ Highest Throughput"),
    ]:
        if metric in df.columns and df[metric].notna().any():
            sorted_df = df.sort_values(metric, ascending=ascending)
            print(f"{label}:")
            for _, row in sorted_df.iterrows():
                val = row[metric]
                if pd.notna(val):
                    if "ms" in metric:
                        suffix = " ms"
                    elif "throughput" in metric:
                        suffix = " windows/s"
                    else:
                        suffix = ""
                    print(f"  {row['model']:30s} → {val:.4f}{suffix}")
            print()
            rankings[metric] = sorted_df[["model", metric]].dropna().to_dict('records')
    
    with open(os.path.join(benchmark_dir, "model_rankings.json"), 'w') as f:
        json.dump(rankings, f, indent=2)
    
    # ==========================================================================
    # SUMMARY
    # ==========================================================================
    
    print("=" * 80)
    print("BENCHMARK COMPLETE")
    print("=" * 80)
    print(f"\n📁 Output: {benchmark_dir}\n")
    
    return df


# =============================================================================
# COMMAND-LINE INTERFACE
# =============================================================================

if __name__ == "__main__":
    # Step 1: List available runs to see what's there
    print_available_runs(OUTPUTS_ROOT)
    
    # Step 2: Run the benchmark with MODEL_SELECTION defined above
    df = run_benchmark()
    
    # Or run with custom selection inline:
    # df = run_benchmark(
    #     model_selection={
    #         "Transformer": "transformer_aug",   # specific run
    #         "Apple": "latest",                   # most recent
    #         "InceptionTime": None,               # skip this one
    #     },
    #     outputs_root=OUTPUTS_ROOT,
    # )